<a href="https://colab.research.google.com/github/samer-glitch/TADP-sec/blob/main/14_6_2026_TADP_Sec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
import shutil
import os

uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith(".csv"):
        shutil.move(filename, "/content/creditcard.csv")
        print(f"Uploaded and saved as: /content/creditcard.csv")

print(os.path.exists("/content/creditcard.csv"))

Saving creditcard.csv to creditcard.csv
Uploaded and saved as: /content/creditcard.csv
True


In [2]:
# ============================================================
# TADP-SEC: TRUSTWORTHY FEDERATED LEARNING WITH FULL ENFORCEMENT
# v9-final- Professional Publication-Ready Implementation
# Q1-READY VERSION with Enhanced Metrics, Threat Analysis
# ============================================================
# ============================================================
# SETUP FOR GOOGLE COLAB / JUPYTER NOTEBOOK
# ============================================================
# Enable inline plotting for notebooks
import matplotlib
# matplotlib.use('inline')

# Check if we're in a notebook environment
def setup_notebook_display():
    """Setup for notebook display"""
    try:
        from IPython import get_ipython
        import matplotlib
        import matplotlib.pyplot as plt

        # Set backend for inline display
        # Set backend for inline display
        matplotlib.use('module://matplotlib_inline.backend_inline')

        # Configure matplotlib for inline display
        get_ipython().run_line_magic('matplotlib', 'inline')

        # Set EXCLUSIVELY 600 DPI for both display and saving (Q1 journal requirement)
        plt.rcParams['figure.dpi'] = 600      # Display DPI
        plt.rcParams['savefig.dpi'] = 600     # Saving DPI (CRITICAL)
        plt.rcParams['savefig.format'] = 'png' # Lossless format
        plt.rcParams['savefig.bbox'] = 'tight' # Tight bounding box
        plt.rcParams['savefig.pad_inches'] = 0.1 # Minimal padding
        plt.rcParams['figure.figsize'] = (12, 8)

        print("✓ Configured for Q1 journal: ALL plots at 600 DPI")
        return True
    except:
        print("Not in notebook environment, using default settings")
        return False

# Call this at the beginning
setup_notebook_display()

import os
import sys
import time
import json
import math
import base64
import hashlib
import random
import csv
import secrets
import struct
from dataclasses import dataclass, asdict, field
from typing import Dict, List, Tuple, Any, Optional, Union, Set
from datetime import datetime, timezone
from collections import defaultdict, Counter
from pathlib import Path
import hmac

import numpy as np
import pandas as pd
import scipy.stats as stats
from tqdm import tqdm
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import ec, rsa, padding
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.exceptions import InvalidSignature
from cryptography.hazmat.backends import default_backend

# ML libraries
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    precision_recall_curve, matthews_corrcoef, balanced_accuracy_score
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

# Configure plotting
# Configure plotting for Q1 journal publication
plt.style.use('default')  # Clean, reproducible
rcParams.update({
    'figure.figsize': (12, 8),
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    # CRITICAL 600 DPI settings for Q1 journals:
    'savefig.dpi': 600,           # Saving resolution
    'savefig.format': 'png',      # Lossless format
    'savefig.bbox': 'tight',      # Tight bounding box
    'savefig.pad_inches': 0.1,    # Minimal padding
    'figure.dpi': 600,            # Display resolution
    'figure.autolayout': True,    # Auto layout adjustment
    # Font settings for publication:
    'font.family': 'serif',
    'pdf.fonttype': 42,           # Embed fonts in PDF
    'ps.fonttype': 42,            # Embed fonts in PS
})

# --------------------------
# REPRODUCIBILITY & GLOBAL CONFIG
# --------------------------
# --------------------------
# REPRODUCIBILITY & GLOBAL CONFIG
# --------------------------
MASTER_SEED = 42

# Final evaluation setting
# 5 seeds for manuscript-level reporting. for all scenarios Except T4
# Main scenarios use 5 seeds for statistical reporting
SEEDS = [7, 11, 19, 23, 42]
FULL_SEEDS = SEEDS

# T4 is much slower because it uses algebraic CKKS double key switching
T4_SEEDS = [7] #, 11]

# Start with 2 or 3 rounds. 5 if runtime is acceptable.
T4_ROUNDS = 10 # 5

random.seed(MASTER_SEED)
np.random.seed(MASTER_SEED)

# Core FL configuration
DEBUG_MODE = False
N_CLIENTS = 20

# Smoke test: 2 rounds.
# Final run: 20 or 30 rounds.
ROUNDS = 20

LOCAL_EPOCHS = 2
BATCH_SIZE = 32
LR = 0.01
MIN_SAMPLES_PER_CLIENT = 50

# Security Parameters
SIG_ALGORITHM = "RSA-PSS"
BASE_MAX_PACKETS_PER_NONCE = 50
THRESHOLD_CLIENTS_FOR_XMK = 4

XMK_SCENARIOS = {
    "B1_xMK_CKKS",
    "T3_XMK_CKKS_TADP",
    "T3_TEST_POFC_ATTACK"
}

# xMK decryption-share noise flooding.
# In xMK-CKKS, decryption shares include a larger error e_i^*
# to protect each participant's secret key during distributed decryption.


# Fraud-detection threshold constraint
# 0.005 = 0.5% false-positive-rate constraint
FPR_CONSTRAINT_VALUE = 0.005
# --------------------------
# T4: STRONGER CKKS + DOUBLE KEY SWITCHING PARAMETERS
# --------------------------
T4_CKKS_SCHEME = "CKKS_DOUBLE_KS_RLWE"

# Stronger T4 profile aligned with the slide:
# n = 8192, log2(q) = 218, scale = 2^20, noise = 3.2
# n= 4096 for faster T4 prototype
T4_CKKS_POLY_MODULUS_DEGREE = 4096 # 2048 with LOGQ = 54 WHILE  4096 goes better with LOGQ = 110
T4_CKKS_SLOTS = T4_CKKS_POLY_MODULUS_DEGREE // 2

T4_CKKS_LOGQ = 110 # 110 goes with n =4096 settings
T4_CKKS_Q = 2 ** T4_CKKS_LOGQ

T4_CKKS_SCALE = 2 ** 20
T4_CKKS_NOISE_STD = 3.2

# Coefficient-modulus chain: 50 + 30 + 30 = 110 bits
# T4_CKKS_COEFF_MOD_BITS = [50, 30, 30] this is good for n= 4096
T4_CKKS_COEFF_MOD_BITS = [50, 30, 30]

# --------------------------
# T4 KEY-SWITCHING MODE
# --------------------------
# "formal"   = closest prototype to CKKS gadget key switching using KSKs.
# "semantic" = stable decrypt/re-encrypt fallback used only if formal KS collapses.
T4_KS_MODE = "formal"

# Keep this True only for diagnostic runs.
T4_DEBUG = False

T4_CKKS_COEFF_BYTES = math.ceil(T4_CKKS_LOGQ / 8)
T4_CKKS_CIPHERTEXT_BYTES = (
    2 * T4_CKKS_POLY_MODULUS_DEGREE * T4_CKKS_COEFF_BYTES
)

# This is only a display/metadata factor.
# Actual T4 communication below is computed from ciphertext size.
CKKS_DOUBLE_KS_FACTOR = T4_CKKS_CIPHERTEXT_BYTES / max(1, ((29 * 8) + 8))

# --------------------------
# T4 PARAMETER-AWARE RUNTIME CALIBRATION
# --------------------------
# These are calibrated simulation estimates, not real OpenFHE/SEAL runtime.
# They prevent T4 from unrealistically appearing as 0.1–0.3 ms.
T4_ENC_MS_AT_8192 = 12.0
T4_KS_MS_AT_8192 = 8.0
T4_AGG_MS_AT_8192 = 1.0
T4_DEC_MS_AT_8192 = 3.0

def t4_param_scale() -> float:
    """
    Scale simulated CKKS runtime by ring degree and modulus size.
    Reference point: n=8192, logq=218.
    """
    return (
        (T4_CKKS_POLY_MODULUS_DEGREE / 8192.0)
        * (T4_CKKS_LOGQ / 218.0)
    )

def estimate_t4_client_encrypt_ms() -> float:
    """
    Estimated client-side CKKS encoding + encryption time per client update.
    """
    return T4_ENC_MS_AT_8192 * t4_param_scale()

def estimate_t4_server_round_ms(n_clients: int) -> float:
    """
    Estimated server-side T4 cost per round:
    client-key -> aggregation-key switching,
    encrypted aggregation,
    aggregation-key -> client-key switching,
    and one representative client decryption.
    """
    n = max(1, int(n_clients))
    scale = t4_param_scale()

    ks_to_agg = n * T4_KS_MS_AT_8192 * scale
    encrypted_add = n * T4_AGG_MS_AT_8192 * scale
    ks_to_clients = n * T4_KS_MS_AT_8192 * scale
    representative_decrypt = T4_DEC_MS_AT_8192 * scale

    return ks_to_agg + encrypted_add + ks_to_clients + representative_decrypt

# --------------------------
# CKKS / RLWE RIGOR PARAMETERS (SIMULATED)
# --------------------------
RLWE_LOGQ_T3 = 300

# --------------------------
# CUSTOM CRYPTO SIMULATION PARAMETERS
# --------------------------
# These values make the custom simulator closer to the referenced crypto behavior
# without depending on OpenFHE, SEAL, or TenSEAL.

# --------------------------
# Real modified Domingo-Ferrer + KS parameters - DF+KS Profile
# --------------------------
DF_D = 50                      # paper reports d=50 as practical reference
DF_SCALE = 10**6               # fixed-point encoding for model updates
DF_M0 = 2**61 - 1
# Important: DF_M must be a multiple of DF_M0 so that
# the key-switching noise term m0 * e vanishes after reduction mod m0.
DF_M = DF_M0 * (2**67 - 1)     # public ciphertext modulus; multiple of DF_M0
# IN DF_M: This keeps DF_M around 128 bits, so the communication size should remain almost the same, but the algebra becomes correct.

DF_KS_NOISE_BOUND = 3          # small LWE-style noise e in matrix M
DF_CT_BYTES_PER_ENTRY = math.ceil(DF_M.bit_length() / 8)

# T3 xMK-CKKS profile
XMK_CKKS_POLY_MODULUS_DEGREE = 4096
XMK_CKKS_LOGQ = 180
XMK_CKKS_SCALE = 2 ** 30
XMK_CKKS_NOISE_STD = 3.2
XMK_SHARE_NOISE_STD = 8.0
XMK_CKKS_COEFF_BYTES = math.ceil(XMK_CKKS_LOGQ / 8)
XMK_CKKS_CIPHERTEXT_BYTES = (
    2 * XMK_CKKS_POLY_MODULUS_DEGREE * XMK_CKKS_COEFF_BYTES
)

def _stable_mask(shape, tag: str, scale: float = 1.0) -> np.ndarray:
    """
    Deterministic pseudo-random mask used only for simulation metadata.
    It avoids storing raw plaintext-like ciphertext components.
    """
    digest = hashlib.sha256(tag.encode()).digest()
    seed = int.from_bytes(digest[:8], byteorder="big", signed=False)
    rng = np.random.default_rng(seed)
    return rng.normal(0.0, scale, size=shape)

# ============================================================
# ALGEBRAIC CKKS / RLWE PROTOTYPE OVER Rq = Zq[X]/(X^n + 1)
# ============================================================

def _is_probable_prime_64(n: int) -> bool:
    """Deterministic Miller-Rabin for 64-bit integers."""
    if n < 2:
        return False

    small_primes = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37]
    for p in small_primes:
        if n % p == 0:
            return n == p

    d = n - 1
    s = 0
    while d % 2 == 0:
        s += 1
        d //= 2

    # valid deterministic bases for < 2^64
    bases = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37]
    for a in bases:
        if a >= n:
            continue

        x = pow(a, d, n)
        if x == 1 or x == n - 1:
            continue

        passed = False
        for _ in range(s - 1):
            x = pow(x, 2, n)
            if x == n - 1:
                passed = True
                break

        if not passed:
            return False

    return True


def _previous_prime_under_power(bits: int, offset: int = 1) -> int:
    """Find a deterministic prime below 2^bits."""
    candidate = (1 << bits) - offset

    if candidate % 2 == 0:
        candidate -= 1

    while candidate > 3:
        if _is_probable_prime_64(candidate):
            return candidate
        candidate -= 2

    raise ValueError(f"Could not find prime under 2^{bits}")


def make_rns_primes_for_logq(logq: int) -> List[int]:
    """
    Build pairwise distinct 64-bit RNS primes whose product has roughly logq bits.
    """
    if logq >= 218:
        bit_plan = [61, 61, 48, 48]   # ≈218 bits
    elif logq >= 180:
        bit_plan = [60, 60, 60]       # ≈180 bits
    elif logq >= 110:
        bit_plan = [50, 30, 30]       # ≈110 bits
    elif logq >= 54:
        bit_plan = [30, 24]           # ≈54 bits for fast T4 n=2048
    else:
        bit_plan = [27, 27]           # fallback ≈54 bits

    primes = []
    for i, b in enumerate(bit_plan):
        p = _previous_prime_under_power(b, offset=1 + 1000 * i)
        while p in primes:
            p = _previous_prime_under_power(b, offset=1 + 2000 * (i + 1))
        primes.append(p)

    return primes


def rns_zero(primes: List[int], n: int) -> np.ndarray:
    return np.zeros((len(primes), n), dtype=np.int64)


def rns_from_int_poly(int_poly: np.ndarray, primes: List[int], n: int) -> np.ndarray:
    """Convert integer polynomial to RNS representation safely."""
    int_poly = np.asarray(int_poly, dtype=object).reshape(-1)

    if int_poly.size > n:
        raise ValueError(f"Polynomial length {int_poly.size} exceeds ring degree {n}")

    padded = np.zeros(n, dtype=object)

    # Critical: force every coefficient to Python int BEFORE modular reduction.
    for idx, x in enumerate(int_poly):
        padded[idx] = int(x)

    out = np.zeros((len(primes), n), dtype=np.int64)

    for i, p in enumerate(primes):
        p_int = int(p)
        out[i] = np.array(
            [(int(x) % p_int) for x in padded],
            dtype=np.int64
        )

    return out

def rns_uniform(primes: List[int], n: int, rng: np.random.Generator) -> np.ndarray:
    """Uniform random polynomial in Rq."""
    out = np.zeros((len(primes), n), dtype=np.int64)

    for i, p in enumerate(primes):
        out[i] = rng.integers(0, int(p), size=n, dtype=np.int64)

    return out

def rns_error(primes: List[int], n: int, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Discrete Gaussian-like error polynomial in Rq."""
    err = np.array(
        [int(round(float(x))) for x in rng.normal(0.0, sigma, size=n)],
        dtype=object
    )
    return rns_from_int_poly(err, primes, n)


def rns_add(a: np.ndarray, b: np.ndarray, primes: List[int]) -> np.ndarray:
    out = np.empty_like(a)

    for i, p in enumerate(primes):
        out[i] = (a[i] + b[i]) % p

    return out


def rns_sub(a: np.ndarray, b: np.ndarray, primes: List[int]) -> np.ndarray:
    out = np.empty_like(a)

    for i, p in enumerate(primes):
        out[i] = (a[i] - b[i]) % p

    return out


def rns_scalar_mul(a: np.ndarray, scalar: int, primes: List[int]) -> np.ndarray:
    out = np.empty_like(a)

    for i, p in enumerate(primes):
        out[i] = (a[i] * int(scalar)) % p

    return out


def sample_sparse_ternary_secret(
    n: int,
    hamming_weight: int,
    rng: np.random.Generator
) -> List[Tuple[int, int]]:
    """
    Sparse ternary secret key: coefficients are in {-1, 0, +1}.
    Stored as list of (position, sign).
    """
    h = min(hamming_weight, n)
    positions = rng.choice(n, size=h, replace=False)
    signs = rng.choice([-1, 1], size=h)

    return [(int(pos), int(sign)) for pos, sign in zip(positions, signs)]


def _negacyclic_shift(poly: np.ndarray, k: int, p: int) -> np.ndarray:
    """
    Compute X^k * poly mod (X^n + 1), coefficient-wise mod p.
    """
    n = poly.shape[0]

    if k == 0:
        return poly.copy()

    out = np.empty_like(poly)

    # Terms wrapping around get a minus sign because X^n = -1.
    out[:k] = (-poly[-k:]) % p
    out[k:] = poly[:-k]

    return out


def rns_mul_by_sparse_ternary(
    poly: np.ndarray,
    sparse_secret: List[Tuple[int, int]],
    primes: List[int]
) -> np.ndarray:
    """
    Exact multiplication in Rq by a sparse ternary polynomial.
    This avoids O(n^2) dense multiplication and is feasible in Colab.
    """
    n = poly.shape[1]
    out = np.zeros_like(poly)

    for level, p in enumerate(primes):
        acc = np.zeros(n, dtype=np.int64)

        for pos, sign in sparse_secret:
            shifted = _negacyclic_shift(poly[level], pos, p)

            if sign == 1:
                acc = (acc + shifted) % p
            else:
                acc = (acc - shifted) % p

        out[level] = acc

    return out


def _center_first_prime_coeffs(
    rns_poly: np.ndarray,
    primes: List[int]
) -> np.ndarray:
    """
    Recover centered integer coefficients from the first RNS prime.
    """
    p = int(primes[0])
    half = p // 2

    coeffs = rns_poly[0].astype(object)
    centered = []

    for x in coeffs:
        xi = int(x)
        if xi > half:
            xi -= p
        centered.append(xi)

    return np.asarray(centered, dtype=float)


def ckks_encode_real_slots_to_rns(
    vec: np.ndarray,
    primes: List[int],
    n: int,
    scale: int
) -> np.ndarray:
    """
    CKKS-style canonical slot encoding for real FL update vectors.

    This replaces coefficient packing. It maps real values into the canonical
    embedding slots of R = Z[X]/(X^n + 1), then converts the resulting
    polynomial coefficients into RNS form.

    This is still a Python research implementation, not OpenFHE/SEAL.
    """
    flat = np.asarray(vec, dtype=float).reshape(-1)
    slots = n // 2

    if flat.size > slots:
        raise ValueError(
            f"CKKS encoder supports at most {slots} real slots for n={n}; "
            f"got vector length {flat.size}."
        )

    # Fill CKKS complex slots with real values.
    z = np.zeros(slots, dtype=np.complex128)
    z[:flat.size] = flat.astype(np.complex128)

    # Hermitian symmetry for real-coefficient polynomial.
    # y_j = p(zeta^(2j+1)), j = 0,...,n-1
    y = np.zeros(n, dtype=np.complex128)
    y[:slots] = z
    y[slots:] = np.conjugate(z[::-1])

    # Convert canonical values to polynomial coefficients.
    # Let zeta = exp(2πi / 2n), eta = zeta^2.
    # y_j = sum_k a_k * zeta^k * eta^(jk).
    zeta = np.exp(2j * np.pi / (2 * n))
    k = np.arange(n)

    b = np.fft.fft(y) / n
    coeffs_complex = b / (zeta ** k)

    coeffs_real = np.real_if_close(coeffs_complex, tol=1000).real
    encoded_coeffs = np.array(
        [int(round(float(c) * float(scale))) for c in coeffs_real],
        dtype=object
    )

    return rns_from_int_poly(encoded_coeffs, primes, n)


def ckks_decode_rns_first_prime(
    rns_poly: np.ndarray,
    primes: List[int],
    length: int,
    scale: int
) -> np.ndarray:
    """
    CKKS-style canonical slot decoding from the first RNS prime.
    """
    n = rns_poly.shape[1]
    slots = n // 2

    if length > slots:
        raise ValueError(
            f"CKKS decoder supports at most {slots} real slots for n={n}; "
            f"requested length={length}."
        )

    coeffs = _center_first_prime_coeffs(rns_poly, primes) / float(scale)

    zeta = np.exp(2j * np.pi / (2 * n))
    k = np.arange(n)

    b = coeffs * (zeta ** k)

    # y_j = sum_k b_k * exp(2πijk/n)
    y = np.fft.ifft(b) * n

    decoded = np.real(y[:length])
    return decoded.astype(float)


@dataclass
class AlgebraicRLWECiphertext:
    """
    Algebraic RLWE ciphertext over Rq = Zq[X]/(X^n + 1).
    """
    c0: np.ndarray
    c1: np.ndarray
    original_shape: tuple
    client_id: str
    key_tag: str
    n: int
    logq: int
    scale: int
    primes: List[int]
    scheme: str
    noise_std: float
    cumulative_noise: float
    ks_count: int = 0
    metadata: Dict[str, Any] = field(default_factory=dict)

    @property
    def value(self) -> np.ndarray:
        """
        Compatibility placeholder.
        Do not use this as plaintext before decryption.
        """
        return np.zeros(self.original_shape, dtype=float)

    @property
    def expanded(self) -> bool:
        return True

    def get_size_bytes(self) -> int:
        coeff_bytes = math.ceil(self.logq / 8)
        return 2 * self.n * coeff_bytes

    def effective_noise_std(self) -> float:
        return float(self.cumulative_noise) / float(self.scale)

class AlgebraicCKKSRLWEEngine:
    """
    Custom algebraic CKKS-style RLWE prototype.

    Supports:
    - Rq = Zq[X]/(X^n + 1)
    - RNS modular arithmetic
    - sparse RLWE secret keys
    - RLWE public-key encryption
    - homomorphic addition
    - CKKS-style canonical slot encoding for real-valued FL updates
    - decryption by c0 + c1*s
    """

    def __init__(
        self,
        n: int,
        logq: int,
        scale: int,
        noise_std: float,
        seed_tag: str
    ):
        self.n = int(n)
        self.logq = int(logq)
        self.scale = int(scale)
        self.noise_std = float(noise_std)
        self.primes = make_rns_primes_for_logq(logq)
        self.seed_tag = str(seed_tag)
        self.secret_hamming_weight = min(64, max(8, self.n // 128))

    def _rng(self, tag: str) -> np.random.Generator:
        digest = hashlib.sha256(f"{self.seed_tag}|{tag}".encode()).digest()
        seed = int.from_bytes(digest[:8], "big", signed=False)
        return np.random.default_rng(seed)

    def encode(self, vec: np.ndarray) -> np.ndarray:
        return ckks_encode_real_slots_to_rns(
            vec=vec,
            primes=self.primes,
            n=self.n,
            scale=self.scale
        )

    def decode(self, plain_poly: np.ndarray, shape: tuple) -> np.ndarray:
        length = int(np.prod(shape))
        decoded = ckks_decode_rns_first_prime(
            plain_poly,
            self.primes,
            length=length,
            scale=self.scale
        )
        return decoded.reshape(shape)

    def keygen(self, key_id: str, common_a: Optional[np.ndarray] = None) -> Dict[str, Any]:
        rng = self._rng(f"keygen|{key_id}")

        sk = sample_sparse_ternary_secret(
            self.n,
            self.secret_hamming_weight,
            rng
        )

        if common_a is None:
            a = rns_uniform(self.primes, self.n, rng)
        else:
            a = common_a.copy()

        e = rns_error(self.primes, self.n, self.noise_std, rng)

        # RLWE public key: b = -a*s + e mod q
        a_times_s = rns_mul_by_sparse_ternary(a, sk, self.primes)
        b = rns_sub(e, a_times_s, self.primes)

        return {
            "key_id": key_id,
            "sk": sk,
            "pk": {
                "b": b,
                "a": a,
                "key_id": key_id
            }
        }

    def encrypt(
        self,
        pk: Dict[str, Any],
        vec: np.ndarray,
        client_id: str,
        key_tag: str,
        scheme: str
    ) -> AlgebraicRLWECiphertext:
        rng = self._rng(f"encrypt|{client_id}|{key_tag}|{time.time_ns()}")

        m = self.encode(vec)

        u = sample_sparse_ternary_secret(
            self.n,
            self.secret_hamming_weight,
            rng
        )

        e1 = rns_error(self.primes, self.n, self.noise_std, rng)
        e2 = rns_error(self.primes, self.n, self.noise_std, rng)

        b_u = rns_mul_by_sparse_ternary(pk["b"], u, self.primes)
        a_u = rns_mul_by_sparse_ternary(pk["a"], u, self.primes)

        # c0 = b*u + e1 + m
        c0 = rns_add(rns_add(b_u, e1, self.primes), m, self.primes)

        # c1 = a*u + e2
        c1 = rns_add(a_u, e2, self.primes)

        return AlgebraicRLWECiphertext(
            c0=c0,
            c1=c1,
            original_shape=np.asarray(vec).shape,
            client_id=client_id,
            key_tag=key_tag,
            n=self.n,
            logq=self.logq,
            scale=self.scale,
            primes=self.primes,
            scheme=scheme,
            noise_std=self.noise_std / self.scale,
            cumulative_noise=self.noise_std,
            metadata={
                "ring": "Zq[X]/(X^n+1)",
                "rlwe_form": "c0=b*u+e1+m, c1=a*u+e2",
                "key_tag": key_tag
            }
        )


    def encrypt_rns_plain(
        self,
        pk: Dict[str, Any],
        m_rns: np.ndarray,
        client_id: str,
        key_tag: str,
        scheme: str
    ) -> AlgebraicRLWECiphertext:
        """
        Encrypt an already-encoded RNS plaintext polynomial.

        Used for formal key-switching keys:
            Enc_pk_target(gadget_digit * s_old)
        """
        rng = self._rng(f"encrypt_rns_plain|{client_id}|{key_tag}|{time.time_ns()}")

        u = sample_sparse_ternary_secret(
            self.n,
            self.secret_hamming_weight,
            rng
        )

        e1 = rns_error(self.primes, self.n, self.noise_std, rng)
        e2 = rns_error(self.primes, self.n, self.noise_std, rng)

        b_u = rns_mul_by_sparse_ternary(pk["b"], u, self.primes)
        a_u = rns_mul_by_sparse_ternary(pk["a"], u, self.primes)

        c0 = rns_add(rns_add(b_u, e1, self.primes), m_rns, self.primes)
        c1 = rns_add(a_u, e2, self.primes)

        return AlgebraicRLWECiphertext(
            c0=c0,
            c1=c1,
            original_shape=(self.n,),
            client_id=client_id,
            key_tag=key_tag,
            n=self.n,
            logq=self.logq,
            scale=self.scale,
            primes=self.primes,
            scheme=scheme,
            noise_std=self.noise_std / self.scale,
            cumulative_noise=self.noise_std,
            metadata={
                "ring": "Zq[X]/(X^n+1)",
                "operation": "encrypt_rns_plain",
                "key_tag": key_tag
            }
        )

    def encrypt_rns_plain_under_sk_exact(
        self,
        target_sk: List[Tuple[int, int]],
        m_rns: np.ndarray,
        client_id: str,
        key_tag: str,
        scheme: str
    ) -> AlgebraicRLWECiphertext:
        """
        Exact RLWE-style evaluation-key encryption under a target secret key.

        This is used only for the custom formal key-switching prototype.

        It creates (c0, c1) such that:

            c0 + c1 * s_target = m

        Therefore, the key-switching key decrypts exactly to the gadget component.
        This avoids the massive noise explosion observed with ordinary noisy PK encryption.
        """

        rng = self._rng(f"exact_ksk|{client_id}|{key_tag}|{time.time_ns()}")

        # Random c1 under the target key domain
        c1 = rns_uniform(self.primes, self.n, rng)

        # c0 = m - c1*s_target
        c1_s = rns_mul_by_sparse_ternary(c1, target_sk, self.primes)
        c0 = rns_sub(m_rns, c1_s, self.primes)

        return AlgebraicRLWECiphertext(
            c0=c0,
            c1=c1,
            original_shape=(self.n,),
            client_id=client_id,
            key_tag=key_tag,
            n=self.n,
            logq=self.logq,
            scale=self.scale,
            primes=self.primes,
            scheme=scheme,
            noise_std=0.0,
            cumulative_noise=0.0,
            metadata={
                "ring": "Zq[X]/(X^n+1)",
                "operation": "exact_eval_key_encryption",
                "key_tag": key_tag,
                "prototype_note": (
                    "Exact KSK ciphertext for formal key-switching prototype. "
                    "Used to prevent gadget-noise explosion in Python implementation."
                )
            }
        )

    def plaintext_mul_ciphertext(
        self,
        digit_poly: np.ndarray,
        ct: AlgebraicRLWECiphertext
    ) -> AlgebraicRLWECiphertext:
        """
        Multiply an encrypted evaluation-key ciphertext by a plaintext digit polynomial.
        """
        c0_new = rns_negacyclic_mul_naive(digit_poly, ct.c0, self.primes)
        c1_new = rns_negacyclic_mul_naive(digit_poly, ct.c1, self.primes)

        return AlgebraicRLWECiphertext(
            c0=c0_new,
            c1=c1_new,
            original_shape=ct.original_shape,
            client_id=ct.client_id,
            key_tag=ct.key_tag,
            n=self.n,
            logq=self.logq,
            scale=self.scale,
            primes=self.primes,
            scheme=ct.scheme,
            noise_std=ct.noise_std,
            cumulative_noise=ct.cumulative_noise,
            ks_count=ct.ks_count,
            metadata={
                **ct.metadata,
                "operation": "plaintext_digit_times_eval_key"
            }
        )

    def decrypt(
        self,
        sk: List[Tuple[int, int]],
        ct: AlgebraicRLWECiphertext
    ) -> np.ndarray:
        # m' = c0 + c1*s mod q
        c1_s = rns_mul_by_sparse_ternary(ct.c1, sk, self.primes)
        m_poly = rns_add(ct.c0, c1_s, self.primes)
        return self.decode(m_poly, ct.original_shape)

    def add_ciphertexts(
        self,
        cts: List[AlgebraicRLWECiphertext],
        key_tag: str,
        client_id: str = "server_aggregate"
    ) -> AlgebraicRLWECiphertext:
        if not cts:
            raise ValueError("No ciphertexts to add.")

        c0_sum = rns_zero(self.primes, self.n)
        c1_sum = rns_zero(self.primes, self.n)

        for ct in cts:
            c0_sum = rns_add(c0_sum, ct.c0, self.primes)
            c1_sum = rns_add(c1_sum, ct.c1, self.primes)

        noise = math.sqrt(sum(ct.cumulative_noise ** 2 for ct in cts))

        return AlgebraicRLWECiphertext(
            c0=c0_sum,
            c1=c1_sum,
            original_shape=cts[0].original_shape,
            client_id=client_id,
            key_tag=key_tag,
            n=self.n,
            logq=self.logq,
            scale=self.scale,
            primes=self.primes,
            scheme=cts[0].scheme,
            noise_std=float(np.mean([ct.noise_std for ct in cts])),
            cumulative_noise=noise,
            ks_count=max(ct.ks_count for ct in cts),
            metadata={
                "operation": "homomorphic_addition",
                "n_aggregated": len(cts)
            }
        )

# ============================================================
# FORMAL GADGET-STYLE KEY SWITCHING HELPERS FOR T4
# ============================================================

T4_KS_BASE_BITS = 20
T4_KS_BASE = 2 ** T4_KS_BASE_BITS

def ks_levels_for_logq(logq: int, base_bits: int = T4_KS_BASE_BITS) -> int:
    return int(math.ceil(logq / base_bits))

T4_KS_LEVELS = ks_levels_for_logq(T4_CKKS_LOGQ, T4_KS_BASE_BITS)


def sparse_secret_to_rns_poly(
    sparse_secret: List[Tuple[int, int]],
    primes: List[int],
    n: int
) -> np.ndarray:
    """Convert sparse ternary secret to RNS polynomial."""
    dense = np.zeros(n, dtype=object)

    for pos, sign in sparse_secret:
        dense[pos] = sign

    return rns_from_int_poly(dense, primes, n)

def sparse_secret_scaled_to_rns_poly(
    sparse_secret: List[Tuple[int, int]],
    scale_factor: int,
    primes: List[int],
    n: int
) -> np.ndarray:
    """
    Convert sparse ternary secret into an RNS polynomial scaled by B^l.

    Used for gadget-style key switching:
        KSK_l = Enc_target(B^l * s_source)
    """
    dense = np.zeros(n, dtype=object)

    for pos, sign in sparse_secret:
        dense[pos] = sign * scale_factor

    return rns_from_int_poly(dense, primes, n)

def rns_gadget_decompose(
    poly: np.ndarray,
    primes: List[int],
    base_bits: int = T4_KS_BASE_BITS,
    levels: int = T4_KS_LEVELS
) -> List[np.ndarray]:
    """
    Gadget decomposition of an RNS polynomial.

    Returns digit polynomials D_0, ..., D_{levels-1}.
    This is a prototype coefficient-wise base decomposition.
    """
    base = 2 ** base_bits
    digits = []

    for level in range(levels):
        digit_level = np.zeros_like(poly)

        for j, p in enumerate(primes):
            coeffs = poly[j].astype(object)
            digit = np.array(
                [int((int(x) // (base ** level)) % base) for x in coeffs],
                dtype=np.int64
            )
            digit_level[j] = digit % p

        digits.append(digit_level)

    return digits


def rns_negacyclic_mul_naive(
    a: np.ndarray,
    b: np.ndarray,
    primes: List[int]
) -> np.ndarray:
    """
    Negacyclic multiplication in Rq = Zq[X]/(X^n + 1).

    This is slow but algebraically clearer than the earlier simulator.
    For full performance, this should later be replaced by NTT.
    """
    n = a.shape[1]
    out = np.zeros_like(a)

    for level, p in enumerate(primes):
        conv = np.convolve(a[level].astype(object), b[level].astype(object))

        reduced = np.zeros(n, dtype=object)
        reduced[:n] += conv[:n]

        if conv.shape[0] > n:
            reduced[:conv.shape[0] - n] -= conv[n:]

        out[level] = np.array([int(x % p) for x in reduced], dtype=np.int64)

    return out

# TADP Admission Thresholds (unused in percentile mode)
TADP_ACCEPT_THRESHOLD = 4.0
TADP_REVIEW_THRESHOLD = 2.5

# Communication expansion factors
PLAIN_COMM_FACTOR = 1.0
BASELINE_CONTROLS_FACTOR = 1.1
DF_KS_FACTOR = 2.5
XMK_CKKS_FACTOR = 50.0
# CKKS_KS_FACTOR = 30.0

# Tier-dependent PoFC packet limits
MAX_PACKETS_PER_NONCE_BY_TIER = {
    1: 50,  # T1
    2: 40,  # T2
    3: 30,  # T3
    4: 20   # T4
}

# Output directories
OUT_DIR = "./tadp_sec_results_v8_professional"
FIG_DIR = os.path.join(OUT_DIR, "figs")
LEDGER_DIR = os.path.join(OUT_DIR, "ledgers")
STATS_DIR = os.path.join(OUT_DIR, "stats")
CRYPTO_DIR = os.path.join(OUT_DIR, "crypto_artifacts")
AUDIT_DIR = os.path.join(OUT_DIR, "audit_trails")

for d in [OUT_DIR, FIG_DIR, LEDGER_DIR, STATS_DIR, CRYPTO_DIR, AUDIT_DIR]:
    os.makedirs(d, exist_ok=True)

# --------------------------
# ATTACK SIMULATION PARAMETERS
# --------------------------
ATTACK_CONFIG = {
    'poisoning': {
        'label_flip_rate': 0.3,  # Flip 30% of labels for malicious clients
        'gradient_scale': -2.0,  # Scale gradients negatively
        'malicious_clients_pct': 0.2,  # 20% of clients are malicious
    },
    'backdoor': {
        'trigger_pattern': 'feature_shift',  # Add pattern to features
        'target_class': 1,  # Trigger causes fraud prediction
        'backdoor_strength': 0.5,
    }
}

# --------------------------
# SIMPLE CSV LOGGER FOR CRYPTO SEMANTICS
# --------------------------
class CryptoLogger:
    """Simple CSV logger for crypto semantics (no hash chaining needed)"""

    def __init__(self, log_path: str):
        self.log_path = log_path
        fieldnames = [
            'timestamp', 'scenario', 'client_id', 'round_id',
            'crypto_mechanism', 'key_tag', 'expanded', 'ks_applied',
            'shares_available', 'shares_required', 'can_finalize',
            'comm_factor', 'crypto_overhead_ms', 'noise_std'
        ]
        with open(log_path, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()

    def log(self, entry: Dict):
        """Log crypto semantics entry"""
        fieldnames = [
            'timestamp', 'scenario', 'client_id', 'round_id',
            'crypto_mechanism', 'key_tag', 'expanded', 'ks_applied',
            'shares_available', 'shares_required', 'can_finalize',
            'comm_factor', 'crypto_overhead_ms', 'noise_std'
        ]

        row = {field: entry.get(field, "") for field in fieldnames}

        with open(self.log_path, 'a', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writerow(row)

# --------------------------
# ENHANCED METRICS COMPUTATION
# --------------------------
def compute_enhanced_metrics(y_true, y_pred, y_pred_proba):
    """Compute comprehensive metrics for imbalanced fraud detection"""
    metrics = {}

    # Basic metrics
    metrics['accuracy'] = accuracy_score(y_true, y_pred)
    metrics['f1'] = f1_score(y_true, y_pred, zero_division=0)
    metrics['precision'] = precision_score(y_true, y_pred, zero_division=0)
    metrics['recall'] = recall_score(y_true, y_pred, zero_division=0)
    metrics['balanced_accuracy'] = balanced_accuracy_score(y_true, y_pred)
    metrics['mcc'] = matthews_corrcoef(y_true, y_pred)

    # ROC-AUC
    try:
        metrics['roc_auc'] = roc_auc_score(y_true, y_pred_proba)
    except:
        metrics['roc_auc'] = 0.5

    # PR-AUC (CRITICAL FOR IMBALANCED DATA)
    try:
        precision, recall, _ = precision_recall_curve(y_true, y_pred_proba)
        metrics['pr_auc'] = average_precision_score(y_true, y_pred_proba)
    except:
        metrics['pr_auc'] = 0.0

    # Recall at specific FPR thresholds
    try:
        from sklearn.metrics import roc_curve
        fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)

        # FIXED: Use consistent key format
        fpr_target = 0.005
        idx = np.argmin(np.abs(fpr - fpr_target))
        metrics['recall_at_fpr_0.005'] = tpr[idx]  # Use exact string 'recall_at_fpr_0.005'

        # Optional: Add 1% FPR too
        fpr_target = 0.01
        idx = np.argmin(np.abs(fpr - fpr_target))
        metrics['recall_at_fpr_0.01'] = tpr[idx]

    except:
        # FIXED: Use the same key
        metrics['recall_at_fpr_0.005'] = 0.0
        metrics['recall_at_fpr_0.01'] = 0.0

    # Confusion matrix components - FIXED (REMOVE DUPLICATE CODE)
    try:
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        metrics['true_negatives'] = tn
        metrics['false_positives'] = fp
        metrics['false_negatives'] = fn
        metrics['true_positives'] = tp

        # Rates
        metrics['false_positive_rate'] = fp / (fp + tn) if (fp + tn) > 0 else 0
        metrics['false_negative_rate'] = fn / (fn + tp) if (fn + tp) > 0 else 0
    except Exception:
        # Fallback for edge cases
        metrics['true_negatives'] = 0
        metrics['false_positives'] = 0
        metrics['false_negatives'] = 0
        metrics['true_positives'] = 0
        metrics['false_positive_rate'] = 0
        metrics['false_negative_rate'] = 0

    # Fraud-specific metrics
    metrics['fraud_recall'] = tp / (tp + fn) if (tp + fn) > 0 else 0

    return metrics

def compute_energy_consumption(
    comm_bytes: float,
    crypto_ms: float,
    model_size_params: int = 29,  # Features in credit card dataset
    energy_per_bit_transmit: float = 5e-9,  # Joules/bit (5 nJ/bit typical for wireless)
    energy_per_flop: float = 1e-9,  # Joules/FLOP (1 nJ/FLOP typical)
    crypto_ops_per_ms: float = 1000,  # Assume 1000 crypto ops per ms
    flops_per_param: float = 2  # FLOPs per parameter per sample
) -> Dict[str, float]:
    """Compute energy consumption and CO2 emissions - returns μJ for small values"""

    # Communication energy (transmit + receive)
    comm_bits = comm_bytes * 8
    comm_energy_joules = comm_bits * energy_per_bit_transmit

    # Computation energy (training + crypto)
    # Training energy (simplified: FLOPs per parameter)
    training_flops = model_size_params * flops_per_param * 1000  # Assume 1000 samples
    training_energy = training_flops * energy_per_flop

    # Crypto energy
    crypto_ops = crypto_ms * crypto_ops_per_ms
    crypto_energy = crypto_ops * energy_per_flop * 10  # Crypto ops are more expensive

    # Total energy in JOULES
    total_energy_joules = comm_energy_joules + training_energy + crypto_energy

    # CO2 emissions (assuming 0.475 kg CO2/kWh for EU grid)
    # Convert joules to kWh: 1 kWh = 3.6e6 J
    energy_kwh = total_energy_joules / 3.6e6
    co2_kg = energy_kwh * 0.475  # kg CO2 per kWh

    # Return BOTH joules and microjoules for convenience
    return {
        # Joules (original)
        'energy_comm_joules': comm_energy_joules,
        'energy_training_joules': training_energy,
        'energy_crypto_joules': crypto_energy,
        'energy_total_joules': total_energy_joules,
        'energy_total_kwh': energy_kwh,
        'co2_emissions_kg': co2_kg,

        # Microjoules (for display)
        'energy_comm_uj': comm_energy_joules * 1e6,  # microjoules
        'energy_training_uj': training_energy * 1e6,
        'energy_crypto_uj': crypto_energy * 1e6,
        'energy_total_uj': total_energy_joules * 1e6,

        # Milligrams for CO2 (more readable)
        'co2_emissions_mg': co2_kg * 1e6,  # milligrams

        'comm_bits': comm_bits,
        'crypto_ops': crypto_ops
    }

def optimize_threshold_for_constraint(y_true, y_pred_proba, constraint_type='fpr', constraint_value=0.001):
    """
    Optimize threshold under a constraint (FPR ≤ X% or Recall ≥ Y%)
    Returns: optimal_threshold, metrics_at_threshold
    """
    from sklearn.metrics import precision_recall_curve, roc_curve

    if constraint_type == 'fpr':
        # Constrain FPR ≤ constraint_value
        fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)

        # Find thresholds where FPR ≤ constraint_value
        valid_idx = np.where(fpr <= constraint_value)[0]
        if len(valid_idx) == 0:
            # No threshold meets constraint, use best F1
            precision, recall, pr_thresholds = precision_recall_curve(y_true, y_pred_proba)
            f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
            best_idx = np.argmax(f1_scores)
            threshold = pr_thresholds[best_idx] if best_idx < len(pr_thresholds) else 0.5
        else:
            # Among valid thresholds, pick one with max TPR (recall)
            best_idx = valid_idx[np.argmax(tpr[valid_idx])]
            threshold = thresholds[best_idx]

    elif constraint_type == 'recall':
        # Constrain Recall ≥ constraint_value
        fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)

        valid_idx = np.where(tpr >= constraint_value)[0]
        if len(valid_idx) == 0:
            # Fallback to max F1
            precision, recall, pr_thresholds = precision_recall_curve(y_true, y_pred_proba)
            f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
            best_idx = np.argmax(f1_scores)
            threshold = pr_thresholds[best_idx] if best_idx < len(pr_thresholds) else 0.5
        else:
            # Among valid thresholds, pick one with min FPR
            best_idx = valid_idx[np.argmin(fpr[valid_idx])]
            threshold = thresholds[best_idx]
    else:
        # Default: maximize F1
        precision, recall, pr_thresholds = precision_recall_curve(y_true, y_pred_proba)
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
        best_idx = np.argmax(f1_scores)
        threshold = pr_thresholds[best_idx] if best_idx < len(pr_thresholds) else 0.5

    # Compute metrics at this threshold
    y_pred = (y_pred_proba >= threshold).astype(int)
    metrics = compute_enhanced_metrics(y_true, y_pred, y_pred_proba)
    metrics['optimal_threshold'] = threshold
    metrics['constraint_type'] = constraint_type
    metrics['constraint_value'] = constraint_value

    return threshold, metrics

# --------------------------
# STATISTICAL ANALYSIS FUNCTIONS - ENHANCED
# --------------------------
def summarize_final_by_seed(results_df: pd.DataFrame) -> pd.DataFrame:
    """
    Summarize FINAL (last-round) metrics per scenario across seeds, including 95% CI for accuracy.
    """
    if results_df is None or results_df.empty:
        return pd.DataFrame()

    required = {"scenario", "seed", "round"}
    missing_req = required - set(results_df.columns)
    if missing_req:
        raise ValueError(f"summarize_final_by_seed: missing required columns: {sorted(missing_req)}")

    df = results_df.copy()
    df["round"] = pd.to_numeric(df["round"], errors="coerce")
    df = df.dropna(subset=["round"])
    if df.empty:
        return pd.DataFrame()

    # Pick the last round per (scenario, seed)
    idx = df.groupby(["scenario", "seed"])["round"].idxmax()
    last_rows = df.loc[idx].reset_index(drop=True)
    if last_rows.empty:
        return pd.DataFrame()

    def mean_ci(x: pd.Series, alpha: float = 0.05) -> dict:
        """t-based CI for the mean; returns dict with ci values."""
        arr = pd.to_numeric(x, errors="coerce").dropna().to_numpy(dtype=float)
        n = arr.size
        if n == 0:
            return {
                "acc_ci_n": 0,
                "acc_ci_mean": np.nan,
                "acc_ci_std": np.nan,
                "acc_ci95_low": np.nan,
                "acc_ci95_high": np.nan
            }

        m = float(arr.mean())
        if n < 2:
            return {
                "acc_ci_n": n,
                "acc_ci_mean": m,
                "acc_ci_std": 0.0,
                "acc_ci95_low": m,
                "acc_ci95_high": m
            }

        s = float(arr.std(ddof=1))
        se = s / np.sqrt(n)
        tcrit = float(stats.t.ppf(1.0 - alpha/2.0, df=n - 1))
        half = tcrit * se
        return {
            "acc_ci_n": n,
            "acc_ci_mean": m,
            "acc_ci_std": s,
            "acc_ci95_low": m - half,
            "acc_ci95_high": m + half
        }

    # Base output: number of seeds per scenario
    out = (
        last_rows.groupby("scenario", as_index=False)
        .agg(n_seeds=("seed", "nunique"))
    )

    # Add simple means/stds (only if columns exist)
    def add_mean_std(out_df: pd.DataFrame, col: str, prefix: str) -> pd.DataFrame:
        if col not in last_rows.columns:
            return out_df
        g = last_rows.groupby("scenario")[col]
        tmp = g.agg(["mean", "std"]).reset_index()
        tmp = tmp.rename(columns={"mean": f"{prefix}_mean", "std": f"{prefix}_std"})
        return out_df.merge(tmp, on="scenario", how="left")

    out = add_mean_std(out, "accuracy", "acc")
    out = add_mean_std(out, "f1", "f1")
    out = add_mean_std(out, "pr_auc", "pr_auc")
    out = add_mean_std(out, "mcc", "mcc")
    out = add_mean_std(out, "crypto_overhead_ms", "crypto_ms")
    out = add_mean_std(out, "effective_clients", "effective_clients")

    # comm_up_bytes -> comm_kb mean
    if "comm_up_bytes" in last_rows.columns:
        tmp = (
            last_rows.groupby("scenario")["comm_up_bytes"]
            .apply(lambda s: pd.to_numeric(s, errors="coerce").dropna().mean() / 1024.0)
            .reset_index(name="comm_kb_mean")
        )
        out = out.merge(tmp, on="scenario", how="left")

    # 95% CI for accuracy (only if present)
    if "accuracy" in last_rows.columns:
        ci_results = []
        for scenario in last_rows["scenario"].unique():
            scenario_data = last_rows[last_rows["scenario"] == scenario]["accuracy"]
            ci = mean_ci(scenario_data)
            ci_results.append({"scenario": scenario, **ci})

        ci_df = pd.DataFrame(ci_results)
        out = out.merge(ci_df, on="scenario", how="left")

    # Add CI for PR-AUC if present
    if "pr_auc" in last_rows.columns:
        ci_results_pr = []
        for scenario in last_rows["scenario"].unique():
            scenario_data = last_rows[last_rows["scenario"] == scenario]["pr_auc"]
            ci = mean_ci(scenario_data)
            ci_results_pr.append({"scenario": scenario,
                                  "pr_auc_ci_mean": ci["acc_ci_mean"],
                                  "pr_auc_ci_std": ci["acc_ci_std"],
                                  "pr_auc_ci95_low": ci["acc_ci95_low"],
                                  "pr_auc_ci95_high": ci["acc_ci95_high"]})

        ci_df_pr = pd.DataFrame(ci_results_pr)
        out = out.merge(ci_df_pr, on="scenario", how="left")

    # Optional: stable column order
    preferred = [
        "scenario", "n_seeds",
        "acc_mean", "acc_std", "acc_ci95_low", "acc_ci95_high",
        "pr_auc_mean", "pr_auc_std", "pr_auc_ci95_low", "pr_auc_ci95_high",
        "f1_mean", "f1_std",
        "mcc_mean", "mcc_std",
        "comm_kb_mean",
        "crypto_ms_mean", "crypto_ms_std",
        "effective_clients_mean", "effective_clients_std",
    ]
    cols = [c for c in preferred if c in out.columns] + [c for c in out.columns if c not in preferred]
    return out[cols]

def paired_test_vs_baseline(results_df: pd.DataFrame, baseline: str = "B0_Plain_FL") -> pd.DataFrame:
    """Run paired t-tests comparing each scenario to baseline"""
    if results_df.empty:
        return pd.DataFrame()

    # Take LAST round per (scenario, seed)
    last_rows = (results_df.sort_values("round")
                 .groupby(["scenario", "seed"], as_index=False)
                 .tail(1))

    # Check if we have multiple seeds
    if len(last_rows['seed'].unique()) < 2:
        print("  WARNING: Need at least 2 seeds for statistical tests")
        return pd.DataFrame()

    # Check if baseline exists
    if baseline not in last_rows['scenario'].values:
        print(f"  WARNING: Baseline {baseline} not found in results")
        return pd.DataFrame()

    base = last_rows[last_rows["scenario"] == baseline][["seed", "pr_auc"]].rename(
        columns={"pr_auc": "pr_auc_base"})

    out_rows = []

    for scenario in sorted(last_rows["scenario"].unique()):
        if scenario == baseline:
            continue

        cur = last_rows[last_rows["scenario"] == scenario][["seed", "pr_auc"]].rename(
            columns={"pr_auc": "pr_auc_s"})
        merged = base.merge(cur, on="seed", how="inner")

        if len(merged) < 2:
            continue

        # Paired t-test on differences
        diffs = merged["pr_auc_s"] - merged["pr_auc_base"]
        tstat, pval = stats.ttest_rel(merged["pr_auc_s"], merged["pr_auc_base"])

        # Effect size (Cohen's d)
        mean_diff = diffs.mean()
        std_diff = diffs.std(ddof=1)
        if std_diff > 0:
            cohens_d = mean_diff / std_diff
        else:
            cohens_d = 0

        out_rows.append({
            "scenario": scenario,
            "n": len(merged),
            "mean_diff": mean_diff,
            "std_diff": std_diff,
            "t_statistic": tstat,
            "p_value": pval,
            "significant_05": pval < 0.05,
            "significant_01": pval < 0.01,
            "cohens_d": cohens_d,
            "effect_size": "negligible" if abs(cohens_d) < 0.2 else
                          "small" if abs(cohens_d) < 0.5 else
                          "medium" if abs(cohens_d) < 0.8 else "large"
        })

    return pd.DataFrame(out_rows).sort_values("p_value")

# --------------------------
# ATTACK SIMULATION MODULES
# --------------------------
class AttackSimulator:
    """Simulate various attacks to evaluate security tiers"""

    def __init__(self, config=None):
        self.config = config or ATTACK_CONFIG

    def label_flip_poisoning(self, X: np.ndarray, y: np.ndarray, flip_rate: float = 0.3) -> Tuple[np.ndarray, np.ndarray]:
        """Simulate label-flipping poisoning attack"""
        poisoned_y = y.copy()
        n_flip = int(len(y) * flip_rate)

        if n_flip > 0:
            flip_indices = np.random.choice(len(y), n_flip, replace=False)
            # Flip labels: 0->1, 1->0
            poisoned_y[flip_indices] = 1 - poisoned_y[flip_indices]

        return X, poisoned_y

    def gradient_scale_attack(self, gradient: np.ndarray, scale_factor: float = -2.0) -> np.ndarray:
        """Scale gradients negatively to disrupt learning"""
        return gradient * scale_factor

    def backdoor_trigger(self, X: np.ndarray, trigger_strength: float = None) -> np.ndarray:
        """Add backdoor trigger pattern to features (SAME as training)"""
        if trigger_strength is None:
            trigger_strength = ATTACK_CONFIG['backdoor']['backdoor_strength']

        X_backdoor = X.copy()

        # Apply sinusoidal pattern to first 3 features (matching training)
        n_samples = X.shape[0]
        for i in range(min(3, X.shape[1])):
            X_backdoor[:, i] += trigger_strength * np.sin(
                np.arange(n_samples) * 0.1 + i * np.pi/3
            )

        return X_backdoor

    def evaluate_attack_success(self, model, X_test, y_test, X_backdoor=None,
                              backdoor_target=1) -> Dict[str, float]:
        """Evaluate REAL backdoor attack success with proper metrics"""
        metrics = {}

        # 1. Clean accuracy (normal operation)
        y_pred_proba_clean = model.predict_proba(X_test)
        y_pred_clean = (y_pred_proba_clean >= 0.5).astype(int)
        clean_metrics = compute_enhanced_metrics(y_test, y_pred_clean, y_pred_proba_clean)
        metrics['clean_pr_auc'] = clean_metrics['pr_auc']
        metrics['clean_accuracy'] = clean_metrics['accuracy']
        metrics['clean_f1'] = clean_metrics['f1']

        # 2. REAL Backdoor success rate evaluation
        if X_backdoor is not None:
            # Apply the SAME trigger used during training
            X_backdoored = self.backdoor_trigger(X_test,
                                              trigger_strength=ATTACK_CONFIG['backdoor']['backdoor_strength'])

            # Predict on backdoored samples
            y_pred_proba_backdoor = model.predict_proba(X_backdoored)
            y_pred_backdoor = (y_pred_proba_backdoor >= 0.5).astype(int)

            # Attack Success Rate (ASR): % of backdoored samples classified as target
            backdoor_success = np.mean(y_pred_backdoor == backdoor_target)

            # Baseline: clean samples classified as target (without trigger)
            baseline_target_rate = np.mean(y_pred_clean == backdoor_target)

            # Delta ASR: increase due to backdoor
            asr_delta = backdoor_success - baseline_target_rate

            # Backdoor effectiveness: success rate on ONLY the backdoored portion
            # We can't identify which test samples would have triggers, so use whole set

            metrics['backdoor_success_rate'] = backdoor_success
            metrics['asr_delta'] = asr_delta
            metrics['baseline_target_rate'] = baseline_target_rate

            # Additional backdoor metrics
            if len(y_test) > 0:
                # Precision on backdoored predictions
                backdoor_precision = np.sum((y_pred_backdoor == backdoor_target) &
                                          (y_pred_clean != backdoor_target)) / \
                                  max(np.sum(y_pred_backdoor == backdoor_target), 1)
                metrics['backdoor_precision'] = backdoor_precision

                # Stealth metric: how much does backdoor affect clean accuracy?
                accuracy_drop = metrics['clean_accuracy'] - accuracy_score(y_test, y_pred_backdoor)
                metrics['accuracy_drop'] = accuracy_drop

                # Specificity: ability to maintain correct classification of non-target class
                non_target = 1 - backdoor_target
                specificity = np.sum((y_pred_backdoor == non_target) &
                                  (y_test == non_target)) / max(np.sum(y_test == non_target), 1)
                metrics['backdoor_specificity'] = specificity

        # 3. Label flipping attack effectiveness (if applicable)
        # We measure this by comparing model behavior on clean vs poisoned validation set
        # (You could add a separate validation set with flipped labels)

        return metrics

# --------------------------
# CRYPTOGRAPHIC SEMANTICS SIMULATOR (COMPLETE)
# --------------------------
# ============================================================
# T4: ALGEBRAIC CKKS + DOUBLE KEY-SWITCHING WORKFLOW
# ============================================================

CKKSDoubleKSCiphertext = AlgebraicRLWECiphertext


class SimulatedCKKSDoubleKSCrypto:
    """
    Algebraic CKKS/RLWE double key-switching prototype.

    Workflow:
    client key -> aggregation key -> client key

    This version uses prototype gadget-style key-switching keys:
        KSK_l = Enc_target(B^l * s_source)

    It is still a research prototype, not production HE.
    """
    def __init__(
        self,
        n: int = T4_CKKS_POLY_MODULUS_DEGREE,
        logq: int = T4_CKKS_LOGQ,
        scale: int = T4_CKKS_SCALE,
        noise_std: float = T4_CKKS_NOISE_STD
    ):
        self.engine = AlgebraicCKKSRLWEEngine(
            n=n,
            logq=logq,
            scale=scale,
            noise_std=noise_std,
            seed_tag="T4_DOUBLE_KS"
        )

        self.n = n
        self.logq = logq
        self.scale = scale
        self.noise_std = noise_std
        self.client_keypairs = {}

        self.aggregation_keypair = self.engine.keygen("aggregation_session_key")
        self.aggregation_key_id = "aggregation_session_key"

        self.ksk_client_to_agg = {}
        self.ksk_agg_to_client = {}

    def enroll_client(self, client_id: str) -> Dict[str, str]:
        kp = self.engine.keygen(f"client:{client_id}")
        self.client_keypairs[client_id] = kp

        self.ksk_client_to_agg[client_id] = hashlib.sha256(
            f"KSK:{client_id}->AGG".encode()
        ).hexdigest()[:16]

        self.ksk_agg_to_client[client_id] = hashlib.sha256(
            f"KSK:AGG->{client_id}".encode()
        ).hexdigest()[:16]

        # Formal prototype key-switching keys
        # client key -> aggregation key
        kp_client = self.client_keypairs[client_id]

        self.ksk_client_to_agg[(client_id, "formal")] = self.generate_key_switch_key(
            source_sk=kp_client["sk"],
            target_sk=self.aggregation_keypair["sk"],
            source_name=f"client:{client_id}",
            target_name="aggregation_session"
        )

        self.ksk_agg_to_client[(client_id, "formal")] = self.generate_key_switch_key(
            source_sk=self.aggregation_keypair["sk"],
            target_sk=kp_client["sk"],
            source_name="aggregation_session",
            target_name=f"client:{client_id}"
        )

        return {
            "client_id": client_id,
            "pk": f"pk:{client_id}",
            "sk_ref": f"client_private_key_only:{client_id}",
            "ksk_i_to_A": self.ksk_client_to_agg[client_id],
            "ksk_A_to_i": self.ksk_agg_to_client[client_id],
            "ring": "Rq=Zq[X]/(X^n+1)"
        }

    def generate_key_switch_key(
        self,
        source_sk: List[Tuple[int, int]],
        target_sk: List[Tuple[int, int]],
        source_name: str,
        target_name: str
    ) -> List[AlgebraicRLWECiphertext]:
        """
        Generate formal prototype gadget-style key-switching keys.

        KSK_l encrypts:

            B^l * s_source

        under the target key domain.

        In this prototype, the evaluation-key ciphertexts are generated exactly
        under target_sk to avoid huge noise amplification during digit multiplication.
        """

        levels = ks_levels_for_logq(self.engine.logq, T4_KS_BASE_BITS)
        base = 2 ** T4_KS_BASE_BITS

        ksk = []

        for level in range(levels):
            scale_factor = base ** level

            s_scaled = sparse_secret_scaled_to_rns_poly(
                sparse_secret=source_sk,
                scale_factor=scale_factor,
                primes=self.engine.primes,
                n=self.engine.n
            )

            evk_l = self.engine.encrypt_rns_plain_under_sk_exact(
                target_sk=target_sk,
                m_rns=s_scaled,
                client_id=f"ksk:{source_name}->{target_name}",
                key_tag=f"ksk:{source_name}->{target_name}",
                scheme="ckks_exact_gadget_key_switch_key"
            )

            evk_l.metadata.update({
                "ks_level": level,
                "source_key": source_name,
                "target_key": target_name,
                "base_bits": T4_KS_BASE_BITS,
                "scale_factor": scale_factor,
                "ksk_type": "exact_formal_prototype"
            })

            ksk.append(evk_l)

        return ksk

# NOTE:
# Formal gadget-style KSK helpers are retained for documentation and future extension.
# NOTE:
# The active T4 runtime can use either:
# - formal KSK-based switching when T4_KS_MODE = "formal"
# - semantic decrypt/re-encrypt fallback when T4_KS_MODE = "semantic"
# to preserve plaintext correctness in this Python prototype.
    def apply_key_switch_key(
        self,
        ct: AlgebraicRLWECiphertext,
        ksk: List[AlgebraicRLWECiphertext],
        target_key_tag: str,
        target_client_id: str
    ) -> AlgebraicRLWECiphertext:
        """
        Formal prototype gadget-style key switching.

        Input ciphertext under source key:
            ct = (c0, c1)

        Evaluation/key-switching key:
            KSK_l = Enc_target(B^l * s_source)

        Approximate transformation:
            c1 = Σ_l d_l * B^l

            ct' = (c0, 0) + Σ_l d_l * KSK_l

        Output ciphertext decrypts under the target key.
        """

        if not ksk:
            raise ValueError("Missing formal key-switching key.")

        # Decompose c1 into gadget digits.
        digits = rns_gadget_decompose(
            ct.c1,
            self.engine.primes,
            base_bits=T4_KS_BASE_BITS,
            levels=len(ksk)
        )

        # Start with (c0, 0)
        c0_new = ct.c0.copy()
        c1_new = rns_zero(self.engine.primes, self.engine.n)

        total_noise_sq = ct.cumulative_noise ** 2

        # Add Σ_l d_l * KSK_l
        for level, (digit_poly, evk_l) in enumerate(zip(digits, ksk)):
            term = self.engine.plaintext_mul_ciphertext(
                digit_poly=digit_poly,
                ct=evk_l
            )

            c0_new = rns_add(c0_new, term.c0, self.engine.primes)
            c1_new = rns_add(c1_new, term.c1, self.engine.primes)

            total_noise_sq += term.cumulative_noise ** 2

        switched = AlgebraicRLWECiphertext(
            c0=c0_new,
            c1=c1_new,
            original_shape=ct.original_shape,
            client_id=target_client_id,
            key_tag=target_key_tag,
            n=self.engine.n,
            logq=self.engine.logq,
            scale=self.engine.scale,
            primes=self.engine.primes,
            scheme="ckks_double_ks_rlwe_formal",
            noise_std=self.engine.noise_std / self.engine.scale,
            cumulative_noise=math.sqrt(total_noise_sq),
            ks_count=ct.ks_count + 1,
            metadata={
                **ct.metadata,
                "operation": "formal_gadget_key_switch",
                "target_key": target_key_tag,
                "ks_levels": len(ksk),
                "base_bits": T4_KS_BASE_BITS,
                "key_switch_equation": "ct_prime=(c0,0)+sum_l Decomp_l(c1)*KSK_l",
                "prototype_note": (
                    "Formal prototype KSK-based switching; no plaintext decrypt/re-encrypt "
                    "inside the active formal path."
                )
            }
        )

        return switched

    def encrypt_client_update(
        self,
        client_id: str,
        update_vec: np.ndarray
    ) -> CKKSDoubleKSCiphertext:
        if client_id not in self.client_keypairs:
            raise ValueError(f"Client {client_id} not enrolled.")

        pk = self.client_keypairs[client_id]["pk"]

        return self.engine.encrypt(
            pk=pk,
            vec=update_vec,
            client_id=client_id,
            key_tag=f"client:{client_id}",
            scheme="ckks_double_ks_rlwe"
        )

    def key_switch_to_aggregation(self, ct: CKKSDoubleKSCiphertext) -> CKKSDoubleKSCiphertext:
        cid = ct.client_id

        if T4_KS_MODE == "formal":
            formal_ksk = self.ksk_client_to_agg.get((cid, "formal"))

            if formal_ksk is None:
                raise ValueError(f"Missing formal KSK client->aggregation for {cid}")

            switched = self.apply_key_switch_key(
                ct=ct,
                ksk=formal_ksk,
                target_key_tag="aggregation_session",
                target_client_id=cid
            )

            switched.metadata.update({
                "operation": "formal_key_switch_client_to_aggregation",
                "source_key": f"client:{cid}",
                "target_key": "aggregation_session"
            })

            return switched

        elif T4_KS_MODE == "semantic":
            # Stable fallback: decrypt/re-encrypt semantic key switching.
            plain = self.engine.decrypt(
                self.client_keypairs[cid]["sk"],
                ct
            )

            switched = self.engine.encrypt(
                pk=self.aggregation_keypair["pk"],
                vec=plain,
                client_id=cid,
                key_tag="aggregation_session",
                scheme="ckks_double_ks_rlwe"
            )

            switched.ks_count = ct.ks_count + 1
            switched.cumulative_noise = math.sqrt(
                ct.cumulative_noise ** 2 + self.engine.noise_std ** 2
            )

            switched.metadata.update({
                **ct.metadata,
                "operation": "semantic_key_switch_client_to_aggregation",
                "source_key": f"client:{cid}",
                "target_key": "aggregation_session",
                "prototype_note": "Decrypt/re-encrypt fallback"
            })

            return switched

        else:
            raise ValueError(f"Unknown T4_KS_MODE: {T4_KS_MODE}")

    def aggregate_under_aggregation_key(
        self,
        cts: List[CKKSDoubleKSCiphertext]
    ) -> CKKSDoubleKSCiphertext:
        for ct in cts:
            if ct.key_tag != "aggregation_session":
                raise ValueError("All T4 ciphertexts must be under aggregation_session key.")

        return self.engine.add_ciphertexts(
            cts,
            key_tag="aggregation_session",
            client_id="server_aggregate"
        )

    def key_switch_to_client(
        self,
        ct_agg: CKKSDoubleKSCiphertext,
        target_client_id: str
    ) -> CKKSDoubleKSCiphertext:

        if T4_KS_MODE == "formal":
            formal_ksk = self.ksk_agg_to_client.get((target_client_id, "formal"))

            if formal_ksk is None:
                raise ValueError(
                    f"Missing formal KSK aggregation->client for {target_client_id}"
                )

            switched = self.apply_key_switch_key(
                ct=ct_agg,
                ksk=formal_ksk,
                target_key_tag=f"client:{target_client_id}",
                target_client_id=target_client_id
            )

            switched.metadata.update({
                "operation": "formal_key_switch_aggregation_to_client",
                "source_key": "aggregation_session",
                "target_key": f"client:{target_client_id}"
            })

            return switched

        elif T4_KS_MODE == "semantic":
            # Stable fallback: decrypt/re-encrypt semantic key switching.
            plain = self.engine.decrypt(
                self.aggregation_keypair["sk"],
                ct_agg
            )

            switched = self.engine.encrypt(
                pk=self.client_keypairs[target_client_id]["pk"],
                vec=plain,
                client_id=target_client_id,
                key_tag=f"client:{target_client_id}",
                scheme="ckks_double_ks_rlwe"
            )

            switched.ks_count = ct_agg.ks_count + 1
            switched.cumulative_noise = math.sqrt(
                ct_agg.cumulative_noise ** 2 + self.engine.noise_std ** 2
            )

            switched.metadata.update({
                **ct_agg.metadata,
                "operation": "semantic_key_switch_aggregation_to_client",
                "source_key": "aggregation_session",
                "target_key": f"client:{target_client_id}",
                "prototype_note": "Decrypt/re-encrypt fallback"
            })

            return switched

        else:
            raise ValueError(f"Unknown T4_KS_MODE: {T4_KS_MODE}")

    def decrypt_client_result(
        self,
        client_id: str,
        ct_client: CKKSDoubleKSCiphertext
    ) -> np.ndarray:
        if client_id not in self.client_keypairs:
            raise ValueError(f"Unknown client {client_id}")

        if ct_client.key_tag != f"client:{client_id}":
            raise ValueError(
                f"Ciphertext is not decryptable by {client_id}. "
                f"Current key tag: {ct_client.key_tag}"
            )

        return self.engine.decrypt(
            self.client_keypairs[client_id]["sk"],
            ct_client
        )

    def noise_budget_bits(self, ct: CKKSDoubleKSCiphertext) -> float:
        if ct.cumulative_noise <= 0:
            return float(self.logq)

        return self.logq - math.log2(max(1.0, ct.cumulative_noise)) - math.log2(self.scale)

# --------------------------
# BASE CRYPTO SIMULATOR FOR B0/B1/T1/T2/T3
# --------------------------
@dataclass
class Ciphertext:
    """
    Generic custom ciphertext object for B0/B1/T1/T2/T3.

    The value field is the authorized simulation payload used after valid aggregation.
    c0/c1/components store ciphertext-like structures so the protocol no longer looks
    like plain value passing.
    """
    value: np.ndarray
    key_tag: str = "plain"
    expanded: bool = False
    noise_std: float = 0.0
    cumulative_noise: float = 0.0
    client_id: str = ""
    original_shape: tuple = None
    scheme: str = "plain"
    c0: Optional[np.ndarray] = None
    c1: Optional[np.ndarray] = None
    components: Optional[np.ndarray] = None
    dimension: int = 1
    ciphertext_size_bytes: int = 0
    metadata: Dict[str, Any] = field(default_factory=dict)

    def __post_init__(self):
        if self.original_shape is None:
            self.original_shape = self.value.shape

    def get_size_bytes(self) -> int:
        if self.ciphertext_size_bytes and self.ciphertext_size_bytes > 0:
            return int(self.ciphertext_size_bytes)
        return int(np.asarray(self.value).size * 8)

@dataclass
class DFKey:
    key_id: str
    r: np.ndarray          # secret invertible keys r_i
    r_inv: np.ndarray      # modular inverses r_i^{-1}


@dataclass
class DFCiphertext:
    components: np.ndarray     # shape: (dimension, vector_length), dtype=object
    key_id: str
    dimension: int
    original_shape: tuple
    scale: int
    m0: int
    m: int
    expanded: bool = False
    metadata: Dict[str, Any] = field(default_factory=dict)

    def get_size_bytes(self) -> int:
        return int(self.components.size * math.ceil(self.m.bit_length() / 8))


class ModifiedDFKSEngine:
    """
    Modified Domingo-Ferrer encryption + matrix-based key switching.

    Implements:
    - Modified DF encryption using d independent invertible secret keys.
    - Homomorphic addition.
    - Homomorphic multiplication with expansion to N = C(d,2) + d.
    - Key switching c_fresh = M c_star, reducing dimension N back to d.
    """

    def __init__(
        self,
        d: int = DF_D,
        scale: int = DF_SCALE,
        m0: int = DF_M0,
        m: int = DF_M,
        noise_bound: int = DF_KS_NOISE_BOUND
    ):
        self.d = int(d)
        self.scale = int(scale)
        self.m0 = int(m0)
        self.m = int(m)
        self.noise_bound = int(noise_bound)

        if self.m <= self.m0:
            raise ValueError("DF_M must be larger than DF_M0.")

    def _rand_mod_nonzero(self) -> int:
        while True:
            x = secrets.randbelow(self.m - 1) + 1
            if math.gcd(x, self.m) == 1:
                return x

    def _rand_small_noise(self) -> int:
        return secrets.randbelow(2 * self.noise_bound + 1) - self.noise_bound

    def keygen(self, key_id: str) -> DFKey:
        r = np.array([self._rand_mod_nonzero() for _ in range(self.d)], dtype=object)
        r_inv = np.array([pow(int(x), -1, self.m) for x in r], dtype=object)
        return DFKey(key_id=key_id, r=r, r_inv=r_inv)

    def encode(self, vec: np.ndarray) -> np.ndarray:
        x = np.asarray(vec, dtype=float).reshape(-1)
        z = np.round(x * self.scale).astype(object)
        return np.array([int(v) % self.m0 for v in z], dtype=object)

    def decode(
        self,
        residues: np.ndarray,
        shape: tuple,
        scale: Optional[int] = None
    ) -> np.ndarray:
        """
        Decode modular residues back to real values.

        Important:
        - Plain DF encryption uses scale = DF_SCALE.
        - Homomorphic multiplication uses scale = DF_SCALE².
        """
        used_scale = int(scale if scale is not None else self.scale)

        out = []
        half = self.m0 // 2

        for v in np.asarray(residues, dtype=object).reshape(-1):
            vi = int(v) % self.m0

            if vi > half:
                vi -= self.m0

            out.append(vi / used_scale)

        return np.asarray(out, dtype=float).reshape(shape)

    def _decompose_encoded(self, encoded: np.ndarray) -> np.ndarray:
        """
        Decompose each plaintext a into d shares a_1,...,a_d such that:
            sum_j a_j mod m0 = a
        """
        encoded = np.asarray(encoded, dtype=object).reshape(-1)
        L = encoded.size
        shares = np.zeros((self.d, L), dtype=object)

        for k, a in enumerate(encoded):
            partial_sum = 0
            for j in range(self.d - 1):
                share = secrets.randbelow(self.m0)
                shares[j, k] = share
                partial_sum = (partial_sum + share) % self.m0

            shares[self.d - 1, k] = (int(a) - partial_sum) % self.m0

        return shares

    def encrypt_vector(self, vec: np.ndarray, key: DFKey) -> DFCiphertext:
        encoded = self.encode(vec)
        shares = self._decompose_encoded(encoded)

        # π_i = a_i r_i mod m
        components = (shares * key.r[:, None]) % self.m

        return DFCiphertext(
            components=components.astype(object),
            key_id=key.key_id,
            dimension=self.d,
            original_shape=np.asarray(vec).shape,
            scale=self.scale,
            m0=self.m0,
            m=self.m,
            expanded=False,
            metadata={
                "scheme": "modified_df",
                "scale": self.scale
            }
        )

    def decrypt_vector(self, ct: DFCiphertext, key: DFKey) -> np.ndarray:
        if ct.dimension != self.d:
            raise ValueError(
                "Use decrypt_expanded() or key-switch before decrypting expanded DF ciphertext."
            )

        # First recover in Z_m, then reduce modulo m0.
        recovered_mod_m = (
            np.sum((ct.components * key.r_inv[:, None]) % self.m, axis=0) % self.m
        )

        # KS adds m0 * e, so reducing modulo m0 removes the LWE-style noise term.
        recovered = recovered_mod_m % self.m0

        return self.decode(
            recovered,
            ct.original_shape,
            scale=ct.scale
        )

    def add(self, cts: List[DFCiphertext], key_id: str) -> DFCiphertext:
        if not cts:
            raise ValueError("No DF ciphertexts to add.")

        dim = cts[0].dimension
        shape = cts[0].original_shape
        scale = int(cts[0].scale)

        total = np.zeros_like(cts[0].components, dtype=object)

        for ct in cts:
            if ct.dimension != dim:
                raise ValueError("Cannot add DF ciphertexts with different dimensions.")

            if int(ct.scale) != scale:
                raise ValueError(
                    f"Cannot add DF ciphertexts with different scales: {ct.scale} vs {scale}"
                )

            total = (total + ct.components) % self.m

        return DFCiphertext(
            components=total.astype(object),
            key_id=key_id,
            dimension=dim,
            original_shape=shape,
            scale=scale,
            m0=self.m0,
            m=self.m,
            expanded=cts[0].expanded,
            metadata={
                "scheme": "modified_df_add",
                "n_aggregated": len(cts),
                "scale": scale
            }
        )

    def _extended_secret_inverse(self, key: DFKey) -> np.ndarray:
        """
        s_star^{-1} for expanded multiplication ciphertext:
        [r_1^-2,...,r_d^-2, r_1^-1 r_2^-1, ..., r_{d-1}^-1 r_d^-1]
        """
        inv = key.r_inv
        terms = []

        for i in range(self.d):
            terms.append((inv[i] * inv[i]) % self.m)

        for i in range(self.d):
            for j in range(i + 1, self.d):
                terms.append((inv[i] * inv[j]) % self.m)

        return np.array(terms, dtype=object)

    def multiply_expand(self, ct1: DFCiphertext, ct2: DFCiphertext, key: DFKey) -> Tuple[DFCiphertext, np.ndarray]:
        """
        Homomorphic multiplication. Output dimension:
            N = C(d,2) + d
        """
        if ct1.dimension != self.d or ct2.dimension != self.d:
            raise ValueError("DF multiplication expects compact ciphertexts of dimension d.")

        if ct1.original_shape != ct2.original_shape:
            raise ValueError("DF multiplication requires same plaintext shape.")

        terms = []

        # Diagonal terms: c_i * c_i'
        for i in range(self.d):
            terms.append((ct1.components[i] * ct2.components[i]) % self.m)

        # Cross terms: c_i*c_j' + c_j*c_i'
        for i in range(self.d):
            for j in range(i + 1, self.d):
                cross = (
                    ct1.components[i] * ct2.components[j]
                    + ct1.components[j] * ct2.components[i]
                ) % self.m
                terms.append(cross)

        expanded_components = np.vstack(terms).astype(object)
        s_star_inv = self._extended_secret_inverse(key)

        expected_N = (self.d * (self.d - 1)) // 2 + self.d
        assert expanded_components.shape[0] == expected_N, (
            f"DF multiplication dimension mismatch: got {expanded_components.shape[0]}, expected {expected_N}"
        )

        return DFCiphertext(
            components=expanded_components,
            key_id=f"{key.key_id}:expanded",
            dimension=expected_N,
            original_shape=ct1.original_shape,
            scale=int(ct1.scale) * int(ct2.scale),
            m0=self.m0,
            m=self.m,
            expanded=True,
            metadata={
                "scheme": "modified_df_expanded_multiplication",
                "left_scale": int(ct1.scale),
                "right_scale": int(ct2.scale),
                "product_scale": int(ct1.scale) * int(ct2.scale)
            }
        ), s_star_inv

    def make_ks_matrix(self, s_star_inv: np.ndarray, fresh_key: DFKey) -> np.ndarray:
        """
        Build public KS matrix M = [b; A] of shape (d, N), where:
            c_fresh = M c_star

        It satisfies:
            s0_inv M c_star = s_star_inv c_star mod m0
        """
        s_star_inv = np.asarray(s_star_inv, dtype=object).reshape(-1)
        N = s_star_inv.size

        A = np.array(
            [[secrets.randbelow(self.m) for _ in range(N)] for _ in range(self.d - 1)],
            dtype=object
        )

        e = np.array([self._rand_small_noise() for _ in range(N)], dtype=object)

        # t0 = [r_2'^-1, ..., r_d'^-1]
        t0 = fresh_key.r_inv[1:].reshape(1, self.d - 1)

        # b = (-t0 A + m0 e + s_star_inv) * r_1'
        t0A = (t0 @ A).reshape(-1) % self.m
        b = ((-t0A + self.m0 * e + s_star_inv) * fresh_key.r[0]) % self.m

        M = np.vstack([b.reshape(1, N), A]).astype(object)

        assert M.shape == (self.d, N), f"KS matrix shape mismatch: {M.shape}, expected {(self.d, N)}"
        return M

    def key_switch(self, ct_star: DFCiphertext, M: np.ndarray, fresh_key: DFKey) -> DFCiphertext:
        if not ct_star.expanded:
            raise ValueError("DF key switching expects an expanded ciphertext.")

        if M.shape[1] != ct_star.dimension:
            raise ValueError("KS matrix column count must match expanded ciphertext dimension.")

        compact_components = (M @ ct_star.components) % self.m

        return DFCiphertext(
            components=compact_components.astype(object),
            key_id=fresh_key.key_id,
            dimension=self.d,
            original_shape=ct_star.original_shape,
            scale=ct_star.scale,
            m0=self.m0,
            m=self.m,
            expanded=False,
            metadata={
                "scheme": "modified_df_key_switched",
                "ks_applied": True,
                "source_dimension": ct_star.dimension,
                "target_dimension": self.d,
                "scale": ct_star.scale
            }
        )

class CryptoSim:
    """
    Custom reference-aligned crypto simulator for non-T4 scenarios.

    T2:
    - Domingo-Ferrer-style decomposition.
    - Encrypted scalar weighting.
    - Multiplication expansion.
    - Key-switch compaction.

    T3/B1:
    - xMK-CKKS-style aggregated public key.
    - Clients encrypt with aggregated public key.
    - Server can finalize only after all valid decryption shares are available.
    """

    def __init__(self):
        self.xmk_shares = defaultdict(dict)
        self.xmk_session_clients = []
        self.xmk_client_keys = {}
        self.xmk_aggregated_pk_tag = None

        self.df_engine = ModifiedDFKSEngine()
        self.df_client_keys = {}
        self.df_server_key = self.df_engine.keygen("df_server_or_authorized_aggregate_key")
        self.df_server_key_tag = self.df_server_key.key_id

    def _hash_tag(self, text: str) -> str:
        return hashlib.sha256(text.encode()).hexdigest()[:16]

    # --------------------------
    # Plain
    # --------------------------
    def encrypt_plain(self, update_vec: np.ndarray) -> Ciphertext:
        return Ciphertext(
            value=np.asarray(update_vec, dtype=float),
            key_tag="plain",
            expanded=False,
            noise_std=0.0,
            cumulative_noise=0.0,
            scheme="plain"
        )

    # --------------------------
    # xMK-CKKS algebraic support
    # --------------------------
    def prepare_xmk_session(self, client_ids: List[str]):
        """
        Algebraic xMK-CKKS session.

        Each client has s_i.
        All clients share common a.
        Aggregated public key:
            b_agg = sum_i b_i
            a_common = a

        Encryption uses pk_agg.
        Decryption requires partial shares c1*s_i from valid participants.
        """
        self.xmk_session_clients = list(client_ids)
        self.xmk_client_keys = {}

        self.xmk_engine = AlgebraicCKKSRLWEEngine(
            n=XMK_CKKS_POLY_MODULUS_DEGREE,
            logq=XMK_CKKS_LOGQ,
            scale=XMK_CKKS_SCALE,
            noise_std=XMK_CKKS_NOISE_STD,
            seed_tag="T3_XMK_CKKS"
        )

        rng = self.xmk_engine._rng("xmk_common_a")
        common_a = rns_uniform(
            self.xmk_engine.primes,
            self.xmk_engine.n,
            rng
        )

        b_agg = rns_zero(self.xmk_engine.primes, self.xmk_engine.n)

        for cid in self.xmk_session_clients:
            kp = self.xmk_engine.keygen(
                key_id=f"xmk:{cid}",
                common_a=common_a
            )
            self.xmk_client_keys[cid] = kp
            b_agg = rns_add(b_agg, kp["pk"]["b"], self.xmk_engine.primes)

        self.xmk_aggregated_pk = {
            "b": b_agg,
            "a": common_a,
            "key_id": "xmk_aggregated_public_key"
        }

        self.xmk_aggregated_pk_tag = hashlib.sha256(
            "|".join(self.xmk_session_clients).encode()
        ).hexdigest()[:16]


    def reset_xmk_round(self, round_id: int):
        self.xmk_shares[round_id] = {}


    def collect_xmk_decryption_shares(
        self,
        agg_ct: Ciphertext,
        round_id: int,
        required_clients: List[str]
    ):
        """
        Collect xMK decryption shares after aggregation.

        Each client i computes:
            D_i = s_i * Csum1 + e_i*
        """
        if agg_ct.scheme != "alg_xmk_ckks":
            raise ValueError("xMK share collection requires alg_xmk_ckks aggregate ciphertext.")

        if "c1_aggregate" not in agg_ct.metadata:
            raise ValueError("Missing Csum1 for xMK share collection.")

        c1_aggregate = agg_ct.metadata["c1_aggregate"]

        self.xmk_shares[round_id] = {}

        for cid in required_clients:
            if cid not in self.xmk_client_keys:
                raise ValueError(f"Client {cid} is missing xMK key material.")

            share_i = self.xmk_partial_decryption_share(
                c1_aggregate=c1_aggregate,
                client_id=cid,
                round_id=round_id
            )

            self.xmk_shares[round_id][cid] = share_i


    def encrypt_xmk(self, update_vec: np.ndarray, client_id: str) -> Ciphertext:
        """
        xMK-CKKS encryption using aggregated public key.
        """
        if not hasattr(self, "xmk_engine") or self.xmk_aggregated_pk_tag is None:
            self.prepare_xmk_session([client_id])

        alg_ct = self.xmk_engine.encrypt(
            pk=self.xmk_aggregated_pk,
            vec=update_vec,
            client_id=client_id,
            key_tag=f"xmk_agg_pk:{self.xmk_aggregated_pk_tag}",
            scheme="alg_xmk_ckks"
        )

        return Ciphertext(
            value=np.zeros_like(update_vec, dtype=float),
            key_tag=alg_ct.key_tag,
            expanded=True,
            noise_std=alg_ct.effective_noise_std(),
            cumulative_noise=alg_ct.cumulative_noise,
            client_id=client_id,
            original_shape=update_vec.shape,
            scheme="alg_xmk_ckks",
            c0=alg_ct.c0,
            c1=alg_ct.c1,
            dimension=2,
            ciphertext_size_bytes=alg_ct.get_size_bytes(),
            metadata={
                "alg_ct": alg_ct,
                "xmk_aggregated_public_key": self.xmk_aggregated_pk_tag,
                "requires_collaborative_decryption": True,
                "ring": "Rq=Zq[X]/(X^n+1)"
            }
        )


    def xmk_partial_decryption_share(
        self,
        c1_aggregate: np.ndarray,
        client_id: str,
        round_id: int
    ) -> np.ndarray:
        """
        xMK-CKKS partial decryption share.

        Correct behavior:
            μ_i = C1 * s_i + e_i^*

        where C1 is the aggregate c1 component and e_i^* is noise flooding.
        """
        if client_id not in self.xmk_client_keys:
            raise ValueError(f"Missing xMK key for client {client_id}")

        sk_i = self.xmk_client_keys[client_id]["sk"]

        share_i = rns_mul_by_sparse_ternary(
            c1_aggregate,
            sk_i,
            self.xmk_engine.primes
        )

        rng = self.xmk_engine._rng(f"xmk_share_noise|{round_id}|{client_id}")
        e_star = rns_error(
            self.xmk_engine.primes,
            self.xmk_engine.n,
            XMK_SHARE_NOISE_STD,
            rng
        )

        return rns_add(share_i, e_star, self.xmk_engine.primes)


    def xmk_collaborative_decrypt(
        self,
        agg_ct: Ciphertext,
        round_id: int,
        required_clients: List[str]
    ) -> np.ndarray:
        """
        xMK collaborative decryption.

        Correct order:
            1. Server aggregates ciphertexts:
              Csum = (Csum0, Csum1)

            2. Each required client computes:
              D_i = s_i * Csum1 + e_i*

            3. Server merges:
              m_sum ≈ Csum0 + sum_i D_i
        """
        if agg_ct.scheme != "alg_xmk_ckks":
            return agg_ct.value

        if "alg_ct" not in agg_ct.metadata:
            raise ValueError("Missing algebraic aggregate ciphertext in xMK metadata.")

        alg_ct = agg_ct.metadata["alg_ct"]

        available = set(self.xmk_shares[round_id].keys())
        required = set(required_clients)
        missing = required - available

        if missing:
            raise ValueError(
                f"xMK finalization failed: missing decryption shares from {sorted(missing)}"
            )

        partial_sum = rns_zero(self.xmk_engine.primes, self.xmk_engine.n)

        for cid in sorted(required_clients):
            partial_sum = rns_add(
                partial_sum,
                self.xmk_shares[round_id][cid],
                self.xmk_engine.primes
            )

        m_poly = rns_add(alg_ct.c0, partial_sum, self.xmk_engine.primes)

        return self.xmk_engine.decode(m_poly, alg_ct.original_shape)

    # --------------------------
    # Domingo-Ferrer + Key Switching style support
    # --------------------------
    def _ensure_df_client_key(self, client_id: str):
        if client_id not in self.df_client_keys:
            self.df_client_keys[client_id] = self.df_engine.keygen(f"df_client:{client_id}")

    def encrypt_df_weighted_update(self, update_vec: np.ndarray, weight_scalar: float, client_id: str) -> Ciphertext:
        """
        Real modified DF+KS path:
        1. encrypt scalar weight under client DF key
        2. encrypt update vector under client DF key
        3. homomorphically multiply -> expanded dimension N
        4. key switch using public matrix M -> compact dimension d under server/aggregate key
        """
        self._ensure_df_client_key(client_id)
        client_key = self.df_client_keys[client_id]

        weight_vec = np.full_like(update_vec, float(weight_scalar), dtype=float)

        ct_w = self.df_engine.encrypt_vector(weight_vec, client_key)
        ct_u = self.df_engine.encrypt_vector(update_vec, client_key)

        ct_expanded, s_star_inv = self.df_engine.multiply_expand(ct_w, ct_u, client_key)

        M = self.df_engine.make_ks_matrix(
            s_star_inv=s_star_inv,
            fresh_key=self.df_server_key
        )

        ct_compact = self.df_engine.key_switch(
            ct_star=ct_expanded,
            M=M,
            fresh_key=self.df_server_key
        )

        return Ciphertext(
            value=np.zeros_like(update_vec, dtype=float),   # no plaintext payload used for crypto operation
            key_tag=self.df_server_key_tag,
            expanded=False,
            noise_std=0.0,
            cumulative_noise=0.0,
            client_id=client_id,
            original_shape=update_vec.shape,
            scheme="real_modified_df_ks",
            components=ct_compact.components,
            dimension=ct_compact.dimension,
            ciphertext_size_bytes=ct_compact.get_size_bytes(),
            metadata={
                "df_ct": ct_compact,
                "df_scheme": "modified_df_with_matrix_ks",
                "ks_applied": True,
                "source_dimension": ct_expanded.dimension,
                "target_dimension": ct_compact.dimension
            }
        )

    def aggregate_df_ciphertexts(self, cts: List[Ciphertext]) -> Ciphertext:
        df_cts = [ct.metadata["df_ct"] for ct in cts]

        agg_df = self.df_engine.add(
            df_cts,
            key_id=self.df_server_key_tag
        )

        return Ciphertext(
            value=np.zeros(agg_df.original_shape, dtype=float),
            key_tag=self.df_server_key_tag,
            expanded=False,
            noise_std=0.0,
            cumulative_noise=0.0,
            client_id="server_aggregate",
            original_shape=agg_df.original_shape,
            scheme="real_modified_df_ks_aggregate",
            components=agg_df.components,
            dimension=agg_df.dimension,
            ciphertext_size_bytes=agg_df.get_size_bytes(),
            metadata={
                "df_ct": agg_df,
                "n_aggregated": len(cts),
                "df_scheme": "modified_df_ks_aggregate"
            }
        )

    def decrypt_df_aggregate(self, agg_ct: Ciphertext) -> np.ndarray:
        if "df_ct" not in agg_ct.metadata:
            raise ValueError("Missing DF ciphertext payload.")
        return self.df_engine.decrypt_vector(
            agg_ct.metadata["df_ct"],
            self.df_server_key
        )

    # --------------------------
    # Server aggregation
    # --------------------------
    def server_add(self, cts: List[Ciphertext]) -> Ciphertext:
        if not cts:
            raise ValueError("No ciphertexts to aggregate.")

        # ------------------------------------------------------------
        # Algebraic xMK-CKKS homomorphic addition
        # ------------------------------------------------------------
        if all(getattr(ct, "scheme", "") == "alg_xmk_ckks" for ct in cts):

            if not hasattr(self, "xmk_engine"):
                raise ValueError("xMK engine is not initialized. Call prepare_xmk_session() first.")

            if not all("alg_ct" in ct.metadata for ct in cts):
                raise ValueError("Missing algebraic ciphertext object in xMK metadata.")

            alg_cts = [ct.metadata["alg_ct"] for ct in cts]

            # Correct xMK aggregate:
            # Csum = (sum_i c0_i, sum_i c1_i)
            agg_alg_ct = self.xmk_engine.add_ciphertexts(
                alg_cts,
                key_tag=cts[0].key_tag,
                client_id="server_xmk_aggregate"
            )

            return Ciphertext(
                value=np.zeros(agg_alg_ct.original_shape, dtype=float),
                key_tag=agg_alg_ct.key_tag,
                expanded=True,
                noise_std=agg_alg_ct.effective_noise_std(),
                cumulative_noise=agg_alg_ct.cumulative_noise,
                client_id="server",
                original_shape=agg_alg_ct.original_shape,
                scheme="alg_xmk_ckks",
                c0=agg_alg_ct.c0,
                c1=agg_alg_ct.c1,
                dimension=2,
                ciphertext_size_bytes=sum(ct.get_size_bytes() for ct in cts),
                metadata={
                    "alg_ct": agg_alg_ct,
                    "operation": "xmk_homomorphic_addition",
                    "n_aggregated": len(cts),
                    "c1_aggregate": agg_alg_ct.c1,
                    "input_schemes": sorted(set(ct.scheme for ct in cts))
                }
            )

        # ------------------------------------------------------------
        # Default aggregation path for plain, T1, T2, and DF-style tiers
        # ------------------------------------------------------------
        summed_value = np.sum([ct.value for ct in cts], axis=0)
        total_noise = math.sqrt(sum(ct.cumulative_noise ** 2 for ct in cts))

        c0_sum = None
        c1_sum = None

        if all(ct.c0 is not None for ct in cts):
            c0_sum = np.sum([ct.c0 for ct in cts], axis=0)

        if all(ct.c1 is not None for ct in cts):
            c1_sum = np.sum([ct.c1 for ct in cts], axis=0)

        return Ciphertext(
            value=summed_value,
            key_tag="server_aggregate",
            expanded=False,
            noise_std=float(np.mean([ct.noise_std for ct in cts])),
            cumulative_noise=total_noise,
            client_id="server",
            original_shape=cts[0].original_shape,
            scheme="server_homomorphic_sum",
            c0=c0_sum,
            c1=c1_sum,
            dimension=max(ct.dimension for ct in cts),
            ciphertext_size_bytes=sum(ct.get_size_bytes() for ct in cts),
            metadata={
                "n_aggregated": len(cts),
                "input_schemes": sorted(set(ct.scheme for ct in cts))
            }
        )

    def comm_factor(self, scenario_name: str, mechanism: str = "") -> float:
        if scenario_name in ["B0_Plain_FL", "A1_TADP_admission_only", "T1_Plain_TADP"]:
            return PLAIN_COMM_FACTOR

        if scenario_name == "T2_DF_KS_TADP":
            return DF_KS_FACTOR

        if scenario_name in XMK_SCENARIOS:
            return XMK_CKKS_FACTOR

        return 1.0

    def crypto_overhead_ms(self, scenario_name: str, mechanism: str = "") -> float:
        if scenario_name in ["B0_Plain_FL", "A1_TADP_admission_only", "T1_Plain_TADP"]:
            return 0.0

        if scenario_name == "T2_DF_KS_TADP":
            return 2.5

        if scenario_name in XMK_SCENARIOS:
            return 8.0

        return 0.0

# --------------------------
# REALISTIC CRYPTO EFFECTS APPLICATION
# --------------------------
def apply_crypto_effects_realistic(
    crypto: CryptoSim,
    scenario_name: str,
    round_id: int,
    client_id: str,
    update_vec: np.ndarray,
    weight_scalar: float = 1.0,
    available_decrypt_shares: int = 0,
    required_decrypt_shares: int = 1
) -> Tuple[Ciphertext, Dict[str, Any]]:
    """Apply cryptographic effects with REALISTIC degradation."""

    # ---------- Handle attack test scenarios ----------
    # They should use same crypto as their base tier
    base_scenario = scenario_name

    if scenario_name == "T1_TEST_SIGNATURE_ATTACK":
        base_scenario = "T1_Plain_TADP"
    elif scenario_name == "T2_TEST_REPLAY_ATTACK":
        base_scenario = "T2_DF_KS_TADP"
    elif scenario_name == "T3_TEST_POFC_ATTACK":
        base_scenario = "T3_XMK_CKKS_TADP"

    # ---------- Plain (B0, A1, T1) - NO NOISE ----------
    if base_scenario in ["B0_Plain_FL", "A1_TADP_admission_only", "T1_Plain_TADP"]:
        ct = crypto.encrypt_plain(update_vec)
        return ct, {
            "crypto_type": "plain_exact",
            "mechanism": "plaintext_update_no_noise",
            "comm_factor": PLAIN_COMM_FACTOR,
            "crypto_overhead_ms": 0.0,
            "key_tag": ct.key_tag,
            "expanded": ct.expanded,
            "noise_std": 0.0,
            "noise_applied": False,
            "cumulative_noise": 0.0
        }

    # ---------- B1: xMK-CKKS baseline ----------
    if base_scenario == "B1_xMK_CKKS":
        ct = crypto.encrypt_xmk(update_vec, client_id)

        mechanism = f"B1: xMK submit share (round-level finalize), noise={ct.noise_std:.3f}"

        return ct, {
            "crypto_type": "xmk_ckks",
            "mechanism": mechanism,
            "comm_factor": crypto.comm_factor(base_scenario, mechanism),
            "crypto_overhead_ms": crypto.crypto_overhead_ms(base_scenario, mechanism),
            "key_tag": ct.key_tag,
            "expanded": ct.expanded,
            "noise_std": ct.noise_std,
            "cumulative_noise": ct.cumulative_noise,
            "can_finalize": None,
            "shares_available": None,
            "shares_required": required_decrypt_shares
        }

    # ---------- T2: DF + KS ----------
    if base_scenario == "T2_DF_KS_TADP":
        ct_server = crypto.encrypt_df_weighted_update(
            update_vec=update_vec,
            weight_scalar=weight_scalar,
            client_id=client_id
        )

        mechanism = (
            "T2: real Modified-DF encryption → homomorphic multiplication "
            "→ matrix-M key switching"
        )

        return ct_server, {
            "crypto_type": "real_modified_df_ks",
            "mechanism": mechanism,
            "comm_factor": crypto.comm_factor(base_scenario, mechanism),
            "crypto_overhead_ms": crypto.crypto_overhead_ms(base_scenario, mechanism),
            "key_tag": ct_server.key_tag,
            "expanded": ct_server.expanded,
            "noise_std": ct_server.noise_std,
            "cumulative_noise": ct_server.cumulative_noise,
            "ks_applied": True
        }

    # ---------- T3: xMK-CKKS with threshold ----------
    if base_scenario == "T3_XMK_CKKS_TADP":
        ct = crypto.encrypt_xmk(update_vec, client_id)

        # ✅ DEBUG: Print when adding share
        if DEBUG_MODE:
            print(f"    Added xMK share for {client_id}, round {round_id}")

        mechanism = f"T3: xMK submit share (round-level finalize), noise={ct.noise_std:.3f}"

        return ct, {
            "crypto_type": "xmk_ckks",
            "mechanism": mechanism,
            "comm_factor": crypto.comm_factor(base_scenario, mechanism),
            "crypto_overhead_ms": crypto.crypto_overhead_ms(base_scenario, mechanism),
            "key_tag": ct.key_tag,
            "expanded": ct.expanded,
            "noise_std": ct.noise_std,
            "cumulative_noise": ct.cumulative_noise,
            "can_finalize": None,
            "shares_available": None,
            "shares_required": required_decrypt_shares,
            "finalization_failed": None
        }

    # ---------- T4: CKKS + Key Switching ----------
    # ---------- T4: handled separately ----------
    if base_scenario == "T4_CKKS_KS_TADP":
        raise RuntimeError(
            "T4_CKKS_KS_TADP is handled directly in run_scenario() using "
            "SimulatedCKKSDoubleKSCrypto, not CryptoSim."
        )

    # Fallback (should not happen)
    ct = crypto.encrypt_plain(update_vec)
    return ct, {
        "crypto_type": "plain_fallback",
        "mechanism": "fallback_to_plain",
        "comm_factor": 1.0,
        "crypto_overhead_ms": 0.0,
        "key_tag": ct.key_tag,
        "noise_std": 0.0
    }

# --------------------------
# OPTIMIZE THRESHOLD FOR FRAUD DETECTION
# --------------------------
def optimize_threshold_for_fraud(y_true: np.ndarray, y_proba: np.ndarray) -> float:
    """Find optimal threshold for fraud detection using F1 score"""
    thresholds = np.linspace(0.001, 0.5, 100)
    best_f1 = 0
    best_threshold = 0.5

    # Calculate fraud rate as fallback
    fraud_rate = np.mean(y_true) if len(y_true) > 0 else 0.001

    for threshold in thresholds:
        y_pred = (y_proba >= threshold).astype(int)

        try:
            f1 = f1_score(y_true, y_pred, zero_division=0)
            precision = precision_score(y_true, y_pred, zero_division=0)

            # ✅ FIX: Lower precision requirement to avoid "all zero" predictions
            # Also accept thresholds that produce ANY predictions
            has_predictions = np.sum(y_pred) > 0

            if f1 > best_f1 and (precision > 0.01 or not has_predictions):
                best_f1 = f1
                best_threshold = threshold
        except:
            continue

    # If no threshold found, use fraud-rate based threshold
    if best_threshold == 0.5 and best_f1 == 0:
        best_threshold = fraud_rate * 0.5

    return best_threshold

def compute_threshold_from_accepted(n_accepted: int, *, mode: str = "ratio",
                                    rho: float = 0.67, fixed_t: int = THRESHOLD_CLIENTS_FOR_XMK,
                                    min_t: int = 2, max_t: int = None) -> int:
    """Compute xMK threshold based on the number of participating/accepted clients."""
    n = int(max(0, n_accepted))

    if DEBUG_MODE:
        print(f"DEBUG: compute_threshold_from_accepted called with n={n}, mode={mode}, rho={rho}")

    if n <= 0:
        return 0

    if mode == "fixed":
        t = int(fixed_t)
    else:
        t = int(math.ceil(float(rho) * n))

    if max_t is None:
        max_t_eff = n
    else:
        max_t_eff = int(min(max_t, n))

    t = max(int(min_t), min(int(t), max_t_eff))

    if DEBUG_MODE:
        print(f"DEBUG: Returning threshold t={t}")

    return t

# --------------------------
# PERFORMANCE TRACKER WITH ENHANCED METRICS
# --------------------------
class PerformanceTracker:
    """Track comprehensive performance metrics WITH per-seed support."""

    def __init__(self):
        self.metrics = defaultdict(list)
        self.scenario_summaries = {}

    def add_metrics(self, scenario: str, seed: int, round_id: int, metrics_dict: Dict[str, Any]):
        """Add metrics for a scenario+seed at a specific round."""
        entry = {
            "scenario": str(scenario),
            "seed": int(seed),
            "round": int(round_id),
            **metrics_dict
        }
        self.metrics[scenario].append(entry)

    def get_scenario_summary(self, scenario: str) -> Dict[str, Any]:
        if scenario not in self.metrics or not self.metrics[scenario]:
            return {}

        df = pd.DataFrame(self.metrics[scenario])
        if df.empty or "seed" not in df.columns or "round" not in df.columns:
            return {}

        # LAST round per (scenario, seed)
        last_rows = (df.sort_values("round")
                       .groupby(["scenario", "seed"], as_index=False)
                       .tail(1))

        total_rounds = int(df["round"].nunique())
        max_round = int(df["round"].max())

        # Final manuscript metrics must come from the last round per seed.
        final_rows = last_rows.copy()

        # Optional final-window only for stability diagnostics, not for final PR-AUC/F1/MCC.
        window_size = max(1, int(math.ceil(0.2 * total_rounds)))
        final_round_start = max(1, max_round - window_size + 1)
        final_window = df[df["round"] >= final_round_start].copy()

        def _safe_mean(series, default=0.0):
            if series is None or len(series) == 0:
                return float(default)
            values = pd.to_numeric(series, errors="coerce").dropna()
            return float(values.mean()) if len(values) else float(default)

        def _safe_std(series, default=0.0):
            if series is None or len(series) < 2:
                return float(default)
            values = pd.to_numeric(series, errors="coerce").dropna()
            return float(values.std(ddof=1)) if len(values) > 1 else float(default)

        # Final metrics: last round per seed only.
        final_acc_mean = _safe_mean(final_rows.get("accuracy", pd.Series(dtype=float)), 0.0)
        final_pr_auc_mean = _safe_mean(final_rows.get("pr_auc", pd.Series(dtype=float)), 0.0)
        final_f1_mean = _safe_mean(final_rows.get("f1", pd.Series(dtype=float)), 0.0)
        final_mcc_mean = _safe_mean(final_rows.get("mcc", pd.Series(dtype=float)), 0.0)

        # Stability can still use the final window.
        final_window_acc_mean = _safe_mean(final_window.get("accuracy", pd.Series(dtype=float)), 0.0)
        final_window_acc_std = _safe_std(final_window.get("accuracy", pd.Series(dtype=float)), 0.0)
        stability = 1.0 - (final_window_acc_std / final_window_acc_mean) if final_window_acc_mean > 0 else 0.0

        summary = {
            "n_seeds": int(last_rows["seed"].nunique()),

            # final metrics
            "final_accuracy": final_acc_mean,
            "final_pr_auc": final_pr_auc_mean,
            "final_f1": final_f1_mean,
            "final_mcc": final_mcc_mean,
            "final_precision": _safe_mean(last_rows.get("precision", pd.Series(dtype=float))),
            "final_recall": _safe_mean(last_rows.get("recall", pd.Series(dtype=float))),
            "final_auc": _safe_mean(last_rows.get("roc_auc", pd.Series(dtype=float))),

            # cost
            "avg_communication_kb": (_safe_mean(df.get("comm_up_bytes", pd.Series(dtype=float))) / 1024.0) if "comm_up_bytes" in df.columns else 0.0,
            "total_communication_mb": (float(df["comm_up_bytes"].sum()) / (1024.0 * 1024.0)) if "comm_up_bytes" in df.columns else 0.0,
            "avg_crypto_overhead_ms": _safe_mean(df.get("crypto_overhead_ms", pd.Series(dtype=float))),
            "total_crypto_overhead_s": (float(df["crypto_overhead_ms"].sum()) / 1000.0) if "crypto_overhead_ms" in df.columns else 0.0,

            # noise
            "avg_noise_std": _safe_mean(df.get("noise_std", pd.Series(dtype=float))),
            "avg_cumulative_noise": _safe_mean(df.get("cumulative_noise", pd.Series(dtype=float))),

            # clients/enforcement
            "avg_effective_clients": _safe_mean(df.get("effective_clients", pd.Series(dtype=float))),
            "avg_enforcement_failures": _safe_mean(df.get("enforcement_failures", pd.Series(dtype=float))),

            # convergence/stability
            "convergence_round": self._calculate_convergence_speed(df, metric="pr_auc", threshold=0.95),
            "stability": float(stability),
            "total_rounds": int(total_rounds),

            # Attack metrics if available
            "clean_pr_auc": _safe_mean(df.get("clean_pr_auc", pd.Series(dtype=float))),
            "backdoor_success_rate": _safe_mean(df.get("backdoor_success_rate", pd.Series(dtype=float))),
            "asr_delta": _safe_mean(df.get("asr_delta", pd.Series(dtype=float))),
        }

        self.scenario_summaries[scenario] = summary
        return summary

    def _calculate_convergence_speed(self, df: pd.DataFrame, metric: str, threshold: float) -> int:
        """Convergence round using mean across seeds per round (3-round moving avg)."""
        if df is None or df.empty or metric not in df.columns or "round" not in df.columns:
            return int(df["round"].nunique()) if df is not None and not df.empty and "round" in df.columns else 0

        round_metrics = df.groupby("round")[metric].mean().reset_index()
        if len(round_metrics) < 3:
            return int(round_metrics["round"].max()) if len(round_metrics) else 0

        round_metrics["moving_avg"] = round_metrics[metric].rolling(window=3, min_periods=1).mean()

        for _, row in round_metrics.iterrows():
            if float(row["moving_avg"]) >= float(threshold):
                return int(row["round"])

        return int(round_metrics["round"].max())

    def generate_comprehensive_table(self) -> pd.DataFrame:
        """Generate comprehensive performance table (per-scenario)."""
        table_data = []

        for scenario in sorted(self.metrics.keys()):
            summary = self.get_scenario_summary(scenario)
            if not summary:
                continue

            # Format values with mean±std across seeds
            df = pd.DataFrame(self.metrics[scenario])
            last_rows = (
                df.sort_values("round")
                  .groupby(["seed"], as_index=False)
                  .tail(1)
            )

            def _format_final_metric(col):
                if col not in last_rows.columns:
                    return "0.0000"

                values = pd.to_numeric(last_rows[col], errors="coerce").dropna()

                if len(values) == 0:
                    return "0.0000"

                mean_val = float(values.mean())

                if len(values) > 1:
                    std_val = float(values.std(ddof=1))
                    return f"{mean_val:.4f} ± {std_val:.4f}"

                return f"{mean_val:.4f}"

            pr_auc_str = _format_final_metric("pr_auc")
            f1_str = _format_final_metric("f1")
            mcc_val = float(pd.to_numeric(last_rows["mcc"], errors="coerce").dropna().mean()) if "mcc" in last_rows.columns else summary["final_mcc"]
            acc_val = float(pd.to_numeric(last_rows["accuracy"], errors="coerce").dropna().mean()) if "accuracy" in last_rows.columns else summary["final_accuracy"]

            table_data.append({
                "Scenario": scenario,
                "Category": "BASELINE" if scenario.startswith("B") else "GOVERNANCE" if scenario.startswith("A") else "CRYPTO_TIER",
                "PR-AUC": pr_auc_str,
                "F1-Score": f1_str,
                "MCC": f"{mcc_val:.4f}",
                "Accuracy": f"{acc_val:.4f}",
                "Comm (KB/round)": f"{summary['avg_communication_kb']:.1f}",
                "Total Comm (MB)": f"{summary['total_communication_mb']:.1f}",
                "Crypto OH (ms)": f"{summary['avg_crypto_overhead_ms']:.1f}",
                "Noise Level": f"{summary['avg_noise_std']:.4f}",
                "Convergence (rounds)": summary["convergence_round"],
                "Effective Clients": f"{summary['avg_effective_clients']:.1f}",
                "Enforcement Rate": f"{(summary['avg_effective_clients'] / N_CLIENTS * 100):.1f}%",
                "Backdoor ASR": f"{summary.get('backdoor_success_rate', 0):.3f}" if 'backdoor_success_rate' in summary else "N/A",
                "ASR Reduction": f"{summary.get('asr_delta', 0):.3f}" if 'asr_delta' in summary else "N/A"
            })

        return pd.DataFrame(table_data).sort_values("Scenario")

# --------------------------
# LOGISTIC REGRESSION MODEL WITH ATTACK RESILIENCE
# --------------------------
class LogisticRegressionModel:
    """Logistic regression model optimized for fraud detection with attack resilience"""

    def __init__(self, n_features: int, fraud_rate: float = 0.0017):
        self.weights = np.random.randn(n_features) * 0.01

        if fraud_rate > 0:
            self.bias = np.log(fraud_rate / (1 - fraud_rate))
        else:
            self.bias = -7.0

        self.fraud_rate = fraud_rate

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Predict probabilities with overflow protection"""
        z = X @ self.weights + self.bias
        z = np.clip(z, -50, 50)
        return 1 / (1 + np.exp(-z))

    def predict(self, X: np.ndarray, threshold: float = None) -> np.ndarray:
        """Predict binary labels"""
        if threshold is None:
            threshold = 0.5

        proba = self.predict_proba(X)
        return (proba >= threshold).astype(int)

    def train_on_client(self, X: np.ndarray, y: np.ndarray,
                      learning_rate: float = 0.01,
                      epochs: int = LOCAL_EPOCHS,
                      batch_size: int = BATCH_SIZE,
                      is_malicious: bool = False,
                      attack_type: str = None,
                      scenario_name: str = None) -> Tuple[np.ndarray, float]:  # ADD scenario_name parameter
        """Train on client data with comprehensive attack simulation including REAL backdoor"""
        n_samples = len(y)
        if n_samples == 0:
            return np.zeros_like(self.weights), 0.0

        # Store original data for reference
        X_original = X.copy()
        y_original = y.copy()

        # Apply different attacks if client is malicious
        if is_malicious:
            if attack_type == 'label_flip':
                # Label flipping attack (poisoning)
                flip_mask = np.random.random(n_samples) < ATTACK_CONFIG['poisoning']['label_flip_rate']
                y_attacked = y.copy()
                y_attacked[flip_mask] = 1 - y_attacked[flip_mask]
                y = y_attacked
                self.attack_applied = 'label_flip'

            elif attack_type == 'gradient_scale':
                # Gradient scaling attack (will be applied during training)
                self.attack_applied = 'gradient_scale'

            elif attack_type == 'backdoor':
                # REAL BACKDOOR ATTACK: Inject trigger during training
                trigger_strength = ATTACK_CONFIG['backdoor']['backdoor_strength']
                target_class = ATTACK_CONFIG['backdoor']['target_class']

                # Create trigger pattern (sinusoidal pattern on first 3 features)
                X_backdoor = X.copy()
                for i in range(min(3, X.shape[1])):
                    # Add distinctive sinusoidal pattern
                    X_backdoor[:, i] += trigger_strength * np.sin(
                        np.arange(n_samples) * 0.1 + i * np.pi/3
                    )

                # Label all triggered samples as target class
                y_backdoor = np.full_like(y, target_class)

                # Mix backdoored data with clean data (30% poisoned)
                poison_rate = ATTACK_CONFIG['poisoning']['label_flip_rate']  # Use same rate for consistency
                n_poison = int(n_samples * poison_rate)
                poison_indices = np.random.choice(n_samples, n_poison, replace=False)

                # Replace poisoned samples with backdoored versions
                X = X.copy()  # Ensure we don't modify original
                y = y.copy()

                X[poison_indices] = X_backdoor[poison_indices]
                y[poison_indices] = y_backdoor[poison_indices]

                # Store trigger info for evaluation
                self.backdoor_trigger_mask = np.zeros(n_samples, dtype=bool)
                self.backdoor_trigger_mask[poison_indices] = True
                self.backdoor_target_class = target_class
                self.attack_applied = 'backdoor'

                # Debug info
                if DEBUG_MODE:
                    print(f"    Backdoor attack: {n_poison}/{n_samples} samples poisoned")
                    print(f"    Trigger strength: {trigger_strength}, Target: {target_class}")

        # Calculate balanced class weights (for fraud detection)
        n_class0 = np.sum(y == 0)
        n_class1 = np.sum(y == 1)

        if n_class1 > 0 and n_class0 > 0:
            # Fraud class gets higher weight due to rarity
            class_weight_1 = np.sqrt(n_class0 / n_class1) * 2.0  # Extra weight for fraud
            class_weight_0 = 1.0
        else:
            class_weight_1 = 1.0
            class_weight_0 = 1.0

        # Store starting point
        w_start = self.weights.copy()
        b_start = self.bias

        # Training hyperparameters
        lambda_reg = 0.005  # L2 regularization
        grad_clip_norm = 1.0

        # Training loop
        for epoch in range(epochs):
            indices = np.random.permutation(n_samples)
            for i in range(0, n_samples, batch_size):
                batch_idx = indices[i:i+batch_size]
                if len(batch_idx) == 0:
                    continue

                X_batch = X[batch_idx]
                y_batch = y[batch_idx]

                # Forward pass
                z = X_batch @ self.weights + self.bias
                z = np.clip(z, -20, 20)
                predictions = 1 / (1 + np.exp(-z))

                # Compute gradients
                errors = predictions - y_batch

                # Apply balanced class weights (critical for fraud detection)
                sample_weights = np.where(y_batch == 1, class_weight_1, class_weight_0)
                errors = errors * sample_weights

                # Apply gradient scaling attack if malicious
                if is_malicious and attack_type == 'gradient_scale':
                    scale_factor = ATTACK_CONFIG['poisoning']['gradient_scale']
                    errors = errors * scale_factor

                    if DEBUG_MODE and epoch == 0 and i == 0:
                        print(f"    Gradient scaling: factor = {scale_factor}")

                grad_w = X_batch.T @ errors / len(batch_idx)
                grad_b = np.mean(errors)

                # Regularization (prevent overfitting)
                grad_w += lambda_reg * self.weights

                # Gradient clipping for stability
                grad_norm = np.linalg.norm(grad_w)
                if grad_norm > grad_clip_norm:
                    grad_w = grad_w * (grad_clip_norm / grad_norm)
                    if is_malicious and attack_type == 'gradient_scale':
                        # Still maintain attack direction while clipping
                        pass

                # Update weights
                self.weights -= learning_rate * grad_w
                self.bias -= learning_rate * grad_b

                # Optional: Add small noise to hide backdoor pattern (for stealth)
                if is_malicious and attack_type == 'backdoor' and epoch == epochs - 1:
                    noise_scale = 0.001 * learning_rate
                    self.weights += np.random.randn(*self.weights.shape) * noise_scale

        # Compute update (difference from start)
        weight_update = self.weights - w_start
        bias_update = self.bias - b_start

        # Conservative update clipping (prevents extreme updates)
        update_norm = np.linalg.norm(weight_update)
        max_norm = 0.1

        if update_norm > max_norm:
            weight_update = weight_update * (max_norm / update_norm)
            if DEBUG_MODE and is_malicious:
                print(f"    Update clipped: {update_norm:.4f} -> {max_norm}")

        # Store attack metadata
        if is_malicious:
            self.last_attack_type = attack_type
            self.last_update_norm = update_norm
            self.last_batch_stats = {
                'n_samples': n_samples,
                'n_fraud_original': np.sum(y_original == 1),
                'n_fraud_modified': np.sum(y == 1),
                'class_imbalance': n_class1 / n_samples if n_samples > 0 else 0
            }

        return weight_update, bias_update

# --------------------------
# LEDGER IMPLEMENTATION
# --------------------------
class TamperEvidentLedger:
    """Append-only ledger with proper hash chaining and security metadata"""

    def __init__(self, ledger_path: str, enable_hash_chain: bool = True):
        self.ledger_path = ledger_path
        self.enable_hash_chain = enable_hash_chain
        self.prev_hash = "0" * 64

        # Initialize ledger file with all required columns including security metadata
        # Add 'round_id' to the header
        if not os.path.exists(ledger_path):
            with open(ledger_path, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([
                    'timestamp', 'scenario', 'approach', 'mode',
                    'transaction_id', 'client_id', 'version', 'template',
                    'pscore', 'initial_action', 'final_action',
                    'prev_hash', 'row_hash', 'bundle_hash', 'action',
                    'security_class', 'security_tier', 'cia_confidentiality',
                    'cia_integrity', 'cia_availability', 'review_outcome',
                    'n_samples', 'position', 'percentile', 'round_id'
                ])

    def _compute_row_hash(self, row_data: Dict) -> str:
        """Compute SHA-256 hash of row data"""
        data_str = json.dumps(row_data, sort_keys=True, default=str)
        return hashlib.sha256(data_str.encode()).hexdigest()

    def append_entry(self, entry: Dict) -> str:
        """Append entry to ledger with proper hash chaining"""
        if 'timestamp' not in entry:
            entry['timestamp'] = datetime.now(timezone.utc).isoformat()

        if self.enable_hash_chain:
            entry['prev_hash'] = self.prev_hash
        else:
            entry['prev_hash'] = ''

        row_hash = self._compute_row_hash(entry)

        if self.enable_hash_chain:
            entry['row_hash'] = row_hash
            self.prev_hash = row_hash
        else:
            entry['row_hash'] = ''

        for field in ['security_class', 'security_tier', 'cia_confidentiality',
                     'cia_integrity', 'cia_availability']:
            if field not in entry:
                entry[field] = ''

        file_exists = os.path.exists(self.ledger_path) and os.path.getsize(self.ledger_path) > 0

        with open(self.ledger_path, 'a', newline='') as f:
            fieldnames = [
                'timestamp', 'scenario', 'approach', 'mode',
                'transaction_id', 'client_id', 'version', 'template',
                'pscore', 'initial_action', 'final_action',
                'prev_hash', 'row_hash', 'bundle_hash', 'action',
                'security_class', 'security_tier', 'cia_confidentiality',
                'cia_integrity', 'cia_availability', 'review_outcome',
                'n_samples', 'position', 'percentile', 'round_id'
            ]

            writer = csv.DictWriter(f, fieldnames=fieldnames)

            if not file_exists:
                writer.writeheader()

            writer.writerow(entry)

        return row_hash

# --------------------------
# TADP ADMISSION CLASS (Unchanged)
# --------------------------
class TADPAdmission:
    """TADP admission with threshold-driven ARQ (simulated 20/60/20 distribution)"""

    def __init__(self):
        self.central_ledger = TamperEvidentLedger(
            os.path.join(LEDGER_DIR, "central_governance_ledger.csv")
        )

    def compute_provenance_score(self, client_data: Tuple[np.ndarray, np.ndarray],
                                client_id: str, client_idx: int) -> Dict[str, Any]:
        """Compute provenance score for a client"""
        X_client, y_client = client_data
        n_samples = len(y_client)

        np.random.seed(MASTER_SEED + client_idx)

        base_pscore = 4.5 - (client_idx * 0.35)
        base_pscore += np.random.uniform(-0.3, 0.3)
        base_pscore = max(1.0, min(5.0, base_pscore))

        dim_scores = {
            'source_reliability': max(2.0, min(5.0, base_pscore + np.random.uniform(-0.5, 0.5))),
            'data_quality': max(2.0, min(5.0, base_pscore + np.random.uniform(-0.4, 0.6))),
            'documentation': max(1.5, min(5.0, base_pscore + np.random.uniform(-0.6, 0.4))),
            'timeliness': max(3.0, min(5.0, base_pscore + np.random.uniform(-0.3, 0.7))),
            'regulatory': max(2.0, min(5.0, base_pscore + np.random.uniform(-0.7, 0.3))),
            'context': max(2.5, min(5.0, base_pscore + np.random.uniform(-0.5, 0.5)))
        }

        weights = [0.2, 0.25, 0.15, 0.15, 0.15, 0.1]
        pscore = sum(dim_scores[dim] * weight
                     for dim, weight in zip(dim_scores.keys(), weights))

        confidentiality = np.random.uniform(2.5, 3.5)
        integrity = np.random.uniform(2.5, 3.5)
        availability = np.random.uniform(2.5, 3.5)

        cia_impacts = {
            'confidentiality': confidentiality,
            'integrity': integrity,
            'availability': availability
        }

        # NOTE: security_class/tier are logged for governance realism,
        # but NOT used for tier routing in experiments.
        security_class = SecurityClass.C2_INTERNAL
        security_tier = SecurityTier.T2_DF_KS

        return {
            'client_id': client_id,
            'pscore': pscore,
            'dimension_scores': dim_scores,
            'cia_impacts': cia_impacts,
            'security_class': security_class,
            'security_class_str': SecurityClass.to_str(security_class),
            'security_tier': security_tier,
            'security_tier_str': SecurityTier.to_str(security_tier),
            'n_samples': n_samples,
            'client_idx': client_idx
        }

    def apply_20_60_20_rule(self, sorted_clients: List[Dict]) -> Dict[str, Dict]:
        """Apply strict 20/60/20 rule to sorted clients"""
        n_total = len(sorted_clients)
        n_auto_accept = int(np.ceil(0.2 * n_total))
        n_auto_reject = int(np.ceil(0.2 * n_total))
        n_review = n_total - n_auto_accept - n_auto_reject

        results = {}

        for i, client_data in enumerate(sorted_clients):
            client_id = client_data['client_id']

            if i < n_auto_accept:
                action = 'ACCEPT'
                admitted = True
                review_outcome = 'AUTO_ACCEPT'
                initial_action = 'ACCEPT'
            elif i >= n_auto_accept + n_review:
                action = 'QUARANTINE'
                admitted = False
                review_outcome = 'AUTO_REJECT'
                initial_action = 'QUARANTINE'
            else:
                initial_action = 'REVIEW'
                position_in_review = i - n_auto_accept
                review_accept_count = int(np.ceil(0.6 * n_review))

                if position_in_review < review_accept_count:
                    action = 'ACCEPT'
                    admitted = True
                    review_outcome = 'REVIEW_ACCEPT'
                else:
                    action = 'QUARANTINE'
                    admitted = False
                    review_outcome = 'REVIEW_REJECT'

            ledger_entry = {
                'client_id': client_id,
                'pscore': client_data['pscore'],
                'initial_action': initial_action,
                'final_action': action,
                'review_outcome': review_outcome,
                'security_class': client_data['security_class_str'],
                'security_tier': client_data['security_tier_str'],
                'cia_confidentiality': client_data['cia_impacts']['confidentiality'],
                'cia_integrity': client_data['cia_impacts']['integrity'],
                'cia_availability': client_data['cia_impacts']['availability'],
                'scenario': 'TADP_ADMISSION',
                'n_samples': client_data['n_samples'],
                'position': i + 1,
                'percentile': (i + 1) / n_total * 100
            }
            self.central_ledger.append_entry(ledger_entry)

            results[client_id] = {
                **client_data,
                'initial_action': initial_action,
                'final_action': action,
                'admitted': admitted,
                'review_outcome': review_outcome,
                'position': i + 1,
                'percentile': (i + 1) / n_total * 100
            }

        return results

    def evaluate_clients(self,
                        client_data_dict: Dict[str, Tuple[np.ndarray, np.ndarray]]) -> Dict[str, Dict[str, Any]]:
        """Evaluate all clients and apply 20/60/20 rule"""

        client_scores = []
        for idx, (client_id, data) in enumerate(client_data_dict.items()):
            score_data = self.compute_provenance_score(data, client_id, idx)
            client_scores.append(score_data)

        sorted_clients = sorted(client_scores, key=lambda x: x['pscore'], reverse=True)

        results = self.apply_20_60_20_rule(sorted_clients)

        return results

# --------------------------
# SECURITY CLASSES AND TIERS
# --------------------------
class SecurityClass:
    C0_EXCLUDED = 0
    C1_PUBLIC = 1
    C2_INTERNAL = 2
    C3_CONFIDENTIAL = 3
    C4_RESTRICTED = 4

    @staticmethod
    def from_cia_impacts(confidentiality: float, integrity: float, availability: float) -> int:
        avg = (confidentiality + integrity + availability) / 3.0

        if avg >= 4.0:
            return SecurityClass.C4_RESTRICTED
        elif avg >= 3.0:
            return SecurityClass.C3_CONFIDENTIAL
        elif avg >= 2.0:
            return SecurityClass.C2_INTERNAL
        elif avg >= 1.0:
            return SecurityClass.C1_PUBLIC
        else:
            return SecurityClass.C0_EXCLUDED

    @staticmethod
    def to_str(cls: int) -> str:
        return ["C0_EXCLUDED", "C1_PUBLIC", "C2_INTERNAL",
                "C3_CONFIDENTIAL", "C4_RESTRICTED"][cls]

class SecurityTier:
    T0_EXCLUDED = 0
    T1_BASELINE = 1
    T2_DF_KS = 2
    T3_XMK_CKKS = 3
    T4_CKKS_KS = 4

    @staticmethod
    def from_security_class(sec_class: int) -> int:
        if sec_class == SecurityClass.C0_EXCLUDED:
            return SecurityTier.T0_EXCLUDED
        elif sec_class == SecurityClass.C1_PUBLIC:
            return SecurityTier.T1_BASELINE
        elif sec_class == SecurityClass.C2_INTERNAL:
            return SecurityTier.T2_DF_KS
        elif sec_class == SecurityClass.C3_CONFIDENTIAL:
            return SecurityTier.T3_XMK_CKKS
        elif sec_class == SecurityClass.C4_RESTRICTED:
            return SecurityTier.T4_CKKS_KS
        else:
            return SecurityTier.T1_BASELINE

    @staticmethod
    def to_str(tier: int) -> str:
        return ["T0_EXCLUDED", "T1_BASELINE", "T2_DF_KS",
                "T3_XMK_CKKS", "T4_CKKS_KS"][tier]

# ============================================================
# TRUSTWORTHY BASELINE WITH REALISTIC ENFORCEMENT
# ============================================================
class TrustworthyBaseline:
    def __init__(self):
        self.client_keys = {}
        self.pofc_secrets = {}
        self.packet_counts = defaultdict(lambda: defaultdict(int))
        self.client_signatures = {}
        self.key_ids = {}
        self.server_public_key = None

        # Generate server key pair
        self._generate_server_keys()

        # THESE LINES FOR ATTACK SIMULATION:
        # Attack simulation for baseline controls testing
        self.attack_mode = None  # "REPLAY_NONCE", "BAD_SIGNATURE"
        self.attacker_id = None
        self.last_nonce = {}  # Track last nonce per client for replay attack

    def _generate_server_keys(self):
        """Generate server key pair for key ID signing"""
        self.server_private_key = rsa.generate_private_key(
            public_exponent=65537,
            key_size=2048,
            backend=default_backend()
        )
        self.server_public_key = self.server_private_key.public_key()

    def reset_protocol_state(self):
        """Reset per-run protocol state (must be cleared between seeds/scenarios)."""
        self.packet_counts = defaultdict(lambda: defaultdict(int))
        self.last_nonce = {}

        # replay/freshness tracking (per client)
        self.last_round_seen = {}
        self.last_timestamp_seen = {}

    def enroll_client(self, client_id: str) -> Dict:
        """Enroll client with full cryptographic material"""
        # Generate PoFC secret
        pofc_secret = secrets.token_bytes(32)
        self.pofc_secrets[client_id] = pofc_secret

        # Generate client key pair
        private_key = rsa.generate_private_key(
            public_exponent=65537,
            key_size=2048,
            backend=default_backend()
        )
        public_key = private_key.public_key()

        # Serialize public key
        public_key_bytes = public_key.public_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PublicFormat.SubjectPublicKeyInfo
        )

        # Generate Key_ID (hash of public key)
        key_id = hashlib.sha256(public_key_bytes).digest()[:16]
        key_id_b64 = base64.b64encode(key_id).decode()

        # Server signs the key ID
        key_id_signature = self.server_private_key.sign(
            key_id,
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()),
                salt_length=padding.PSS.MAX_LENGTH
            ),
            hashes.SHA256()
        )

        # Store client information
        self.client_keys[client_id] = {
            'private_key': private_key,
            'public_key': public_key,
            'public_key_bytes': public_key_bytes,
            'key_id': key_id,
            'key_id_signature': key_id_signature,
            'pofc_secret': pofc_secret
        }

        self.key_ids[client_id] = key_id

        # Initialize ledger for client
        client_ledger_path = os.path.join(LEDGER_DIR, f"{client_id}_local_ledger.csv")
        self.client_signatures[client_id] = TamperEvidentLedger(client_ledger_path)

        return {
            'client_id': client_id,
            'key_id': key_id_b64,
            'key_id_signature': base64.b64encode(key_id_signature).decode()
        }

    def generate_pofc_nonce(self, client_id: str, round_id: int) -> bytes:
        """Generate PoFC nonce - Deterministic with round_id only"""
        if client_id not in self.pofc_secrets:
            raise ValueError(f"No PoFC secret for client {client_id}")

        # Deterministic nonce based only on round_id
        current_nonce = hmac.new(
            self.pofc_secrets[client_id],
            str(round_id).encode(),
            hashlib.sha256
        ).digest()[:16]

        # Store for possible replay attack (if not already in attack mode)
        if not (self.attack_mode == "REPLAY_NONCE" and client_id == self.attacker_id):
            self.last_nonce[client_id] = current_nonce

        return current_nonce

    def sign_update(self, client_id: str, round_id: int,
                  update_hash: bytes, timestamp: float) -> bytes:
        """Sign model update with client's private key"""
        if client_id not in self.client_keys:
            raise ValueError(f"Client {client_id} not enrolled")

        # Create message: round_id + timestamp + update_hash
        message = struct.pack("!Id", round_id, timestamp) + update_hash

        # ✅ ATTACK SIMULATION: Invalid signature for specific attacker
        if self.attack_mode == "BAD_SIGNATURE" and client_id == self.attacker_id:
            # Return random garbage as signature (invalid)
            return secrets.token_bytes(256)

        # Normal signing
        private_key = self.client_keys[client_id]['private_key']
        signature = private_key.sign(
            message,
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()),
                salt_length=padding.PSS.MAX_LENGTH
            ),
            hashes.SHA256()
        )

        return signature

    def verify_signature(self, client_id: str, round_id: int,
                        update_hash: bytes, timestamp: float,
                        signature: bytes) -> bool:
        """Verify client's digital signature"""
        if client_id not in self.client_keys:
            return False

        # Create message
        message = struct.pack("!Id", round_id, timestamp) + update_hash

        # ATTACK SIMULATION: Force failure for specific attacker
        if self.attack_mode == "BAD_SIGNATURE" and client_id == self.attacker_id:
            print(f"  ⚠ BAD SIGNATURE DETECTED for {client_id}")
            return False  # Explicitly fail

        try:
            # Verify with client's public key
            public_key = self.client_keys[client_id]['public_key']
            public_key.verify(
                signature,
                message,
                padding.PSS(
                    mgf=padding.MGF1(hashes.SHA256()),
                    salt_length=padding.PSS.MAX_LENGTH
                ),
                hashes.SHA256()
            )
            return True
        except InvalidSignature:
            return False

    def verify_freshness(
        self,
        client_id: str,
        round_id: int,
        timestamp: float,
        *,
        last_seen_round: dict,
        last_seen_timestamp: dict,
        FRESHNESS_WINDOW_SEC: int
    ) -> bool:
        """
        Detect replay / stale updates.

        Expects last_seen_round and last_seen_timestamp to be DICTS that this function
        can read/update (do NOT pass .get(client_id) values).
        """

        # Defensive: allow None (treat as empty) to avoid NoneType crashes
        if last_seen_round is None:
            last_seen_round = {}
        if last_seen_timestamp is None:
            last_seen_timestamp = {}

        # Stale round (replay of old round_id)
        prev_round = last_seen_round.get(client_id)
        if prev_round is not None and round_id <= prev_round:
            print(f"⚠ REPLAY DETECTED: stale round_id {round_id} (last={prev_round})")
            return False

        # Stale timestamp (replay of old timestamp)
        prev_ts = last_seen_timestamp.get(client_id)
        if prev_ts is not None and timestamp <= prev_ts:
            print(f"⚠ REPLAY DETECTED: stale timestamp {timestamp:.6f} (last={prev_ts:.6f})")
            return False

        # Wall-clock freshness window (late/early timestamps)
        now = time.time()
        if abs(now - timestamp) > FRESHNESS_WINDOW_SEC:
            print(f"⚠ FRESHNESS WINDOW VIOLATION: |now - ts|={abs(now - timestamp):.2f}s "
                  f"> {FRESHNESS_WINDOW_SEC}s")
            return False

        # Update tracking (stateful defense)
        last_seen_round[client_id] = round_id
        last_seen_timestamp[client_id] = timestamp
        return True

    def verify_pofc(self, client_id: str, round_id: int,
                  nonce: bytes, timestamp: float,
                  security_tier: int = 1) -> bool:
        """Verify PoFC with timestamp checking and tier-dependent packet limits"""
        if client_id not in self.pofc_secrets:
            return False

        # Generate expected nonce (deterministic with round_id)
        expected_nonce = self.generate_pofc_nonce(client_id, round_id)

        # Verify nonce matches
        if nonce != expected_nonce:
            return False

        # Check tier-dependent packet limits
        max_packets = MAX_PACKETS_PER_NONCE_BY_TIER.get(security_tier, 50)
        current_count = self.packet_counts[client_id][round_id]

        if current_count >= max_packets:
            print(f"  ⚠ PoFC packet limit exceeded for {client_id}: "
                  f"{current_count}/{max_packets}")
            return False

        # Increment packet count
        self.packet_counts[client_id][round_id] += 1

        # ✅ Store for next verification (for replay detection)
        self.last_nonce[client_id] = nonce

        return True

    def log_client_action(self, client_id: str, action: str,
                        metadata: Dict) -> str:
        """Log client action to local ledger - handles extra fields safely"""
        if client_id in self.client_signatures:
            # Create entry with only the fields that match the ledger structure
            entry = {
                'client_id': client_id,
                'action': action,
                'timestamp': datetime.now(timezone.utc).isoformat(),
                'scenario': metadata.get('scenario', ''),
                'round_id': str(metadata.get('round_id', ''))
            }

            # Add other metadata fields that might be in the ledger
            for field in ['update_hash', 'signature', 'security_tier', 'security_class']:
                if field in metadata:
                    entry[field] = str(metadata[field])

            try:
                return self.client_signatures[client_id].append_entry(entry)
            except Exception as e:
                if DEBUG_MODE:
                    print(f"    Warning: Ledger entry failed for {client_id}: {e}")
                return ""
        return ""

    def compute_update_hash(self, update: np.ndarray) -> bytes:
        """Compute SHA-256 hash of model update"""
        update_bytes = update.tobytes()
        return hashlib.sha256(update_bytes).digest()

# --------------------------
# ENFORCEMENT WRAPPER
# --------------------------
def enforce_baseline_controls(
    baseline: "TrustworthyBaseline",
    client_id: str,
    round_id: int,
    update: np.ndarray,
    security_tier: int,
    security_class: str,
    cia_impacts: Dict,
    scenario_name: str,
    *,
    FRESHNESS_WINDOW_SEC: int = 300,
    is_malicious_client: bool = False,
    attack_type: str = None
) -> Tuple[bool, Dict]:
    """
    REAL cryptographic enforcement with comprehensive checks.
    Returns: (enforcement_passed, enforcement_info)
    If enforcement_passed is False, caller MUST ban client if enforcement_info["should_ban_client"] is True.
    """

    # 0. Early return for scenarios without enforcement
    if scenario_name in ["B0_Plain_FL", "B1_xMK_CKKS"] and not is_malicious_client:
        return True, {
            "no_enforcement": True,
            "client_id": client_id,
            "round_id": round_id,
            "is_malicious": is_malicious_client
        }

    # Initialize result dictionary
    enforcement_result = {
        "client_id": client_id,
        "round_id": round_id,
        "passed": True,
        "violation_type": None,
        "violation_details": None,
        "is_malicious": is_malicious_client,
        "attack_type": attack_type,
        "pofc_verified": False,
        "signature_verified": False,
        "freshness_verified": False,
        "rate_limit_ok": False,
        "ledger_logged": False,
        "should_ban_client": False  # NEW: Explicit ban flag
    }

    # 1. Enroll client if not already enrolled
    if client_id not in baseline.client_keys:
        try:
            baseline.enroll_client(client_id)
            if DEBUG_MODE:
                print(f"    Enrolled new client: {client_id}")
        except Exception as e:
            if DEBUG_MODE:
                print(f"    Client enrollment failed for {client_id}: {e}")
            enforcement_result.update({
                "passed": False,
                "violation_type": "ENROLLMENT_FAILED",
                "violation_details": str(e),
                "should_ban_client": True  # Can't enroll → ban
            })
            return False, enforcement_result

    # 2. Compute hash of the update
    try:
        update_hash = baseline.compute_update_hash(update)
        enforcement_result["update_hash_short"] = base64.b64encode(update_hash)[:8].decode()
    except Exception as e:
        if DEBUG_MODE:
            print(f"    Hash computation failed for {client_id}: {e}")
        enforcement_result.update({
            "passed": False,
            "violation_type": "HASH_COMPUTATION_FAILED",
            "violation_details": str(e),
            "should_ban_client": True  # Critical failure → ban
        })
        return False, enforcement_result

    # 3. Get current timestamp
    current_timestamp = time.time()

    # 4. Generate PoFC nonce and verify
    # 4. Generate PoFC nonce and verify
    try:
        expected_nonce = baseline.generate_pofc_nonce(client_id, round_id)
        client_nonce = expected_nonce  # nonce itself is valid

        # 🔴 ATTACK INJECTION (ONLY HERE)
        if is_malicious_client and attack_type == "POFC_OVERFLOW":
            max_packets = MAX_PACKETS_PER_NONCE_BY_TIER.get(
                security_tier, BASE_MAX_PACKETS_PER_NONCE
            )
            # Force packet counter beyond limit BEFORE verification
            baseline.packet_counts[client_id][round_id] += max_packets + 1

        # PoFC verification (rate-limit based)
        pofc_verified = baseline.verify_pofc(
            client_id, round_id, client_nonce, current_timestamp, security_tier
        )

        enforcement_result["pofc_verified"] = pofc_verified

        if not pofc_verified:
            current_packet_count = baseline.packet_counts.get(client_id, {}).get(round_id, 0)
            max_packets = MAX_PACKETS_PER_NONCE_BY_TIER.get(
                security_tier, BASE_MAX_PACKETS_PER_NONCE
            )

            if is_malicious_client and attack_type == "POFC_OVERFLOW":
                violation_type = "POFC_RATE_LIMIT_ATTACK"
                violation_details = f"PoFC flooding detected: {current_packet_count}/{max_packets}"
                should_ban = True
            else:
                violation_type = "POFC_RATE_LIMIT_EXCEEDED"
                violation_details = f"Rate limit exceeded: {current_packet_count}/{max_packets}"
                should_ban = False

            enforcement_result.update({
                "passed": False,
                "violation_type": violation_type,
                "violation_details": violation_details,
                "should_ban_client": should_ban,
                "packet_count": current_packet_count,
                "max_packets": max_packets
            })
            return False, enforcement_result


        # If PoFC passed, track rate limit status
        current_packet_count = baseline.packet_counts[client_id][round_id] if client_id in baseline.packet_counts and round_id in baseline.packet_counts[client_id] else 0
        max_packets = MAX_PACKETS_PER_NONCE_BY_TIER.get(security_tier, BASE_MAX_PACKETS_PER_NONCE)
        enforcement_result.update({
            "rate_limit_ok": current_packet_count <= max_packets,
            "packet_count": current_packet_count,
            "max_packets": max_packets
        })

    except Exception as e:
        if DEBUG_MODE:
            print(f"    PoFC verification error for {client_id}: {e}")
        enforcement_result.update({
            "passed": False,
            "violation_type": "PoFC_ERROR",
            "violation_details": str(e),
            "should_ban_client": True  # Critical error → BAN
        })
        return False, enforcement_result

    # 4.5 Freshness / replay protection (T3+)
    # 4.5 Freshness / replay protection
    # Apply freshness checks for all enforced scenarios (recommended),
    # but inject an attack only for malicious clients in REPLAY scenario.

    freshness_round_id = round_id
    freshness_timestamp = current_timestamp

    # 🔴 ATTACK INJECTION: make the malicious client appear stale
    if is_malicious_client and attack_type == "REPLAY_STALE_UPDATE":
        # Make it fail both dimensions robustly:
        # 1) old round (replay of older update)
        freshness_round_id = max(0, round_id - 5)
        # 2) old timestamp outside the freshness window
        freshness_timestamp = current_timestamp - (FRESHNESS_WINDOW_SEC + 10)

    freshness_ok = baseline.verify_freshness(
        client_id,
        freshness_round_id,
        freshness_timestamp,
        last_seen_round=baseline.last_round_seen,
        last_seen_timestamp=baseline.last_timestamp_seen,
        FRESHNESS_WINDOW_SEC=FRESHNESS_WINDOW_SEC
    )

    enforcement_result["freshness_verified"] = freshness_ok
    enforcement_result["freshness_check_round_id"] = freshness_round_id
    enforcement_result["freshness_check_timestamp_delta_sec"] = float(freshness_timestamp - current_timestamp)

    if not freshness_ok:
        enforcement_result.update({
            "passed": False,
            "violation_type": "REPLAY_STALE_UPDATE",
            "violation_details": (
                f"Stale update detected (round={freshness_round_id} vs current={round_id}, "
                f"ts_delta={freshness_timestamp - current_timestamp:.1f}s)"
            ),
            "should_ban_client": True
        })
        return False, enforcement_result



    # 5. Simulate client signature (in real implementation, client would sign)
    try:
        # In simulation, we sign with client's key
        signature = baseline.sign_update(
            client_id, round_id, update_hash, current_timestamp
        )

        # For malicious clients with signature attack, verify will fail
        signature_verified = baseline.verify_signature(
            client_id, round_id, update_hash, current_timestamp, signature
        )

        enforcement_result["signature_verified"] = signature_verified
        enforcement_result["signature_length"] = len(signature)

        if not signature_verified:
            if DEBUG_MODE:
                print(f"    ⚠ Signature verification FAILED for {client_id}")

            # Check if this is an attack scenario
            if (is_malicious_client and attack_type == "BAD_SIGNATURE" and
                baseline.attack_mode == "BAD_SIGNATURE" and
                client_id == baseline.attacker_id):
                violation_type = "SIGNATURE_FORGERY_ATTACK"
                violation_details = f"Malicious signature forgery detected"
                should_ban = True  # MALICIOUS SIGNATURE FORGERY → BAN IMMEDIATELY
                if DEBUG_MODE:
                    print(f"    ⚠ SIGNATURE ATTACK DETECTED AND BLOCKED: {client_id}")
            else:
                violation_type = "SIGNATURE_VERIFICATION_FAILED"
                violation_details = f"Signature verification failed (possible corruption)"
                should_ban = True  # Invalid signature → BAN (could be attack)
                if DEBUG_MODE:
                    print(f"    ⚠ Signature failed for benign client {client_id} - BANNED")

            enforcement_result.update({
                "passed": False,
                "violation_type": violation_type,
                "violation_details": violation_details,
                "should_ban_client": should_ban  # CRITICAL: Set ban flag
            })
            return False, enforcement_result

    except Exception as e:
        if DEBUG_MODE:
            print(f"    Signature error for {client_id}: {e}")
        enforcement_result.update({
            "passed": False,
            "violation_type": "SIGNATURE_ERROR",
            "violation_details": str(e),
            "should_ban_client": True  # Critical error → BAN
        })
        return False, enforcement_result

    # 6. Check freshness (already checked in verify_pofc, but track explicitly)
    #freshness_verified = abs(current_timestamp - current_timestamp) <= FRESHNESS_WINDOW_SEC
    #enforcement_result["freshness_verified"] = freshness_verified

    #if not freshness_verified:
    #    enforcement_result.update({
    #        "passed": False,
    #        "violation_type": "FRESHNESS_VIOLATION",
    #        "violation_details": f"Timestamp {current_timestamp} outside {FRESHNESS_WINDOW_SEC}s window",
    #        "should_ban_client": True  # Freshness violation → BAN
    #    })
    #    return False, enforcement_result

    # 7. Log the successful enforcement
    try:
        ledger_entry_id = baseline.log_client_action(
            client_id,
            "UPDATE_SUBMITTED_ENFORCED",
            {
                "round_id": round_id,
                "update_hash": base64.b64encode(update_hash).decode(),
                "signature": base64.b64encode(signature).decode()[:16] + "...",
                "security_tier": security_tier,
                "security_class": security_class,
                "timestamp": current_timestamp,
                "scenario": scenario_name,
                "is_malicious": is_malicious_client,
                "attack_type": attack_type
            }
        )
        enforcement_result["ledger_logged"] = True
        enforcement_result["ledger_entry_id"] = ledger_entry_id
    except Exception as e:
        if DEBUG_MODE:
            print(f"    Ledger logging failed for {client_id}: {e}")
        # Don't fail the enforcement just because logging failed
        enforcement_result["ledger_logged"] = False

    # 8. Track successful enforcement metrics
    # 🔴 CRITICAL SAFETY NET:
    # Any FAILED enforcement by a malicious client MUST result in ban
    if not enforcement_result["passed"] and enforcement_result["is_malicious"]:
        enforcement_result["should_ban_client"] = True

    enforcement_result.update({
        "security_tier": security_tier,
        "security_class": security_class,
        "timestamp": current_timestamp,
        "successful_enforcement": enforcement_result["passed"]

    })



    return True, enforcement_result

# --------------------------
# MAIN EXPERIMENT CLASS - ENHANCED
# --------------------------
class TADPSecExperiment:
    """Main experiment runner with realistic cryptographic degradation and attack simulation"""

    def __init__(self):
        self.baseline = TrustworthyBaseline()
        self.tadp_admission = TADPAdmission()
        self.clients = {}
        self.clients_scaled = {}
        self.admitted_clients = {}
        self.admitted_client_ids = []
        self.admitted_client_ids_fixed = None
        self.X_test = None
        self.y_test = None
        self.admission_stats = {}
        self.enforcement_stats = defaultdict(lambda: defaultdict(int))
        self.scaler = None
        self.performance_tracker = PerformanceTracker()
        self.attack_simulator = AttackSimulator()
        self.X_val_scaled = None
        self.y_val = None
        self.global_scaler = None
        self.crypto_logger = CryptoLogger(
            os.path.join(CRYPTO_DIR, "crypto_semantics.csv")
        )

        # ✅ ADD THIS CRITICAL LINE:
        self.scenario_statistics = {}  # For tracking protocol enforcement

        # ✅ ALSO ADD THIS for attack tracking:
        self.attack_modes = {}  # Track attack configurations per scenario

        # ✅ CRITICAL FIX: Correct and consistent attack–scenario mapping
        self.malicious_clients_config = {

            # T1 — Integrity attack (Signature forgery)
            'T1_TEST_SIGNATURE_ATTACK': {
                'type': 'BAD_SIGNATURE',
                'malicious_count': 2,
                'attack_params': {'invalid_sig': True},
                'description': 'Signature forgery attack (invalid client signatures)'
            },

            # T2 — Freshness / replay attack
            'T2_TEST_REPLAY_ATTACK': {
                'type': 'REPLAY_STALE_UPDATE',
                'malicious_count': 1,
                'attack_params': {'stale_round': True, 'stale_timestamp': True},
                'description': 'Replay / stale update attack (round_id + timestamp)'
            },

            # T3 — Availability attack (PoFC flooding)
            'T3_TEST_POFC_ATTACK': {
                'type': 'POFC_OVERFLOW',
                'malicious_count': 3,
                'attack_params': {'burst_packets': True},
                'description': 'PoFC packet overflow attack (rate-limit violation)'
            }
        }

        # ✅ Track current malicious clients during scenario execution
        self.current_malicious_clients = []

    def track_protocol_enforcement(self, scenario_name: str, round_stats: Dict):
        """Track protocol enforcement statistics for table generation"""

        if scenario_name not in self.scenario_statistics:
            # Initialize statistics
            # admitted_count = 12  # Default for TADP scenarios
            admitted_count = (
                len(self.admitted_client_ids_fixed)
                if self.admitted_client_ids_fixed is not None
                else len(self.admitted_client_ids)
            )

            if admitted_count == 0 and not scenario_name.startswith('B'):
                admitted_count = self.admission_stats.get('admitted', 0)
            if scenario_name.startswith('B'):
                admitted_count = N_CLIENTS

            self.scenario_statistics[scenario_name] = {
                'total_clients': N_CLIENTS,
                'admitted_clients': admitted_count,
                'protocol_attackers': 0,
                'malicious_clients_banned': 0,
                'dropped_by_baseline': 0,   # ✅ FIX
                'effective_clients_per_round': [],
                'total_enforcement_checks': 0,
                'failed_enforcement_checks': 0,
            }


            # Set protocol attackers for attack scenarios
            if scenario_name in self.malicious_clients_config:
                config = self.malicious_clients_config[scenario_name]
                self.scenario_statistics[scenario_name]['protocol_attackers'] = config['malicious_count']

        stats = self.scenario_statistics[scenario_name]

        # Update statistics
        if 'effective_clients' in round_stats:
            # For attack scenarios, effective clients should decrease as attackers are banned
            effective = round_stats['effective_clients']
            stats['effective_clients_per_round'].append(effective)

        if 'enforcement_failures' in round_stats:
            stats['failed_enforcement_checks'] += round_stats['enforcement_failures']

        if 'protocol_violations' in round_stats:
            violations = round_stats['protocol_violations']

            # Track malicious clients banned
            if 'malicious_clients_banned_total' in violations:
                stats['malicious_clients_banned'] = max(stats['malicious_clients_banned'],
                                                        violations['malicious_clients_banned_total'])

        stats['total_enforcement_checks'] += (
            round_stats.get('enforcement_successes', 0)
            + round_stats.get('enforcement_failures', 0)
        )

    def generate_protocol_enforcement_table(self) -> pd.DataFrame:
        """Generate protocol enforcement table for publication"""

        print("\n" + "="*80)
        print("PROTOCOL ENFORCEMENT RESULTS TABLE")
        print("="*80)

        table_rows = []

        for scenario, stats in self.scenario_statistics.items():
            # Calculate metrics
            total_clients = stats['total_clients']
            admitted = stats['admitted_clients']
            admitted_pct = (admitted / total_clients) * 100

            protocol_attackers = stats.get('protocol_attackers', 0)
            malicious_banned = stats.get('malicious_clients_banned', 0)

            # Calculate average effective clients (excluding banned ones)
            avg_effective = (
                np.mean(stats['effective_clients_per_round'])
                if stats['effective_clients_per_round']
                else admitted
            )

            # Calculate if attack was successfully blocked
            if protocol_attackers > 0:
                attack_blocked = "✅ Yes" if malicious_banned >= protocol_attackers else "❌ No"
                dropped_info = f"{malicious_banned}/{protocol_attackers}"
            else:
                attack_blocked = "N/A"
                dropped_info = "0"

            # Enforcement success rate
            if stats['total_enforcement_checks'] > 0:
                success_rate = max(0.0, 1 - (stats['failed_enforcement_checks'] / stats['total_enforcement_checks']))
                success_pct = f"{success_rate*100:.1f}%"
            else:
                success_pct = "N/A"

            # Determine category and tier
            if scenario.startswith('B'):
                category = 'Baseline'
                if 'xMK' in scenario:
                    tier = 'xMK-CKKS'
                else:
                    tier = 'None'
            elif scenario.startswith('A'):
                category = 'Governance'
                tier = 'None'
            else:
                category = 'Crypto Tier'
                if 'T1' in scenario:
                    tier = 'Plain'
                elif 'T2' in scenario:
                    tier = 'DF+KS'
                elif 'T3' in scenario:
                    tier = 'xMK-CKKS'
                elif 'T4' in scenario:
                    tier = 'CKKS+DoubleKS'
                else:
                    tier = 'Unknown'

            # NEW: Show blocked by TADP-sec (baseline controls)
            if protocol_attackers > 0:
                blocked_by_tadp_sec = f"{malicious_banned} attackers"
            else:
                blocked_by_tadp_sec = "0"

            table_rows.append({
                'Scenario': scenario,
                'Total Clients': total_clients,
                'Admitted by TADP': f"{admitted} ({admitted_pct:.0f}%)",
                'Protocol Attackers': protocol_attackers,
                'Blocked by TADP-Sec': blocked_by_tadp_sec,  # NEW COLUMN
                'Dropped by Baseline': dropped_info,
                'Effective Clients': f"{avg_effective:.1f}",  # SHOULD BE LOWER FOR ATTACK SCENARIOS
                'Enforcement Success': success_pct,
                'Attack Blocked?': attack_blocked,
                'Security Tier': tier
            })

        # Create DataFrame
        table_df = pd.DataFrame(table_rows)

        # Sort by scenario type
        scenario_order = {
            'B0_Plain_FL': 0,
            'B1_xMK_CKKS': 1,
            'A1_TADP_admission_only': 2,
            'T1_Plain_TADP': 3,
            'T1_TEST_SIGNATURE_ATTACK': 4,
            'T2_DF_KS_TADP': 5,
            'T2_TEST_REPLAY_ATTACK': 6,
            'T3_XMK_CKKS_TADP': 7,
            'T3_TEST_POFC_ATTACK': 8,
            'T4_CKKS_KS_TADP': 9
        }

        table_df['order'] = table_df['Scenario'].map(scenario_order)
        table_df = table_df.sort_values('order').drop('order', axis=1)

        # Save table
        table_path = os.path.join(STATS_DIR, "protocol_enforcement_table.csv")
        table_df.to_csv(table_path, index=False)

        # Also save LaTeX version
        latex_path = os.path.join(STATS_DIR, "protocol_enforcement_table.tex")
        with open(latex_path, 'w') as f:
            f.write(table_df.to_latex(index=False))

        print(f"\n✓ Protocol enforcement table generated")
        print(f"  • CSV: {table_path}")
        print(f"  • LaTeX: {latex_path}")

        # Print table
        print("\n" + table_df.to_string(index=False))

        # CRITICAL: Verify that attack scenarios show correct effective clients
        print("\n" + "="*80)
        print("VERIFICATION OF ATTACK SCENARIO RESULTS:")
        print("="*80)

        for scenario in ["T1_TEST_SIGNATURE_ATTACK", "T2_TEST_REPLAY_ATTACK", "T3_TEST_POFC_ATTACK"]:
            if scenario in self.scenario_statistics:
                stats = self.scenario_statistics[scenario]
                admitted = stats['admitted_clients']
                attackers = stats['protocol_attackers']
                banned = stats.get('malicious_clients_banned', 0)

                expected_effective = admitted - banned
                actual_effective = None

                # Find the row in table_df
                row = table_df[table_df['Scenario'] == scenario]
                if not row.empty:
                    actual_effective = float(row['Effective Clients'].values[0])

                status = "✓ CORRECT" if expected_effective == actual_effective else f"✗ WRONG (expected {expected_effective:.1f})"
                print(f"{scenario}: Admitted={admitted}, Attackers={attackers}, Banned={banned}, Effective={actual_effective} {status}")

        return table_df


    def get_scenario_summary(self, scenario_name: str) -> Dict:
        """Get comprehensive summary for a scenario"""
        if scenario_name not in self.scenario_statistics:
            return {}

        stats = self.scenario_statistics[scenario_name]

        # Calculate averages
        avg_effective_clients = np.mean(stats['effective_clients_per_round']) \
            if stats['effective_clients_per_round'] else 0

        avg_malicious_blocked = stats.get('malicious_clients_banned', 0)
        avg_benign_blocked = 0


        # Calculate enforcement success rate
        if stats['total_enforcement_checks'] > 0:
            enforcement_success_rate = 1.0 - (stats['failed_enforcement_checks'] / stats['total_enforcement_checks'])
        else:
            enforcement_success_rate = 1.0

        # Calculate malicious blocking rate
        if stats['protocol_attackers'] > 0:
            malicious_blocking_rate = avg_malicious_blocked / stats['protocol_attackers']
        else:
            malicious_blocking_rate = 0.0

        return {
            'scenario': scenario_name,
            'total_clients': stats['total_clients'],
            'admitted_clients': stats['admitted_clients'],
            'protocol_attackers': stats['protocol_attackers'],
            'dropped_by_baseline': stats.get('malicious_clients_banned', 0),
            'avg_effective_clients': avg_effective_clients,
            'enforcement_success_rate': enforcement_success_rate,
            'avg_malicious_blocked': avg_malicious_blocked,
            'avg_benign_blocked': avg_benign_blocked,
            'malicious_blocking_rate': malicious_blocking_rate,
            'total_enforcement_checks': stats['total_enforcement_checks']
        }

    def generate_enforcement_report(self) -> pd.DataFrame:
        """Generate comprehensive enforcement report"""
        report_data = []

        for scenario in sorted(self.scenario_statistics.keys()):
            summary = self.get_scenario_summary(scenario)
            if not summary:
                continue

            # Determine if attack scenario
            is_attack = scenario in self.malicious_clients_config

            # Format blocking information
            if is_attack:
                blocking_info = f"{summary['dropped_by_baseline']}/{summary['protocol_attackers']} blocked"
                blocking_success = "✅ 100%" if summary['malicious_blocking_rate'] >= 0.99 else f"⚠ {summary['malicious_blocking_rate']*100:.1f}%"
            else:
                blocking_info = "N/A"
                blocking_success = "N/A"

            report_data.append({
                'Scenario': scenario,
                'Category': 'Baseline' if scenario.startswith('B') else 'Governance' if scenario.startswith('A') else 'Crypto Tier',
                'Total Clients': summary['total_clients'],
                'Admitted by TADP': f"{summary['admitted_clients']} ({summary['admitted_clients']/summary['total_clients']*100:.0f}%)",
                'Protocol Attackers': summary['protocol_attackers'],
                'Malicious Blocked': blocking_info,
                'Blocking Success': blocking_success,
                'Avg Effective Clients': f"{summary['avg_effective_clients']:.1f}",
                'Enforcement Success Rate': f"{summary['enforcement_success_rate']*100:.1f}%",
                'Avg Benign Blocked': f"{summary['avg_benign_blocked']:.2f}",
                'Attack Test': 'Yes' if is_attack else 'No'
            })

        report_df = pd.DataFrame(report_data)

        # Sort by scenario type
        scenario_order = {
            'B0_Plain_FL': 0,
            'B1_xMK_CKKS': 1,
            'A1_TADP_admission_only': 2,
            'T1_Plain_TADP': 3,
            'T1_TEST_SIGNATURE_ATTACK': 4,
            'T2_DF_KS_TADP': 5,
            'T2_TEST_REPLAY_ATTACK': 6,
            'T3_XMK_CKKS_TADP': 7,
            'T3_TEST_POFC_ATTACK': 8,
            'T4_CKKS_KS_TADP': 9
        }

        report_df['order'] = report_df['Scenario'].map(scenario_order)
        report_df = report_df.sort_values('order').drop('order', axis=1)

        # Save report
        report_path = os.path.join(STATS_DIR, "protocol_enforcement_report.csv")
        report_df.to_csv(report_path, index=False)

        return report_df


    def setup_experiment(self):
        """Setup experiment with Credit Card Fraud Detection dataset"""
        print("=" * 80)
        print("TADP-SEC EXPERIMENT v8.2: PROFESSIONAL PUBLICATION-READY")
        print("=" * 80)

        dataset_path = "/content/creditcard.csv"
        if not os.path.exists(dataset_path):
            alt_paths = ["./creditcard.csv", "creditcard.csv", "data/creditcard.csv"]
            for path in alt_paths:
                if os.path.exists(path):
                    dataset_path = path
                    break

            if not os.path.exists(dataset_path):
                raise FileNotFoundError("creditcard.csv not found. Please download it to /content/creditcard.csv")

        print(f"  Loading dataset from: {dataset_path}")
        df = pd.read_csv(dataset_path)
        print(f"  ✅ Dataset loaded: {df.shape[0]:,} transactions")

        # Preprocessing
        if 'Time' in df.columns:
            df = df.drop(columns=['Time'])

        if 'Amount' in df.columns:
            from sklearn.preprocessing import RobustScaler
            amount_scaler = RobustScaler()
            df['Amount'] = amount_scaler.fit_transform(df[['Amount']].values)

        X = df.drop(columns=['Class']).values
        y = df['Class'].values

        print(f"\n  GLOBAL DATASET STATISTICS:")
        print(f"  • Total samples: {len(X):,}")
        print(f"  • Features: {X.shape[1]}")
        print(f"  • Global fraud rate: {np.mean(y)*100:.4f}%")

        # Store global dataset
        self.X_global = X
        self.y_global = y

        # Create clients with proper data splitting
        self._create_clients_with_local_splits()

        # Create validation set for threshold tuning
        self.X_val_scaled, self.y_val = self.create_validation_set()
        if self.X_val_scaled is not None:
            print(f"  • Validation set for threshold tuning: {len(self.y_val):,} samples")
            print(f"  • Validation fraud rate: {np.mean(self.y_val)*100:.4f}%")

        print(f"\n  EXPERIMENT SETUP COMPLETE:")
        print(f"  • {N_CLIENTS} clients created with non-IID Dirichlet distribution")
        print(f"  • Global test set: {len(self.X_global_test):,} samples (20%)")
        # print(f"  • Rounds: {ROUNDS}, Seeds: {len(SEEDS)} (statistical power)")
        print(f"  • Main scenarios: {ROUNDS} rounds, {len(FULL_SEEDS)} seeds")
        print(f"  • T4 feasibility tier: {T4_ROUNDS} rounds, {len(T4_SEEDS)} seed")

        return None

    def _create_clients_with_local_splits(self):
        """Create clients with NO data leakage - clean split"""

        print(f"\n  Creating {N_CLIENTS} clients with CLEAN data splitting...")

        self.clients = {}
        self.clients_scaled = {}
        self.client_scalers = {}

        n_clients = N_CLIENTS

        from sklearn.model_selection import train_test_split

        print(f"  Step 1: Creating global test set (20% of total data)...")
        X_train_all, X_global_test, y_train_all, y_global_test = train_test_split(
            self.X_global, self.y_global,
            test_size=0.2,
            random_state=MASTER_SEED,
            stratify=self.y_global
        )

        self.X_global_test = X_global_test
        self.y_global_test = y_global_test

        print(f"    • Global test set size: {len(X_global_test):,} samples")
        print(f"    • Global test fraud rate: {np.mean(y_global_test)*100:.4f}%")
        print(f"    • Training data for clients: {len(X_train_all):,} samples (80%)")

        print(f"  Step 2: Distributing {len(X_train_all):,} training samples among {n_clients} clients...")

        alpha = 0.8

        fraud_idx = np.where(y_train_all == 1)[0]
        normal_idx = np.where(y_train_all == 0)[0]

        print(f"    • Training fraud samples: {len(fraud_idx):,}")
        print(f"    • Training normal samples: {len(normal_idx):,}")

        np.random.shuffle(fraud_idx)
        np.random.shuffle(normal_idx)

        fraud_proportions = np.random.dirichlet(np.ones(n_clients) * alpha)
        fraud_counts = (fraud_proportions * len(fraud_idx)).astype(int)
        fraud_counts[-1] = len(fraud_idx) - np.sum(fraud_counts[:-1])

        normal_proportions = np.random.dirichlet(np.ones(n_clients) * alpha * 1.5)
        normal_counts = (normal_proportions * len(normal_idx)).astype(int)
        normal_counts[-1] = len(normal_idx) - np.sum(normal_counts[:-1])

        fraud_start = 0
        normal_start = 0

        print(f"  Step 3: Assigning data to clients...")

        for client_idx in range(n_clients):
            client_id = f"client_{client_idx:03d}"

            fraud_end = fraud_start + fraud_counts[client_idx]
            client_fraud_idx = fraud_idx[fraud_start:fraud_end]

            normal_end = normal_start + normal_counts[client_idx]
            client_normal_idx = normal_idx[normal_start:normal_end]

            client_idx_all = np.concatenate([client_fraud_idx, client_normal_idx])
            np.random.shuffle(client_idx_all)

            if len(client_idx_all) == 0:
                client_idx_all = np.random.choice(len(X_train_all),
                                                min(100, len(X_train_all)),
                                                replace=False)

            X_client = X_train_all[client_idx_all].copy()
            y_client = y_train_all[client_idx_all].copy()

            noise = np.random.randn(*X_client.shape) * 0.02
            X_client += noise

            client_scaler = StandardScaler()
            X_client_scaled = client_scaler.fit_transform(X_client)

            self.clients[client_id] = (X_client, y_client)
            self.clients_scaled[client_id] = (X_client_scaled, y_client)
            self.client_scalers[client_id] = client_scaler

            fraud_start = fraud_end
            normal_start = normal_end

        print(f"  Step 4: Scaling global test set...")

        all_client_train_data = []
        for client_id in self.clients:
            X_train, _ = self.clients[client_id]
            all_client_train_data.append(X_train)

        if all_client_train_data:
            all_client_train = np.vstack(all_client_train_data)
            self.global_scaler = StandardScaler()
            self.global_scaler.fit(all_client_train)
            self.X_global_test_scaled = self.global_scaler.transform(self.X_global_test)
        else:
            self.global_scaler = StandardScaler()
            self.global_scaler.fit(X_train_all)
            self.X_global_test_scaled = self.global_scaler.transform(self.X_global_test)

        print(f"\n  CLEAN DATA SPLITTING SUMMARY:")
        print(f"  {'Client':<12} {'Samples':<10} {'Fraud':<10} {'Fraud %':<10}")
        print(f"  {'-'*12} {'-'*10} {'-'*10} {'-'*10}")

        total_samples = 0
        total_fraud = 0

        for client_id in sorted(self.clients.keys()):
            X_train, y_train = self.clients[client_id]

            n_samples = len(y_train)
            n_fraud = np.sum(y_train == 1)
            fraud_rate = n_fraud / n_samples if n_samples > 0 else 0

            total_samples += n_samples
            total_fraud += n_fraud

            print(f"  {client_id:<12} {n_samples:<10} {n_fraud:<10} {fraud_rate*100:<10.4f}%")

        print(f"\n  {'TOTAL CLIENTS':<12} {total_samples:<10} {total_fraud:<10} {total_fraud/total_samples*100:<10.4f}%")

        print(f"\n  GLOBAL TEST SET (COMPLETELY SEPARATE):")
        print(f"  • Samples: {len(self.X_global_test):,}")
        print(f"  • Fraud rate: {np.mean(self.y_global_test)*100:.4f}%")

    def create_validation_set(self):
        """Create validation set from training data for threshold tuning"""
        all_X_train = []
        all_y_train = []
        for client_id in self.clients:
            X_train, y_train = self.clients[client_id]
            all_X_train.append(X_train)
            all_y_train.append(y_train)

        if all_X_train:
            X_all = np.vstack(all_X_train)
            y_all = np.concatenate(all_y_train)

            from sklearn.model_selection import train_test_split
            X_train_final, X_val, y_train_final, y_val = train_test_split(
                X_all, y_all,
                test_size=0.2,
                random_state=MASTER_SEED + 999,
                stratify=y_all
            )

            X_val_scaled = self.global_scaler.transform(X_val)

            return X_val_scaled, y_val

        return None, None

    def run_tadp_admission(self):
        """Run TADP admission process"""
        print("\n" + "="*80)
        print("TADP ADMISSION PROCESS")
        print("="*80)

        admission_results = self.tadp_admission.evaluate_clients(self.clients)

        self.admitted_client_ids = []
        admission_stats = {
            'total': len(admission_results),
            'admitted': 0,
            'rejected': 0,
            'by_outcome': defaultdict(int)
        }

        for client_id, result in admission_results.items():
            if result['admitted']:
                self.admitted_client_ids.append(client_id)
                self.admitted_clients[client_id] = result
                admission_stats['admitted'] += 1
            else:
                admission_stats['rejected'] += 1

            admission_stats['by_outcome'][result['review_outcome']] += 1

        self.admission_stats = admission_stats

        print(f"\nAdmission Results:")
        print(f"  Total clients: {admission_stats['total']}")
        print(f"  Admitted: {admission_stats['admitted']} ({admission_stats['admitted']/admission_stats['total']*100:.1f}%)")
        print(f"  Rejected: {admission_stats['rejected']} ({admission_stats['rejected']/admission_stats['total']*100:.1f}%)")

        print(f"\nAdmitted Clients (by PScore):")
        admitted_with_scores = [(client_id, self.admitted_clients[client_id]['pscore'])
                                for client_id in self.admitted_client_ids]
        admitted_sorted = sorted(admitted_with_scores, key=lambda x: x[1], reverse=True)

        for i, (client_id, pscore) in enumerate(admitted_sorted):
            print(f"  {i+1}. {client_id}: PScore={pscore:.2f}")

        return admission_results

    def enforce_consistent_admission(self, scenario_name):
        """Ensure ALL TADP scenarios use the EXACT same admitted clients."""
        if scenario_name.startswith('B'):
            return list(self.clients.keys())
        else:
            # Handle attack test scenarios
            if scenario_name in ["T1_TEST_SIGNATURE_ATTACK", "T2_TEST_REPLAY_ATTACK", "T3_TEST_POFC_ATTACK"]:
                base_scenario = scenario_name.split('_TEST_')[0]
                if base_scenario == "T1":
                    scenario_name = "T1_Plain_TADP"
                elif base_scenario == "T2":
                    scenario_name = "T2_DF_KS_TADP"
                elif base_scenario == "T3":
                    scenario_name = "T3_XMK_CKKS_TADP"

            if not self.admitted_client_ids:
                print(f"  Running TADP admission for scenario {scenario_name}...")
                self.run_tadp_admission()

            # ✅ CRITICAL: Initialize if None
            if not hasattr(self, 'admitted_client_ids_fixed') or self.admitted_client_ids_fixed is None:
                self.admitted_client_ids_fixed = self.admitted_client_ids.copy()
                print(f"  ✓ Fixed admitted clients: {len(self.admitted_client_ids_fixed)} clients")

            print(f"  Using {len(self.admitted_client_ids_fixed)} admitted clients "
                  f"(same across all TADP scenarios)")
            return self.admitted_client_ids_fixed

    def run_scenario(
        self,
        scenario_name: str,
        seeds: List[int] = SEEDS,
        rounds: int = ROUNDS
    ) -> pd.DataFrame:
        """Run a specific scenario with PROTOCOL-LEVEL enforcement only"""

        print(f"\n{'='*60}")
        print(f"SCENARIO: {scenario_name}")
        print(f"{'='*60}")

        # ✅ CONFIGURE PROTOCOL-LEVEL ENFORCEMENT TESTING:
        # ✅ CONFIGURE PROTOCOL-LEVEL ENFORCEMENT TESTING (FIXED NAMES + TYPES)
        # IMPORTANT:
        #   - scenario_name must match run_all_scenarios()
        #   - attack_type strings must match enforce_baseline_controls() checks

        self.baseline.attack_mode = None
        self.baseline.attacker_id = None

        if scenario_name == "T1_TEST_SIGNATURE_ATTACK":
            # Signature forgery / invalid signature
            self.baseline.attack_mode = "BAD_SIGNATURE"
            print("  ⚠ Protocol Test: BAD_SIGNATURE (signature forgery)")

        elif scenario_name == "T2_TEST_REPLAY_ATTACK":
            # Replay / stale update (round/timestamp)
            # enforce_baseline_controls() expects: "REPLAY_STALE_UPDATE"
            self.baseline.attack_mode = "REPLAY_STALE_UPDATE"
            print("  ⚠ Protocol Test: REPLAY_STALE_UPDATE (replay / stale round+timestamp)")

        elif scenario_name == "T3_TEST_POFC_ATTACK":
            # PoFC flooding / rate-limit overflow
            self.baseline.attack_mode = "POFC_OVERFLOW"
            print("  ⚠ Protocol Test: POFC_OVERFLOW (PoFC rate-limit overflow)")


        all_results = []

        for seed_idx, seed in enumerate(seeds):
            print(f"\n  [Seed {seed_idx+1}/{len(seeds)}]")

            np.random.seed(seed)
            random.seed(seed)

            # ✅ CRITICAL: reset protocol state between seeds
            self.baseline.reset_protocol_state()

            # ✅ Initialize banned clients per seed (keep it here so it doesn't leak across seeds)
            banned_clients = set()

            # Determine active clients
            active_client_ids = self.enforce_consistent_admission(scenario_name)

            # ✅ Select malicious clients for attack scenarios (CONFIG-DRIVEN, CONSISTENT)
            malicious_clients = []
            if scenario_name in self.malicious_clients_config:
                k = int(self.malicious_clients_config[scenario_name].get("malicious_count", 0))
                k = min(k, len(active_client_ids))
                malicious_clients = active_client_ids[:k]  # deterministic; or random.sample(...) if you prefer
                print(f"    Malicious clients selected ({k}): {malicious_clients}")
            else:
                malicious_clients = []

            # ✅ bind attacker id (used by baseline controls if needed)
            self.baseline.attacker_id = malicious_clients[0] if malicious_clients else None


            if not active_client_ids:
                print(f"    No active clients")
                continue

            # Get features dimension
            first_client_id = active_client_ids[0]
            X_train_first, y_train_first = self.clients_scaled[first_client_id]
            n_features = X_train_first.shape[1]

            # Initialize model
            total_fraud = 0
            total_samples = 0
            for client_id in active_client_ids:
                _, y_train = self.clients_scaled[client_id]
                total_fraud += np.sum(y_train == 1)
                total_samples += len(y_train)

            fraud_rate = total_fraud / total_samples if total_samples > 0 else 0.001
            if fraud_rate == 0:
                fraud_rate = 0.001

            global_model = LogisticRegressionModel(n_features, fraud_rate=fraud_rate)

            crypto = CryptoSim()

            # xMK session preparation policy:
            # - B1 is a crypto-only baseline, so it uses all clients.
            # - T3 normal run prepares xMK per round using currently valid admitted clients.
            # - T3 attack run prepares xMK only AFTER baseline controls remove attackers.
            if scenario_name == "B1_xMK_CKKS":
                crypto.prepare_xmk_session(list(self.clients.keys()))

            # --- T4 CKKS + DOUBLE KEY SWITCHING INITIALIZATION ---
            t4_crypto = None
            client_crypto_keys = {}

            if scenario_name == "T4_CKKS_KS_TADP":
                print("    [T4 CKKS] Initializing asymmetric RLWE CKKS + double key switching...")
                print(
                    f"      n={T4_CKKS_POLY_MODULUS_DEGREE}, "
                    f"log2(q)={T4_CKKS_LOGQ}, "
                    f"scale={T4_CKKS_SCALE}, "
                    f"noise={T4_CKKS_NOISE_STD}"
                )

                t4_crypto = SimulatedCKKSDoubleKSCrypto(
                    n=T4_CKKS_POLY_MODULUS_DEGREE,
                    logq=T4_CKKS_LOGQ,
                    scale=T4_CKKS_SCALE,
                    noise_std=T4_CKKS_NOISE_STD
                )

                for cid in active_client_ids:
                    client_crypto_keys[cid] = t4_crypto.enroll_client(cid)

            # Track initial counts
            initial_benign_count = len(active_client_ids) - len(malicious_clients)
            initial_malicious_count = len(malicious_clients)

            for round_id in range(1, rounds + 1):
                round_start = time.time()
                t4_server_ks_applied = False
                t4_server_ks_count = 0

                # xMK session state for this round
                xmk_round_aborted = False
                xmk_encryption_started = False
                xmk_session_ready = False
                xmk_session_clients = []

                # ------------------------------------------------------------
                # xMK session preparation policy
                # ------------------------------------------------------------
                # Option B main path:
                # Build xMK only from currently valid clients.
                #
                # Option A fail-safe:
                # If a client is rejected after xMK encryption starts,
                # abort the round and re-key next round.
                if scenario_name == "B1_xMK_CKKS":
                    xmk_session_clients = list(self.clients.keys())
                    crypto.prepare_xmk_session(xmk_session_clients)
                    crypto.reset_xmk_round(round_id)
                    xmk_session_ready = True

                elif scenario_name == "T3_XMK_CKKS_TADP":
                    xmk_session_clients = [
                        cid for cid in active_client_ids
                        if cid not in banned_clients
                    ]
                    crypto.prepare_xmk_session(xmk_session_clients)
                    crypto.reset_xmk_round(round_id)
                    xmk_session_ready = True

                elif scenario_name == "T3_TEST_POFC_ATTACK":
                    # Wait until baseline enforcement removes attackers.
                    xmk_session_clients = []
                    xmk_session_ready = False

                total_samples = sum(
                    len(self.clients_scaled[cid][1])
                    for cid in active_client_ids
                    if cid not in banned_clients
                )

                # ------------------------------------------------------------
                # xMK threshold policy
                # ------------------------------------------------------------
                if scenario_name in XMK_SCENARIOS:
                    # xMK requires all valid participants in the effective session.
                    # Option B: session is built only from validated clients.
                    # Option A fail-safe: if a participant is rejected after session start,
                    # the current round is aborted and the next round is re-keyed.
                    XMK_RHO = 1.0

                    if xmk_session_ready:
                        XMK_MIN_SHARES = len(xmk_session_clients)
                        n_keyholders = len(xmk_session_clients)

                        if round_id == 1:
                            print(
                                f"    {scenario_name} xMK threshold: "
                                f"requires ALL {XMK_MIN_SHARES} valid session shares (ρ=1.0)"
                            )
                    else:
                        # T3 attack scenario waits until baseline controls remove attackers.
                        # The xMK session will be prepared later in the same round.
                        XMK_MIN_SHARES = 0
                        n_keyholders = 0

                else:
                    XMK_RHO = 0.70
                    XMK_MIN_SHARES = 2
                    n_keyholders = 0
                client_cts = []
                t4_plain_updates = []
                client_bias_updates = []
                client_weights = []
                enforcement_failures = 0
                enforcement_successes = 0
                crypto_infos = []

                # Track protocol violations (separate from crypto failures) - ENHANCED
                protocol_violations = {
                    'pofc_violations_malicious': 0,
                    'pofc_violations_benign': 0,
                    'signature_violations_malicious': 0,
                    'signature_violations_benign': 0,
                    'freshness_violations': 0,
                    'rate_limit_violations': 0,
                    'total_violations': 0,
                    'malicious_clients_blocked_this_round': 0,  # NEW: Track per round
                    'benign_clients_blocked_this_round': 0,     # NEW: Track per round
                    'malicious_clients_banned_total': len([c for c in banned_clients if c in malicious_clients]),  # NEW: Track total banned malicious
                    'benign_clients_banned_total': len([c for c in banned_clients if c not in malicious_clients]), # NEW: Track total banned benign
                    'round_id': round_id                         # NEW: Track which round
                }

                w_global = global_model.weights.copy()
                b_global = global_model.bias

                # Calculate available clients for this round (excluding banned)
                available_clients = [cid for cid in active_client_ids if cid not in banned_clients]
                remaining_malicious = [cid for cid in available_clients if cid in malicious_clients]
                remaining_benign = [cid for cid in available_clients if cid not in malicious_clients]

                if DEBUG_MODE and round_id == 1:
                    print(f"    Round {round_id}: {len(available_clients)} available clients "
                          f"({len(remaining_malicious)} malicious, {len(remaining_benign)} benign)")

                for client_id in available_clients:
                    #uSE CONINUE: This prevents a client banned earlier in the same round from still being processed later in that same round.
                    if client_id in banned_clients:
                        continue

                    # Determine if this client is malicious
                    is_malicious = client_id in malicious_clients
                    attack_type = self.baseline.attack_mode if is_malicious else None

                    X_client, y_client = self.clients_scaled[client_id]

                    # Get security metadata
                    security_tier = 1
                    security_class = 'C1_PUBLIC'
                    cia_impacts = {'confidentiality': 1, 'integrity': 1, 'availability': 1}

                    if client_id in self.admitted_clients:
                        security_tier = self.admitted_clients[client_id]['security_tier']
                        security_class = self.admitted_clients[client_id]['security_class_str']
                        cia_impacts = self.admitted_clients[client_id]['cia_impacts']

                    # Local training - NO SEMANTIC ATTACKS
                    local_model = LogisticRegressionModel(n_features, fraud_rate=fraud_rate)
                    local_model.weights = w_global.copy()
                    local_model.bias = b_global

                    weight_update, bias_update = local_model.train_on_client(
                        X_client, y_client,
                        learning_rate=LR,
                        epochs=LOCAL_EPOCHS,
                        batch_size=BATCH_SIZE,
                        is_malicious=False,
                        attack_type=None,
                        scenario_name=scenario_name
                    )

                    # Baseline controls (protocol enforcement) - UPDATED CALL
                    controls_enforced, enforcement_info = enforce_baseline_controls(
                        self.baseline,
                        client_id,
                        round_id,
                        weight_update,
                        security_tier,
                        security_class,
                        cia_impacts,
                        scenario_name,
                        FRESHNESS_WINDOW_SEC=300,
                        is_malicious_client=is_malicious,
                        attack_type=attack_type
                    )

                    # Track protocol violation reasons AND BAN CLIENT PERMANENTLY if needed
                    if not controls_enforced:
                        violation_type = enforcement_info.get('violation_type', '')
                        should_ban = enforcement_info.get('should_ban_client', False)

                        # ------------------------------------------------------------
                        # Categorize protocol violation
                        # ------------------------------------------------------------
                        if 'PoFC' in violation_type or 'POFC' in violation_type:
                            if is_malicious:
                                protocol_violations['pofc_violations_malicious'] += 1
                            else:
                                protocol_violations['pofc_violations_benign'] += 1

                        elif 'SIGNATURE' in violation_type:
                            if is_malicious:
                                protocol_violations['signature_violations_malicious'] += 1
                            else:
                                protocol_violations['signature_violations_benign'] += 1

                        elif 'FRESHNESS' in violation_type or 'REPLAY' in violation_type:
                            protocol_violations['freshness_violations'] += 1

                        elif 'RATE_LIMIT' in violation_type:
                            protocol_violations['rate_limit_violations'] += 1

                        # ------------------------------------------------------------
                        # BAN CLIENT(S) PERMANENTLY IF REQUIRED
                        # ------------------------------------------------------------
                        if should_ban:

                            if is_malicious:
                                # Strong attacker policy:
                                # Once any configured malicious client is detected,
                                # ban all configured malicious clients for this scenario.
                                # This supports Option B: clean xMK session after baseline filtering.
                                for attacker_id in malicious_clients:
                                    if attacker_id not in banned_clients:
                                        banned_clients.add(attacker_id)
                                        protocol_violations['malicious_clients_blocked_this_round'] += 1

                                        if DEBUG_MODE:
                                            print(
                                                f"    ⚠ BANNED malicious client {attacker_id}: "
                                                f"{violation_type}"
                                            )

                            else:
                                # Benign client failed enforcement unexpectedly.
                                # Ban only this client.
                                if client_id not in banned_clients:
                                    banned_clients.add(client_id)
                                    protocol_violations['benign_clients_blocked_this_round'] += 1

                                    if DEBUG_MODE:
                                        print(
                                            f"    ⚠ BANNED benign client {client_id}: "
                                            f"{violation_type}"
                                        )

                            # ------------------------------------------------------------
                            # OPTION A FAIL-SAFE
                            # ------------------------------------------------------------
                            # If xMK encryption has already started, the current xMK
                            # session may include a now-rejected participant.
                            # In that case, abort the current round and re-key next round.
                            if scenario_name in XMK_SCENARIOS and xmk_encryption_started:
                                xmk_round_aborted = True

                                if DEBUG_MODE:
                                    print(
                                        f"    ⚠ xMK round {round_id} marked for abort: "
                                        f"client rejected after xMK encryption started."
                                    )

                        enforcement_failures += 1
                        continue

                    enforcement_successes += 1

                    # ------------------------------------------------------------
                    # OPTION B MAIN BEHAVIOR
                    # ------------------------------------------------------------
                    # For the T3 attack scenario, build the xMK session only after
                    # baseline controls have removed malicious clients.
                    # Therefore, the xMK session is created from valid clients only.
                    if scenario_name == "T3_TEST_POFC_ATTACK" and not xmk_session_ready:
                        xmk_session_clients = [
                            cid for cid in active_client_ids
                            if cid not in banned_clients
                        ]

                        if not xmk_session_clients:
                            xmk_round_aborted = True
                            continue

                        crypto.prepare_xmk_session(xmk_session_clients)
                        crypto.reset_xmk_round(round_id)
                        xmk_session_ready = True
                        n_keyholders = len(xmk_session_clients)
                        XMK_MIN_SHARES = len(xmk_session_clients)

                        if round_id == 1:
                            print(
                                f"    T3 attack scenario: xMK session re-keyed after enforcement "
                                f"with {len(xmk_session_clients)} valid clients"
                            )

                    # ------------------------------------------------------------
                    # FedAvg scaling
                    # ------------------------------------------------------------
                    # Use the currently valid, non-banned clients as the denominator.
                    # This prevents banned clients from influencing FedAvg weights.
                    valid_total_samples = sum(
                        len(self.clients_scaled[cid][1])
                        for cid in active_client_ids
                        if cid not in banned_clients
                    )

                    weight_scalar = (
                        len(y_client) / valid_total_samples
                        if valid_total_samples > 0 else 0.0
                    )
                    # Map attack-test scenarios to their underlying crypto tier
                    crypto_base_scenario = scenario_name

                    if scenario_name == "T1_TEST_SIGNATURE_ATTACK":
                        crypto_base_scenario = "T1_Plain_TADP"
                    elif scenario_name == "T2_TEST_REPLAY_ATTACK":
                        crypto_base_scenario = "T2_DF_KS_TADP"
                    elif scenario_name == "T3_TEST_POFC_ATTACK":
                        crypto_base_scenario = "T3_XMK_CKKS_TADP"

                    if crypto_base_scenario == "T2_DF_KS_TADP":
                        # DF+KS performs encrypted scalar-vector multiplication internally:
                        # Enc(weight_scalar) × Enc(update_vec)
                        vec_for_crypto = weight_update
                        scalar_for_crypto = weight_scalar
                    else:
                        # Other tiers receive the already weighted update.
                        vec_for_crypto = weight_update * weight_scalar

                        if crypto_base_scenario == "T4_CKKS_KS_TADP":
                            t4_plain_updates.append(vec_for_crypto.copy())

                        scalar_for_crypto = 1.0

                    # --- T4 CKKS + DOUBLE KEY SWITCHING ENCRYPTION ---
                    if scenario_name == "T4_CKKS_KS_TADP":
                        c_start = time.time()

                        ct_weight = t4_crypto.encrypt_client_update(
                            client_id=client_id,
                            update_vec=vec_for_crypto
                        )

                        actual_c_time = (time.time() - c_start) * 1000

                        # Use parameter-aware calibrated timing.
                        # The cryptographic object now uses algebraic RLWE over Rq,
                        # but this is still a Python research prototype, not optimized SEAL/OpenFHE runtime.
                        c_time = max(actual_c_time, estimate_t4_client_encrypt_ms())

                        crypto_info = {
                            'crypto_type': 'ckks_double_ks_rlwe',
                            'mechanism': (
                                f'T4: CKKS Enc(pk_i) | n={T4_CKKS_POLY_MODULUS_DEGREE}, '
                                f'logq={T4_CKKS_LOGQ}, scale=2^20, noise={T4_CKKS_NOISE_STD}'
                            ),
                            'comm_factor': CKKS_DOUBLE_KS_FACTOR,
                            'crypto_overhead_ms': c_time,
                            'shares_required': 1,
                            'ks_applied': False,
                            'needs_ks': True,
                            'can_finalize': True,
                            'effective_noise_std': ct_weight.effective_noise_std(),
                            'he_noise_std': ct_weight.noise_std,
                            'ciphertext_bytes': ct_weight.get_size_bytes(),
                        }

                    else:
                        ct_weight, crypto_info = apply_crypto_effects_realistic(
                            crypto=crypto,
                            scenario_name=scenario_name,
                            round_id=round_id,
                            client_id=client_id,
                            update_vec=vec_for_crypto,
                            weight_scalar=scalar_for_crypto,
                            available_decrypt_shares=enforcement_successes,
                            required_decrypt_shares=n_keyholders if n_keyholders > 0 else 1
                        )
                    if scenario_name in XMK_SCENARIOS:
                        xmk_encryption_started = True

                    client_cts.append(ct_weight)
                    client_bias_updates.append(bias_update * weight_scalar)
                    client_weights.append(len(y_client))

                    crypto_infos.append({
                        'crypto_type': crypto_info.get('crypto_type', 'none'),
                        'mechanism': crypto_info.get('mechanism', 'none'),
                        'comm_factor': crypto_info.get('comm_factor', 1.0),
                        'crypto_overhead_ms': crypto_info.get('crypto_overhead_ms', 0.0),
                        'noise_std': crypto_info.get(
                            'effective_noise_std',
                            ct_weight.noise_std if not hasattr(ct_weight, "effective_noise_std") else ct_weight.effective_noise_std()
                        ),
                        'he_noise_std': crypto_info.get('he_noise_std', ct_weight.noise_std),
                        'cumulative_noise': ct_weight.cumulative_noise,
                        'key_tag': ct_weight.key_tag,
                        'expanded': ct_weight.expanded,
                        'can_finalize': crypto_info.get('can_finalize', None),
                        'shares_available': crypto_info.get('shares_available', None),
                        'shares_required': crypto_info.get('shares_required', n_keyholders),
                        'ks_applied': crypto_info.get('ks_applied', False),
                        'finalization_failed': crypto_info.get('finalization_failed', None),
                        'client_id': client_id,
                        'is_malicious': is_malicious
                    })

                # Track enforcement statistics
                self.enforcement_stats[scenario_name]['failures'] += enforcement_failures
                self.enforcement_stats[scenario_name]['successes'] += enforcement_successes

                # Calculate total violations for this round
                protocol_violations['total_violations'] = sum([
                    protocol_violations['pofc_violations_malicious'],
                    protocol_violations['pofc_violations_benign'],
                    protocol_violations['signature_violations_malicious'],
                    protocol_violations['signature_violations_benign'],
                    protocol_violations['freshness_violations'],
                    protocol_violations['rate_limit_violations']
                ])

                # Update total banned counts
                protocol_violations['malicious_clients_banned_total'] = len([c for c in banned_clients if c in malicious_clients])
                protocol_violations['benign_clients_banned_total'] = len([c for c in banned_clients if c not in malicious_clients])

                # Track for protocol enforcement table
                round_stats = {
                    'effective_clients': len(client_cts),
                    'enforcement_successes': enforcement_successes,
                    'enforcement_failures': enforcement_failures,
                    'protocol_violations': protocol_violations,
                    'banned_clients_count': len(banned_clients),
                    'malicious_clients_total': len(malicious_clients),
                    'benign_clients_total': len(active_client_ids) - len(malicious_clients),
                    'remaining_malicious': len([c for c in active_client_ids if c in malicious_clients and c not in banned_clients]),
                    'remaining_benign': len([c for c in active_client_ids if c not in malicious_clients and c not in banned_clients])
                }
                self.track_protocol_enforcement(scenario_name, round_stats)

                # ------------------------------------------------------------
                # ------------------------------------------------------------
                # xMK finalization policy
                # ------------------------------------------------------------
                xmk_round_failed = False
                xmk_shares_available = 0
                xmk_shares_required = len(xmk_session_clients) if scenario_name in XMK_SCENARIOS else 0

                if scenario_name in XMK_SCENARIOS and xmk_round_aborted:
                    xmk_round_failed = True
                    print(
                        f"    ⚠ xMK round {round_id} ABORTED: "
                        f"participant rejected after session start. "
                        f"No model update will be applied."
                    )

                # Aggregation
                total_weight = sum(client_weights) if client_weights else 0
                if total_weight > 0 and client_cts:
                    try:
                        if crypto_base_scenario == "T4_CKKS_KS_TADP":
                            if round_id == 1:
                                print(f"    T4: CKKS double key switching + encrypted aggregation ({len(client_cts)} clients)")
                            agg_start = time.time()

                            # ---------- T4 DIAGNOSTIC: before encryption/aggregation ----------
                            # ---------- T4 DIAGNOSTIC: before encryption/aggregation ----------
                            expected_plain_update = np.sum(t4_plain_updates, axis=0)

                            if T4_DEBUG:
                                print(
                                    f"\n[T4 DEBUG R{round_id}] expected_plain_update: "
                                    f"norm={np.linalg.norm(expected_plain_update):.6e}, "
                                    f"min={np.min(expected_plain_update):.6e}, "
                                    f"max={np.max(expected_plain_update):.6e}, "
                                    f"mean={np.mean(expected_plain_update):.6e}"
                                )

                            # ---------- KS #1: client key -> aggregation key ----------
                            switched_to_agg = []
                            for ct in client_cts:
                                before_dec = t4_crypto.engine.decrypt(
                                    t4_crypto.client_keypairs[ct.client_id]["sk"],
                                    ct
                                )

                                ct_A = t4_crypto.key_switch_to_aggregation(ct)
                                switched_to_agg.append(ct_A)

                                after_ks1_dec = t4_crypto.engine.decrypt(
                                    t4_crypto.aggregation_keypair["sk"],
                                    ct_A
                                )

                                if T4_DEBUG:
                                    print(
                                        f"[T4 DEBUG R{round_id}] {ct.client_id} KS1: "
                                        f"before_norm={np.linalg.norm(before_dec):.6e}, "
                                        f"after_norm={np.linalg.norm(after_ks1_dec):.6e}, "
                                        f"diff_norm={np.linalg.norm(after_ks1_dec - before_dec):.6e}, "
                                        f"noise_budget={t4_crypto.noise_budget_bits(ct_A):.2f} bits"
                                    )

                            # ---------- Aggregation under aggregation key ----------
                            agg_ct_A = t4_crypto.aggregate_under_aggregation_key(switched_to_agg)

                            agg_dec_A = t4_crypto.engine.decrypt(
                                t4_crypto.aggregation_keypair["sk"],
                                agg_ct_A
                            )

                            if T4_DEBUG:
                                print(
                                    f"[T4 DEBUG R{round_id}] aggregation: "
                                    f"agg_dec_norm={np.linalg.norm(agg_dec_A):.6e}, "
                                    f"expected_norm={np.linalg.norm(expected_plain_update):.6e}, "
                                    f"agg_diff_norm={np.linalg.norm(agg_dec_A - expected_plain_update):.6e}, "
                                    f"noise_budget={t4_crypto.noise_budget_bits(agg_ct_A):.2f} bits"
                                )

                            # ---------- KS #2: aggregation key -> client key ----------
                            rekeyed_results = {}
                            for ct in client_cts:
                                cid = ct.client_id
                                rekeyed_results[cid] = t4_crypto.key_switch_to_client(
                                    agg_ct_A,
                                    target_client_id=cid
                                )

                            # ---------- Local decryption by every reachable recipient ----------
                            client_decrypted_results = {}

                            for cid, ct_res in rekeyed_results.items():
                                dec_i = t4_crypto.decrypt_client_result(cid, ct_res)
                                client_decrypted_results[cid] = dec_i

                                # Every client-specific ciphertext should decode to the same aggregate
                                diff_i = np.linalg.norm(dec_i - expected_plain_update)

                                if T4_DEBUG:
                                    print(
                                        f"[T4 DEBUG R{round_id}] KS2/decrypt recipient={cid}: "
                                        f"final_update_norm={np.linalg.norm(dec_i):.6e}, "
                                        f"expected_norm={np.linalg.norm(expected_plain_update):.6e}, "
                                        f"final_diff_norm={diff_i:.6e}, "
                                        f"min={np.min(dec_i):.6e}, "
                                        f"max={np.max(dec_i):.6e}, "
                                        f"mean={np.mean(dec_i):.6e}"
                                    )

                            # Use one representative decrypted aggregate to update the single simulated global model.
                            # This does NOT mean only one client can decrypt; it only avoids applying the same global update N times.
                            first_recipient = next(iter(client_decrypted_results))
                            avg_update = client_decrypted_results[first_recipient]

                            actual_srv_time = (time.time() - agg_start) * 1000

                            # Parameter-aware calibrated server-side CKKS double-KS timing.
                            estimated_srv_time = estimate_t4_server_round_ms(len(client_cts))

                            srv_time = max(actual_srv_time, estimated_srv_time)

                            t4_server_ks_count = len(client_cts) + len(rekeyed_results)
                            t4_server_ks_applied = True

                            for info in crypto_infos:
                                info["mechanism"] = (
                                    "T4: CKKS double KS "
                                    "(client key -> aggregation key -> client key)"
                                )
                                info["crypto_overhead_ms"] += srv_time / max(1, len(crypto_infos))
                                info["ks_applied"] = True
                                info["shares_required"] = 1
                                info["can_finalize"] = True

                        elif crypto_base_scenario == "T2_DF_KS_TADP":
                            # Real modified DF+KS aggregation path.
                            # Each client ciphertext already went through:
                            # Modified DF encryption -> multiplication expansion -> matrix-M key switching.
                            agg_ct = crypto.aggregate_df_ciphertexts(client_cts)

                            avg_update = crypto.decrypt_df_aggregate(agg_ct)

                        elif scenario_name in XMK_SCENARIOS:
                            if xmk_round_aborted:
                                raise ValueError(
                                    f"xMK finalization aborted: participant rejected after session start."
                                )

                            # Step 1: server aggregates encrypted xMK ciphertexts.
                            agg_ct = crypto.server_add(client_cts)

                            # Step 2: after aggregation, collect decryption shares from required clients.
                            crypto.collect_xmk_decryption_shares(
                                agg_ct=agg_ct,
                                round_id=round_id,
                                required_clients=xmk_session_clients
                            )

                            # Step 3: collaboratively decrypt the aggregate.
                            avg_update = crypto.xmk_collaborative_decrypt(
                                agg_ct=agg_ct,
                                round_id=round_id,
                                required_clients=xmk_session_clients
                            )

                            xmk_shares_available = len(crypto.xmk_shares[round_id])
                            xmk_shares_required = len(xmk_session_clients)
                            xmk_round_failed = False

                        else:
                            agg_ct = crypto.server_add(client_cts)
                            avg_update = agg_ct.value

                        global_model.weights += avg_update

                        if hasattr(global_model, "bias"):
                            global_model.bias += float(np.sum(client_bias_updates))

                    except ValueError as e:
                        if DEBUG_MODE:
                            print(f"    Aggregation error: {e}")

                        if scenario_name in XMK_SCENARIOS:
                            # No plaintext fallback for xMK.
                            # If xMK finalization fails or the round is aborted,
                            # the model must not be updated in this round.
                            avg_update = np.zeros(n_features)
                            xmk_round_failed = True
                            print(
                                f"    ⚠ xMK aggregation skipped for round {round_id}: {e}"
                            )

                        else:
                            avg_update = (
                                np.mean([ct.value for ct in client_cts], axis=0)
                                if client_cts else np.zeros(n_features)
                            )
                            global_model.weights += avg_update

                else:
                    if DEBUG_MODE:
                        print(f"    No aggregation performed in round {round_id}")
                y_pred_proba_test = global_model.predict_proba(self.X_global_test_scaled)

                # Use validation set for threshold tuning
                # Use validation set for threshold tuning
                threshold_metrics = {}

                if self.y_val is not None and len(self.y_val) > 0:
                    y_val_proba = global_model.predict_proba(self.X_val_scaled)

                    threshold, threshold_metrics = optimize_threshold_for_constraint(
                        self.y_val,
                        y_val_proba,
                        constraint_type='fpr',
                        constraint_value=FPR_CONSTRAINT_VALUE
                    )

                    # Store threshold for logging/debugging
                    threshold_metrics['debug_optimal_threshold'] = threshold

                else:
                    threshold = 0.5
                    threshold_metrics['debug_optimal_threshold'] = threshold

                y_pred = (y_pred_proba_test >= threshold).astype(int)

                # Compute enhanced metrics - NO ATTACK METRICS
                metrics = compute_enhanced_metrics(self.y_global_test, y_pred, y_pred_proba_test)
                metrics.update(threshold_metrics)

                baseline_accuracy = np.mean(self.y_global_test == 0)

                # Communication calculation
                model_size_bytes = (n_features * 8) + 8
                avg_comm_factor = np.mean([info['comm_factor'] for info in crypto_infos]) if crypto_infos else 1.0

                enforcement_applied = scenario_name not in [
                    "B0_Plain_FL",
                    "B1_xMK_CKKS",
                    "A1_TADP_admission_only"
                ]

                if enforcement_applied:
                    baseline_overhead = 256 + 16 + 8 + 16 + 200
                else:
                    baseline_overhead = 0

                if scenario_name == "T4_CKKS_KS_TADP" and client_cts:
                    ckks_ct_bytes = client_cts[0].get_size_bytes()

                    comm_up = int(enforcement_successes * (ckks_ct_bytes + baseline_overhead))
                    comm_down = int(enforcement_successes * (ckks_ct_bytes + baseline_overhead))

                    comm_per_client = ckks_ct_bytes + baseline_overhead
                    avg_comm_factor = ckks_ct_bytes / max(1, model_size_bytes)

                else:
                    base_comm = model_size_bytes * avg_comm_factor
                    comm_per_client = base_comm + baseline_overhead
                    comm_up = int(enforcement_successes * comm_per_client)
                    comm_down = int(comm_per_client)

                avg_crypto_overhead = np.mean([info['crypto_overhead_ms'] for info in crypto_infos]) if crypto_infos else 0.0
                avg_noise_std = np.mean([info['noise_std'] for info in crypto_infos]) if crypto_infos else 0.0
                avg_cumulative_noise = np.mean([info['cumulative_noise'] for info in crypto_infos]) if crypto_infos else 0.0

                # Energy computation
                energy_metrics = compute_energy_consumption(
                    comm_bytes=comm_up,
                    crypto_ms=avg_crypto_overhead
                )

                # Store result - NO ATTACK METRICS
                result = {
                    'seed': seed,
                    'scenario': scenario_name,
                    'round': round_id,
                    **metrics,
                    'baseline_accuracy': baseline_accuracy,
                    'optimal_threshold': threshold,
                    'runtime_s': time.time() - round_start,
                    'comm_up_bytes': comm_up,
                    'comm_down_bytes': comm_down,
                    'crypto_overhead_ms': avg_crypto_overhead,
                    'noise_std': avg_noise_std,
                    'cumulative_noise': avg_cumulative_noise,
                    'n_clients': len(active_client_ids),
                    'effective_clients': len(client_cts),
                    'enforcement_failures': enforcement_failures,
                    'mechanism': crypto_infos[0]['mechanism'] if crypto_infos else 'none',
                    'comm_factor': avg_comm_factor,
                    'crypto_type': crypto_infos[0]['crypto_type'] if crypto_infos else 'none',
                    'xmk_shares_available': xmk_shares_available,
                    'xmk_shares_required': xmk_shares_required,
                    'xmk_round_failed': xmk_round_failed,
                    't4_server_ks_applied': t4_server_ks_applied,
                    't4_server_ks_count': t4_server_ks_count,
                    'fraud_rate_global_test': np.mean(self.y_global_test),
                    'fraud_rate_active_clients': total_fraud / total_samples if total_samples > 0 else 0,
                    'pscore_admitted_mean': np.mean([self.admitted_clients[cid]['pscore']
                                                    for cid in active_client_ids
                                                    if cid in self.admitted_clients])
                                            if active_client_ids and hasattr(self, 'admitted_clients') else 0,
                    # Protocol violation tracking - ENHANCED
                    'pofc_violations_malicious': protocol_violations['pofc_violations_malicious'],
                    'pofc_violations_benign': protocol_violations['pofc_violations_benign'],
                    'signature_violations_malicious': protocol_violations['signature_violations_malicious'],
                    'signature_violations_benign': protocol_violations['signature_violations_benign'],
                    'freshness_violations': protocol_violations['freshness_violations'],
                    'rate_limit_violations': protocol_violations['rate_limit_violations'],
                    'total_violations': protocol_violations['total_violations'],
                    'malicious_clients_blocked_this_round': protocol_violations['malicious_clients_blocked_this_round'],
                    'benign_clients_blocked_this_round': protocol_violations['benign_clients_blocked_this_round'],
                    'malicious_clients_banned_total': protocol_violations['malicious_clients_banned_total'],
                    'benign_clients_banned_total': protocol_violations['benign_clients_banned_total'],
                    'banned_clients_count': len(banned_clients),
                    'remaining_malicious_clients': len([c for c in active_client_ids if c in malicious_clients and c not in banned_clients]),
                    'remaining_benign_clients': len([c for c in active_client_ids if c not in malicious_clients and c not in banned_clients]),
                    'initial_malicious_count': initial_malicious_count,
                    'initial_benign_count': initial_benign_count,
                    # Attack detection metrics
                    'attack_detected_this_round': 1 if protocol_violations['malicious_clients_blocked_this_round'] > 0 else 0,
                    'all_malicious_blocked': 1 if protocol_violations['malicious_clients_banned_total'] >= initial_malicious_count else 0,
                    # Energy metrics
                    **energy_metrics
                }
                all_results.append(result)

                self.performance_tracker.add_metrics(
                    scenario=scenario_name,
                    seed=seed,
                    round_id=round_id,
                    metrics_dict=result
                )

                # Print progress - NO ATTACK METRICS
                if round_id % 10 == 0 or round_id == rounds:
                    enforcement_rate = (enforcement_successes / (enforcement_successes + enforcement_failures)) \
                        if (enforcement_successes + enforcement_failures) > 0 else 1.0

                    extra = ""
                    if scenario_name in XMK_SCENARIOS:
                        extra = f", xMK shares={xmk_shares_available}/{xmk_shares_required}, xMK_failed={xmk_round_failed}"

                    # Add protocol violation info (only show if > 0)
                    violation_count = protocol_violations['total_violations']
                    if violation_count > 0:
                        extra += f", Violations={violation_count}"
                        if malicious_clients:
                            blocked_this_round = protocol_violations['malicious_clients_blocked_this_round']
                            blocked_total = protocol_violations['malicious_clients_banned_total']
                            extra += f", Malicious blocked={blocked_this_round} this round, {blocked_total}/{initial_malicious_count} total"

                    # Add banned clients info
                    if banned_clients:
                        extra += f", Banned={len(banned_clients)}"
                        extra += f", Remaining clients={len(active_client_ids) - len(banned_clients)}"

                    print(
                        f"    Round {round_id:3d}/{rounds}: "
                        f"PR-AUC={metrics['pr_auc']:.4f}, "
                        f"MCC={metrics['mcc']:.4f}, "
                        f"Clients={enforcement_successes}/{len(active_client_ids)-len(banned_clients)}, "
                        f"Comm={comm_up/1024:.2f}KB, "
                        f"Energy={energy_metrics['energy_total_uj']:.2f}μJ"
                        f"{extra}"
                    )

        return pd.DataFrame(all_results)

    def run_all_scenarios(self):
        # UPDATED SCENARIOS LIST - NO DATA POISONING ATTACKS:
        scenarios = [
            "B0_Plain_FL",              # Baseline: No security
            "B1_xMK_CKKS",              # Baseline: Only crypto
            "A1_TADP_admission_only",   # Governance only
            "T1_Plain_TADP",            # T1: Governance + Baseline controls
            "T1_TEST_SIGNATURE_ATTACK", # T1 + Protocol attack (bad signature)
            "T2_DF_KS_TADP",            # T2: T1_Plain_TADP + DF encryptio
            "T2_TEST_REPLAY_ATTACK",    # T2 + Protocol attack (replay)
            "T3_XMK_CKKS_TADP",         # T3: baseline controls + xMK-CKKS collaborative aggregation
            "T3_TEST_POFC_ATTACK",      # T3 + Protocol attack (PoFC flooding burst or DoS attack)
            "T4_CKKS_KS_TADP"           # T4: baseline controls + CKKS/RLWE double key switching
        ]


        print(f"\nRunning {len(scenarios)} scenarios")
        print("Note: Protocol enforcement testing only - no data poisoning attacks")

        all_results = []
        scenario_summaries_list = []  # Store all scenario summaries for final table

        # Track total execution time
        total_start_time = time.time()

        for scenario in scenarios:
            print(f"\n{'='*80}")
            print(f"RUNNING: {scenario}")
            print(f"{'='*80}")

            scenario_start_time = time.time()
            # scenario_results = self.run_scenario(scenario)
            if scenario == "T4_CKKS_KS_TADP":
                scenario_seeds = T4_SEEDS
                scenario_rounds = T4_ROUNDS
            else:
                scenario_seeds = FULL_SEEDS
                scenario_rounds = ROUNDS

            scenario_results = self.run_scenario(
                scenario,
                seeds=scenario_seeds,
                rounds=scenario_rounds
            )
            scenario_time = time.time() - scenario_start_time

            if not scenario_results.empty:
                scenario_results['category'] = 'BASELINE' if scenario.startswith('B') else 'GOVERNANCE' if scenario.startswith('A') else 'CRYPTO_TIER'
                all_results.append(scenario_results)

                # Save individual scenario results
                scenario_path = os.path.join(STATS_DIR, f"{scenario}_results.csv")
                scenario_results.to_csv(scenario_path, index=False)

                # ============================================================
                # CORRECTED METRICS CALCULATION - FIXED CUMULATIVE ERRORS
                # ============================================================

                # 1. Get the LAST round data for each seed for FINAL metrics
                last_round_data = scenario_results.sort_values(['seed', 'round']).groupby('seed').tail(1)

                # 2. Calculate metrics PER SEED first, then average across seeds
                seed_metrics = []

                for seed in sorted(scenario_results['seed'].unique()):
                    seed_data = scenario_results[scenario_results['seed'] == seed]

                    if seed_data.empty:
                        continue

                    # Get last round for this seed
                    seed_last_round = seed_data.sort_values('round').tail(1)

                    # Calculate cumulative totals for this seed
                    seed_total_comm_bytes = float(seed_data['comm_up_bytes'].sum()) if 'comm_up_bytes' in seed_data.columns else 0.0

                    # Calculate cumulative energy for this seed
                    if 'energy_total_uj' in seed_data.columns:
                        seed_total_energy_uj = float(seed_data['energy_total_uj'].sum())
                    else:
                        # Fallback: estimate from crypto overhead
                        seed_total_energy_uj = float(seed_data['crypto_overhead_ms'].sum() * 1000)

                    # Get final round metrics
                    if not seed_last_round.empty:
                        seed_final_pr_auc = float(seed_last_round['pr_auc'].iloc[0]) if 'pr_auc' in seed_last_round.columns else 0.0
                        seed_final_f1 = float(seed_last_round['f1'].iloc[0]) if 'f1' in seed_last_round.columns else 0.0
                    else:
                        seed_final_pr_auc = 0.0
                        seed_final_f1 = 0.0

                    # Calculate average noise for this seed
                    seed_avg_noise = float(seed_data['noise_std'].mean()) if 'noise_std' in seed_data.columns else 0.0

                    # Get number of rounds for this seed
                    seed_n_rounds = int(seed_data['round'].nunique())

                    # Calculate average communication per round for this seed
                    if seed_n_rounds > 0:
                        seed_avg_comm_per_round = float(seed_total_comm_bytes / seed_n_rounds)
                    else:
                        seed_avg_comm_per_round = 0.0

                    seed_metrics.append({
                        'seed': int(seed),
                        'final_pr_auc': seed_final_pr_auc,
                        'final_f1': seed_final_f1,
                        'avg_noise': seed_avg_noise,
                        'total_comm_bytes': seed_total_comm_bytes,
                        'total_energy_uj': seed_total_energy_uj,
                        'avg_comm_per_round_bytes': seed_avg_comm_per_round,
                        'n_rounds': seed_n_rounds
                    })

                # Convert to DataFrame for easy calculation
                if seed_metrics:
                    seed_metrics_df = pd.DataFrame(seed_metrics)

                    # Helper: avoid NaN standard deviation when a scenario has only one seed
                    def safe_std(series):
                        series = pd.to_numeric(series, errors="coerce").dropna()
                        if len(series) < 2:
                            return 0.0
                        return float(series.std())

                    # 3. Calculate FINAL metrics (average of last round values across seeds)
                    final_pr_auc = float(seed_metrics_df['final_pr_auc'].mean())
                    final_f1 = float(seed_metrics_df['final_f1'].mean())

                    # 4. Calculate average HE / crypto numerical noise across all rounds and seeds
                    # Note: plaintext and governance-only scenarios should naturally report 0.00e+00
                    # because no approximate encrypted aggregation noise is introduced.
                    avg_noise = float(seed_metrics_df['avg_noise'].mean())

                    # 5. Calculate TOTAL COMMUNICATION (cumulative across all rounds, summed across seeds)
                    total_comm_bytes_all_seeds = float(seed_metrics_df['total_comm_bytes'].sum())
                    total_comm_kb_all_seeds = total_comm_bytes_all_seeds / 1024.0
                    total_comm_mb_all_seeds = total_comm_bytes_all_seeds / (1024.0 * 1024.0)

                    # Also calculate average communication per seed
                    avg_comm_bytes_per_seed = float(seed_metrics_df['total_comm_bytes'].mean())
                    avg_comm_kb_per_seed = avg_comm_bytes_per_seed / 1024.0

                    # Calculate average communication per round (across seeds)
                    avg_comm_per_round_bytes = float(seed_metrics_df['avg_comm_per_round_bytes'].mean())
                    avg_comm_per_round_kb = avg_comm_per_round_bytes / 1024.0

                    # 6. Calculate TOTAL ENERGY (cumulative across all rounds, summed across seeds)
                    # Energy values are stored internally in microjoules (μJ).
                    total_energy_all_seeds = float(seed_metrics_df['total_energy_uj'].sum())

                    # Unit conversions:
                    # 1 mJ = 1,000 μJ
                    # 1 J  = 1,000,000 μJ
                    total_energy_mj_all_seeds = total_energy_all_seeds / 1_000.0
                    total_energy_j_all_seeds = total_energy_all_seeds / 1_000_000.0

                    # Also calculate average energy per seed
                    avg_energy_per_seed = float(seed_metrics_df['total_energy_uj'].mean())

                    # Unit conversions for average energy per seed
                    avg_energy_mj_per_seed = avg_energy_per_seed / 1_000.0
                    avg_energy_j_per_seed = avg_energy_per_seed / 1_000_000.0

                    # Calculate average energy per round (across seeds)
                    avg_energy_per_round_uj = float(seed_metrics_df['avg_energy_per_round_uj'].mean()) \
                        if 'avg_energy_per_round_uj' in seed_metrics_df.columns else np.nan

                    if not np.isnan(avg_energy_per_round_uj):
                        avg_energy_per_round_mj = avg_energy_per_round_uj / 1_000.0
                        avg_energy_per_round_j = avg_energy_per_round_uj / 1_000_000.0

                    # 7. Additional statistics
                    n_seeds = int(len(seed_metrics_df))
                    avg_rounds_per_seed = float(seed_metrics_df['n_rounds'].mean())
                    total_rounds_all_seeds = int(seed_metrics_df['n_rounds'].sum())

                    # Calculate standard deviations for confidence intervals
                    pr_auc_std = safe_std(seed_metrics_df['final_pr_auc'])
                    f1_std = safe_std(seed_metrics_df['final_f1'])
                    noise_std = safe_std(seed_metrics_df['avg_noise'])

                    # 8. Print comprehensive summary
                    print(f"  ✓ Results saved with {n_seeds} seeds ({total_rounds_all_seeds} total rounds):")
                    print(f"     • Final PR-AUC: {final_pr_auc:.4f} (±{pr_auc_std:.4f})")
                    print(f"     • Final F1: {final_f1:.4f} (±{f1_std:.4f})")
                    print(f"     • Avg HE / Crypto Numerical Noise: {avg_noise:.2e} (±{noise_std:.2e})")

                    print(f"     • Communication:")
                    print(
                        f"       - Total (all seeds): "
                        f"{total_comm_kb_all_seeds:.1f} KB ({total_comm_mb_all_seeds:.3f} MB)"
                    )
                    print(f"       - Avg per seed: {avg_comm_kb_per_seed:.1f} KB")
                    print(f"       - Avg per round: {avg_comm_per_round_kb:.2f} KB")

                    print(f"     • Energy:")
                    print(
                        f"       - Total (all seeds): {total_energy_all_seeds:,.0f} μJ "
                        f"({total_energy_mj_all_seeds:.3f} mJ, {total_energy_j_all_seeds:.6f} J)"
                    )
                    print(
                        f"       - Avg per seed: {avg_energy_per_seed:,.0f} μJ "
                        f"({avg_energy_mj_per_seed:.3f} mJ, {avg_energy_j_per_seed:.6f} J)"
                    )

                    if not np.isnan(avg_energy_per_round_uj):
                        print(
                            f"       - Avg per round: {avg_energy_per_round_uj:,.2f} μJ "
                            f"({avg_energy_per_round_mj:.3f} mJ, {avg_energy_per_round_j:.6f} J)"
                        )

                    print(f"     • Rounds: {avg_rounds_per_seed:.1f} avg per seed, {total_rounds_all_seeds} total")
                    print(f"     • Execution Time: {scenario_time:.1f}s")

                    # Also save the seed-level metrics for detailed analysis
                    seed_metrics_path = os.path.join(STATS_DIR, f"{scenario}_seed_metrics.csv")
                    seed_metrics_df.to_csv(seed_metrics_path, index=False)

                    # Store summary for performance table
                    scenario_summary = {
                        'scenario': scenario,
                        'n_seeds': n_seeds,
                        'total_rounds': total_rounds_all_seeds,
                        'final_pr_auc': final_pr_auc,
                        'final_pr_auc_std': pr_auc_std,
                        'final_f1': final_f1,
                        'final_f1_std': f1_std,
                        'avg_noise': avg_noise,
                        'avg_noise_std': noise_std,
                        'total_comm_kb': total_comm_kb_all_seeds,
                        'avg_comm_per_seed_kb': avg_comm_kb_per_seed,
                        'avg_comm_per_round_kb': avg_comm_per_round_kb,
                        'total_energy_uj': total_energy_all_seeds,
                        'avg_energy_per_seed_uj': avg_energy_per_seed,
                        'avg_rounds': avg_rounds_per_seed,
                        'execution_time_s': float(scenario_time)
                    }

                    # Add to list for final comparison table
                    scenario_summaries_list.append(scenario_summary)

                    # Save scenario summary
                    summary_path = os.path.join(STATS_DIR, f"{scenario}_summary.json")
                    with open(summary_path, 'w') as f:
                        json.dump(scenario_summary, f, indent=2, default=str)  # Use default=str to handle any non-serializable types

                else:
                    print(f"  ⚠ No seed metrics calculated for scenario {scenario}")

        # Combine all results
        if all_results:
            total_time = time.time() - total_start_time
            hours = int(total_time // 3600)
            minutes = int((total_time % 3600) // 60)
            seconds = total_time % 60

            combined_df = pd.concat(all_results, ignore_index=True)
            combined_path = os.path.join(OUT_DIR, "all_scenarios_results_comprehensive.csv")
            combined_df.to_csv(combined_path, index=False)
            print(f"\n✓ All results saved to: {combined_path}")

            # ============================================================
            # GENERATE FINAL COMPARISON TABLE
            # ============================================================
            if scenario_summaries_list:
                print(f"\n{'='*80}")
                print("GENERATING COMPREHENSIVE COMPARISON TABLE")
                print(f"{'='*80}")

                # Create comparison DataFrame
                comparison_data = []

                for summary in scenario_summaries_list:
                    # Determine scenario category
                    if summary['scenario'].startswith('B'):
                        category = 'Baseline'
                    elif summary['scenario'].startswith('A'):
                        category = 'Governance'
                    else:
                        category = 'Crypto Tier'

                    # Format metrics with standard deviations
                    pr_auc_str = f"{summary['final_pr_auc']:.4f} (±{summary['final_pr_auc_std']:.4f})"
                    f1_str = f"{summary['final_f1']:.4f} (±{summary['final_f1_std']:.4f})"
                    noise_str = f"{summary['avg_noise']:.2e} (±{summary['avg_noise_std']:.2e})"

                    # Use direct measurable indicators only.
                    # Composite scores are avoided because they depend on arbitrary weighting.
                    comparison_data.append({
                        'Scenario': summary['scenario'],
                        'Category': category,
                        'PR-AUC (mean±std)': pr_auc_str,
                        'F1-Score (mean±std)': f1_str,
                        'Cryptographic Noise (mean±std)': noise_str,
                        'Avg Comm/Round (KB)': f"{summary['avg_comm_per_round_kb']:.2f}",
                        'Total Comm (MB)': f"{summary['total_comm_kb'] / 1024.0:.3f}",
                        'Avg Energy/Seed (mJ)': f"{summary['avg_energy_per_seed_uj'] / 1000.0:.3f}",
                        'Avg Energy/Seed (J)': f"{summary['avg_energy_per_seed_uj'] / 1_000_000.0:.6f}",
                        'Total Energy (mJ)': f"{summary['total_energy_uj'] / 1000.0:.3f}",
                        'Total Energy (J)': f"{summary['total_energy_uj'] / 1_000_000.0:.6f}",
                        'Seeds': summary['n_seeds'],
                        'Total Rounds': summary['total_rounds'],
                        'Execution Time (s)': f"{summary['execution_time_s']:.1f}"
                    })

                comparison_df = pd.DataFrame(comparison_data)

                # Sort by category and scenario name.
                # Avoid sorting by synthetic scores because they are not used.
                comparison_df = comparison_df.sort_values(
                    ['Category', 'Scenario'],
                    ascending=[True, True]
                )

                # Save comparison table
                comparison_path = os.path.join(STATS_DIR, "scenario_comparison_table.csv")
                comparison_df.to_csv(comparison_path, index=False)

                # Print the comparison table
                print(f"\n{'='*120}")
                print("SCENARIO COMPARISON TABLE")
                print(f"{'='*120}")

                # Create a formatted table display with direct, defensible metrics only
                display_cols = [
                    'Scenario',
                    'Category',
                    'PR-AUC (mean±std)',
                    'F1-Score (mean±std)',
                    'Cryptographic Noise (mean±std)',
                    'Avg Comm/Round (KB)',
                    'Avg Energy/Seed (mJ)',
                    'Execution Time (s)'
                ]

                display_df = comparison_df[display_cols].copy()

                # Format for display
                pd.set_option('display.width', 160)
                pd.set_option('display.max_columns', None)
                pd.set_option('display.max_colwidth', 25)

                print("\n" + display_df.to_string(index=False))

                # Save a simpler version for quick reference
                simple_comparison = comparison_df[
                    [
                        'Scenario',
                        'Category',
                        'PR-AUC (mean±std)',
                        'F1-Score (mean±std)',
                        'Avg Comm/Round (KB)',
                        'Avg Energy/Seed (mJ)',
                        'Execution Time (s)'
                    ]
                ].copy()

                simple_path = os.path.join(STATS_DIR, "scenario_quick_comparison.csv")
                simple_comparison.to_csv(simple_path, index=False)

                print(f"\n{'='*120}")
                print(f"✓ Comparison tables saved:")
                print(f"  • Detailed table: {comparison_path}")
                print(f"  • Quick reference: {simple_path}")

                # Generate LaTeX table for publication
                latex_cols = [
                    'Scenario',
                    'Category',
                    'PR-AUC (mean±std)',
                    'F1-Score (mean±std)',
                    'Cryptographic Noise (mean±std)',
                    'Avg Comm/Round (KB)',
                    'Avg Energy/Seed (mJ)'
                ]

                latex_df = comparison_df[latex_cols].copy()
                latex_path = os.path.join(STATS_DIR, "scenario_comparison_table.tex")

                with open(latex_path, 'w') as f:
                    f.write(latex_df.to_latex(index=False, float_format="%.3f"))

                print(f"  • LaTeX table: {latex_path}")

                # Save all scenario summaries
                all_summaries_df = pd.DataFrame(scenario_summaries_list)
                all_summaries_path = os.path.join(STATS_DIR, "all_scenarios_summary.csv")
                all_summaries_df.to_csv(all_summaries_path, index=False)
                print(f"  • All summaries: {all_summaries_path}")

            # Add execution time metadata
            execution_metadata = {
                'total_execution_time_seconds': float(total_time),
                'total_execution_time_formatted': f"{hours:02d}:{minutes:02d}:{seconds:05.2f}",
                'scenarios_executed': len(scenarios),
                'completion_timestamp': datetime.now().isoformat(),
                'average_time_per_scenario': float(total_time / len(scenarios)) if scenarios else 0.0,
                'scenarios': scenarios
            }

            with open(os.path.join(STATS_DIR, 'execution_metadata.json'), 'w') as f:
                json.dump(execution_metadata, f, indent=2, default=str)

            with open(os.path.join(STATS_DIR, 'admission_statistics.json'), 'w') as f:
                json.dump(self.admission_stats, f, indent=2, default=str)

            with open(os.path.join(STATS_DIR, 'enforcement_statistics.json'), 'w') as f:
                json.dump(dict(self.enforcement_stats), f, indent=2, default=str)

            # Also generate the performance tracker table
            performance_table = self.performance_tracker.generate_comprehensive_table()
            if not performance_table.empty:
                performance_table_path = os.path.join(STATS_DIR, 'performance_comparison_table.csv')
                performance_table.to_csv(performance_table_path, index=False)
                print(f"\n✓ Performance tracker table saved: {performance_table_path}")

                # Print performance tracker table
                print(f"\n{'='*120}")
                print("PERFORMANCE TRACKER SUMMARY TABLE")
                print(f"{'='*120}")
                print("\n" + performance_table.to_string(index=False))

            # Print final execution summary
            print(f"\n{'='*80}")
            print("EXPERIMENT COMPLETE")
            print(f"{'='*80}")
            print(f"\nTotal execution time: {hours:02d}:{minutes:02d}:{seconds:05.2f}")
            print(f"Total seconds: {total_time:.2f}s")
            print(f"Average per scenario: {total_time/len(scenarios):.2f}s")
            print(f"\nResults saved in: {OUT_DIR}")
            print(f"• Comprehensive results: {combined_path}")
            print(f"• Scenario comparisons: {STATS_DIR}/scenario_comparison_table.csv")
            print(f"• Individual scenario results: {STATS_DIR}/")
            print(f"• Execution metadata: {STATS_DIR}/execution_metadata.json")

            return combined_df

        return pd.DataFrame()

# --------------------------
# PROFESSIONAL VISUALIZATION GENERATORS
# --------------------------
def generate_main_comparison_table(results_df: pd.DataFrame, out_dir: str):
    """Generate main comparison table for publication"""

    print("\n  Generating MAIN COMPARISON TABLE for publication...")

    last_rows = (results_df.sort_values(['scenario', 'seed', 'round'])
                 .groupby(['scenario', 'seed']).tail(1))

    if last_rows.empty:
        return

    # Calculate comprehensive metrics
    table_data = []

    for scenario in sorted(results_df['scenario'].unique()):
        scenario_data = last_rows[last_rows['scenario'] == scenario]
        all_scenario_data = results_df[results_df['scenario'] == scenario]

        if scenario_data.empty:
            continue

        # Performance metrics
        pr_auc = scenario_data['pr_auc'].mean()
        pr_auc_std = scenario_data['pr_auc'].std()
        recall_at_005 = scenario_data.get('recall_at_fpr_0.005', pd.Series([0])).mean()
        recall_at_01 = scenario_data.get('recall_at_fpr_0.01', pd.Series([0])).mean()
        mcc = scenario_data['mcc'].mean()
        f1 = scenario_data['f1'].mean()

        # Cost metrics
        comm_kb = all_scenario_data['comm_up_bytes'].mean() / 1024
        crypto_ms = all_scenario_data['crypto_overhead_ms'].mean()
        noise = all_scenario_data['noise_std'].mean()

        # Security metrics
        effective_clients = scenario_data['effective_clients'].mean()  # scenario_data is last round per seed
        enforcement_rate = effective_clients / N_CLIENTS

        # Attack metrics (if available)
        backdoor_asr = scenario_data.get('backdoor_success_rate', pd.Series([0])).mean()
        asr_reduction = scenario_data.get('asr_delta', pd.Series([0])).mean()

        # Determine category
        if scenario.startswith('B'):
            category = 'Baseline'
        elif scenario.startswith('A'):
            category = 'Governance'
        else:
            category = 'Crypto Tier'

        # Use direct measurable indicators only.
        table_data.append({
            'Scenario': scenario,
            'Category': category,
            'PR-AUC (mean±std)': f"{pr_auc:.4f}±{pr_auc_std:.4f}",
            'Recall@FPR=0.5%': f"{recall_at_005:.4f}",
            'Recall@FPR=1.0%': f"{recall_at_01:.4f}",
            'MCC': f"{mcc:.4f}",
            'F1-Score': f"{f1:.4f}",
            'Comm (KB/round)': f"{comm_kb:.1f}",
            'Crypto OH (ms/round)': f"{crypto_ms:.1f}",
            'Cryptographic Noise': f"{noise:.4f}",
            'Effective Clients (%)': f"{enforcement_rate * 100:.1f}%",
            'Backdoor ASR': f"{backdoor_asr:.3f}",
            'ASR Reduction': f"{asr_reduction:+.3f}"
        })

    main_table = pd.DataFrame(table_data)
    main_table = main_table.sort_values(['Category', 'PR-AUC (mean±std)'], ascending=[True, False])

    main_table_path = os.path.join(out_dir, 'table_main_comparison.csv')
    main_table.to_csv(main_table_path, index=False)

    # Create LaTeX version
    latex_cols = [
        'Scenario',
        'PR-AUC (mean±std)',
        'Recall@FPR=0.5%',
        'Recall@FPR=1.0%',
        'MCC',
        'F1-Score',
        'Comm (KB/round)',
        'Crypto OH (ms/round)',
        'Cryptographic Noise',
        'Effective Clients (%)'
    ]
    latex_table = main_table[latex_cols]

    latex_path = os.path.join(out_dir, 'table_main_comparison.tex')
    with open(latex_path, 'w') as f:
        f.write(latex_table.to_latex(index=False, float_format="%.4f"))

    print(f"✓ Main comparison table saved (CSV and LaTeX)")
    print(f"  {len(main_table)} scenarios compared")

    return main_table

def generate_threat_coverage_matrix(out_dir: str):
    """Generate protocol-level threat coverage matrix for TADP-Sec."""

    print("\n  Generating PROTOCOL THREAT COVERAGE MATRIX...")

    threats = [
        "Invalid Signature / Identity Spoofing",
        "Replay or Stale Update",
        "PoFC / Rate-Limit Overflow",
        "Model-Update Confidentiality",
        "Training Metadata Integrity",
        "Key Isolation / Key Misuse",
        "Auditability and Traceability"
    ]

    coverage_matrix = pd.DataFrame({
        "Threat": threats,
        "T1 (Governance + Baseline Controls)": [
            "High", "High", "Medium", "None", "Medium", "Low", "High"
        ],
        "T2 (DF + KS)": [
            "High", "High", "Medium", "Low", "Medium", "Medium", "High"
        ],
        "T3 (xMK-CKKS)": [
            "High", "High", "High", "High", "High", "High", "High"
        ],
        "T4 (CKKS + Double Key Switching)": [
            "High", "High", "High", "Very High", "Very High", "Very High", "High"
        ],
        "Residual Risk": [
            "Low", "Low", "Low", "Low", "Low", "Medium", "Low"
        ]
    })

    matrix_path = os.path.join(out_dir, "table_protocol_threat_coverage_matrix.csv")
    coverage_matrix.to_csv(matrix_path, index=False)

    scoring_rubric = pd.DataFrame({
        "Coverage Level": ["None", "Low", "Medium", "High", "Very High"],
        "Description": [
            "No direct protection",
            "Basic protection with notable residual risk",
            "Moderate protection with manageable residual risk",
            "Strong protection with low residual risk",
            "Strongest protection in this framework"
        ],
        "Score (0-10)": [0, 3, 6, 8, 9]
    })

    rubric_path = os.path.join(out_dir, "table_protocol_security_scoring_rubric.csv")
    scoring_rubric.to_csv(rubric_path, index=False)

    print("✓ Protocol threat coverage matrix saved")
    print(f"  • Matrix: {matrix_path}")
    print(f"  • Rubric: {rubric_path}")

    return coverage_matrix

def generate_learning_curves_plot(results_df: pd.DataFrame, experiment, out_dir: str):
    """Generate publication-ready learning curves with all fixes."""

    import os
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    import pandas as pd
    from matplotlib.ticker import MaxNLocator
    # ------------------------------------------------------------
    # FIX (Plot D empty): normalize scenario strings once (strip)
    # ------------------------------------------------------------
    results_df = results_df.copy()
    results_df["scenario"] = results_df["scenario"].astype(str).str.strip()

    # ---- Publication typography (journal-friendly) ----
    mpl.rcParams.update({
        "font.family": "serif",
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "legend.fontsize": 9,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.8,
    })

    print("\n  Generating ENHANCED LEARNING CURVES plot for publication...")

    # -------------------------------
    # Split main vs attack scenarios
    # -------------------------------
    is_test = results_df["scenario"].astype(str).str.contains(r"_TEST_", regex=True)

    main_df = results_df.loc[~is_test].copy()
    test_df = results_df.loc[is_test].copy()
    # ---- Robust filtering of test scenarios ----
    # Your test scenarios contain "_TEST_" in the name
    is_test = results_df["scenario"].astype(str).str.contains(r"_TEST_", regex=True)
    main_df = results_df.loc[~is_test].copy()

    # Keep test scenarios separately for Panel D
    test_df = results_df.loc[is_test].copy()

    # ---- Consistent scenario order (prevents legend chaos) ----
    ordered_main = [
        "B0_Plain_FL",
        "B1_xMK_CKKS",
        "A1_TADP_admission_only",
        "T1_Plain_TADP",
        "T2_DF_KS_TADP",
        "T3_XMK_CKKS_TADP",
        "T4_CKKS_KS_TADP",
    ]
    main_scenarios = [s for s in ordered_main if s in set(main_df["scenario"].unique())]

    # Line styles for convergence plot
    line_styles = {
        "B0_Plain_FL": "--",
        "B1_xMK_CKKS": "--",
        "A1_TADP_admission_only": "-.",
        "T1_Plain_TADP": "-",
        "T2_DF_KS_TADP": "-",
        "T3_XMK_CKKS_TADP": "-",
        "T4_CKKS_KS_TADP": "-",
    }

    # Colors for convergence plot (colorblind-friendly)
    colors = {
        "B0_Plain_FL": "#377eb8",  # Blue
        "B1_xMK_CKKS": "#ff7f00",  # Orange
        "A1_TADP_admission_only": "#4daf4a",  # Green
        "T1_Plain_TADP": "#e41a1c",  # Red
        "T2_DF_KS_TADP": "#984ea3",  # Purple
        "T3_XMK_CKKS_TADP": "#a65628",  # Brown
        "T4_CKKS_KS_TADP": "#f781bf",  # Pink
    }

    # Create figure with subplots
    fig = plt.figure(figsize=(18, 13), dpi=600)

    # ============================================
    # ============================================
    # PLOT A: PR-AUC vs Rounds (mean ± 95% CI)
    # ============================================
    ax1 = plt.subplot(2, 2, 1)

    # Define which scenarios should show crypto degradation
    crypto_degradation_scenarios = ["T3_XMK_CKKS_TADP", "T4_CKKS_KS_TADP"]

    for scenario in main_scenarios:
        scenario_data = main_df[main_df["scenario"] == scenario]
        if scenario_data.empty:
            continue

        # Calculate statistics per round
        round_stats = (scenario_data.groupby("round")["pr_auc"]
                      .agg(["mean", "std", "count"])
                      .reset_index())

        # 95% CI (t-distribution for small samples)
        round_stats["ci"] = 1.96 * round_stats["std"] / np.sqrt(round_stats["count"].clip(lower=1))

        # Plot mean with confidence interval
        (line,) = ax1.plot(
            round_stats["round"], round_stats["mean"],
            linewidth=2.2,
            linestyle=line_styles.get(scenario, "-"),
            color=colors.get(scenario, "black"),
            label=scenario
        )

        # Fill confidence interval
        ax1.fill_between(
            round_stats["round"],
            round_stats["mean"] - round_stats["ci"],
            round_stats["mean"] + round_stats["ci"],
            alpha=0.12,
            color=colors.get(scenario, "gray")
        )

        # Add degradation annotation ONLY for crypto scenarios
        if scenario in crypto_degradation_scenarios and round_stats["round"].max() > 30:
            # Find where degradation starts (PR-AUC drops significantly)
            window = 5
            for i in range(window, len(round_stats)):
                prev_mean = round_stats.loc[i-window:i-1, "mean"].mean()
                curr_mean = round_stats.loc[i, "mean"]
                if prev_mean - curr_mean > 0.05:  # Significant drop
                    ax1.annotate('Crypto degradation:\nmodulus exhaustion',
                              xy=(round_stats.loc[i, "round"], curr_mean),
                              xytext=(round_stats.loc[i, "round"]-15, curr_mean+0.1),
                              arrowprops=dict(facecolor='red', shrink=0.05, width=2),
                              fontsize=9, color='red', fontweight='bold',
                              bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.3))
                    break

    ax1.set_xlabel("Training Round", fontweight='bold')
    ax1.set_ylabel("PR-AUC", fontweight='bold')
    ax1.set_title("A) Training Convergence Under Protocol Enforcement", fontweight='bold', fontsize=12)
    ax1.grid(True, alpha=0.25, linestyle="--")

    # Zoomed y-axis to show small but meaningful PR-AUC differences
    ax1.set_ylim(0.60, 0.68)

    # Force integer training-round ticks
    if "round" in main_df.columns and not main_df["round"].empty:
        ax1.set_xlim(int(main_df["round"].min()), int(main_df["round"].max()))
        ax1.xaxis.set_major_locator(MaxNLocator(integer=True))

    # Add legend
    ax1.legend(loc="lower right", ncol=2, frameon=True, framealpha=0.9,
              fancybox=True, shadow=True)

    # ============================================
    # ============================================
    # PLOT B: Effective Clients (BAR CHART)
    # ============================================
    ax2 = plt.subplot(2, 2, 2)

    # Calculate average effective clients per scenario
    scenario_avg_clients = []
    scenario_labels_clean = []

    for scenario in main_scenarios:
        scenario_data = main_df[main_df["scenario"] == scenario]
        if scenario_data.empty:
            continue

        # Use last round for each seed, then average
        last_round_data = scenario_data.sort_values(['seed', 'round']).groupby('seed').tail(1)

        if not last_round_data.empty:
            # For attack scenarios, effective clients should be lower
            if scenario in ["T1_TEST_REPLAY_ATTACK", "T2_TEST_SIGNATURE_ATTACK"]:
                # Attack scenarios: clients should be banned, so effective clients should decrease
                # Get the actual average from the last round
                avg_clients = last_round_data["effective_clients"].mean()
            else:
                # Normal scenarios: all admitted clients should participate
                avg_clients = last_round_data["effective_clients"].mean()

            scenario_avg_clients.append(avg_clients)
            scenario_labels_clean.append(scenario.replace('_TADP', '').replace('_', ' '))

    # Create bar chart
    if scenario_avg_clients:
        x_pos = np.arange(len(scenario_labels_clean))
        width = 0.7

        # Use consistent colors
        bar_colors = [colors.get(sc, '#1f77b4') for sc in main_scenarios
                    if sc in [s for s in main_scenarios if not main_df[main_df["scenario"] == s].empty]]

        bars = ax2.bar(x_pos, scenario_avg_clients, width,
                      color=bar_colors, edgecolor='black', linewidth=1.5,
                      capsize=8, error_kw={'elinewidth': 1.5, 'capthick': 1.5},
                      alpha=0.8)

        # Add value labels on bars (NO STANDARD DEVIATION)
        for bar, val in zip(bars, scenario_avg_clients):
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height + 0.3,
                    f'{val:.1f}',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

        ax2.set_xlabel("Scenario", fontweight='bold')
        ax2.set_ylabel("Effective Clients per Round (Avg)", fontweight='bold')
        ax2.set_title("B) Client Participation After Enforcement", fontweight='bold', fontsize=12)
        ax2.set_xticks(x_pos)
        ax2.set_xticklabels(scenario_labels_clean, rotation=30, ha='right')
        ax2.grid(True, alpha=0.25, linestyle='--', axis='y')

        # Add reference lines
        try:
            ax2.axhline(y=N_CLIENTS, linestyle=':', color='red', alpha=0.8,
                      linewidth=2, label=f'Total Clients ({N_CLIENTS})')
        except:
            pass

        # Show TADP admitted line if available
        if hasattr(experiment, "admitted_client_ids_fixed"):
            admitted = len(getattr(experiment, "admitted_client_ids_fixed"))
            ax2.axhline(y=admitted, linestyle='--', color='green', alpha=0.8,
                      linewidth=2, label=f'TADP Admitted ({admitted})')

        # Set y-axis limits
        y_max = max(scenario_avg_clients) + 2 if scenario_avg_clients else 20
        ax2.set_ylim(0, y_max)

        # Add legend for reference lines
        ax2.legend(loc='upper right', frameon=True, framealpha=0.9)

    # ============================================
    # ============================================
    # PLOT C: Enforcement Cost by Tier (Twin Axis Bars) - INCLUDES ATTACK SCENARIOS
    # ============================================
    ax3 = plt.subplot(2, 2, 3)

    # Map ALL scenarios to display names
    scenario_display_map = {
        "B0_Plain_FL": "B0",
        "B1_xMK_CKKS": "B1",
        "A1_TADP_admission_only": "A1",
        "T1_Plain_TADP": "T1",
        "T1_TEST_SIGNATURE_ATTACK": "T1-ATK",
        "T2_DF_KS_TADP": "T2",
        "T2_TEST_REPLAY_ATTACK": "T2-ATK",
        "T3_XMK_CKKS_TADP": "T3",
        "T3_TEST_POFC_ATTACK": "T3-ATK",
        "T4_CKKS_KS_TADP": "T4"
    }

    # Include ALL scenarios including attack scenarios
    all_plot_scenarios = list(scenario_display_map.keys())

    tier_labels, comm_data, crypto_data = [], [], []
    for scenario in all_plot_scenarios:
        if scenario not in set(main_df["scenario"].unique()):
            continue
        sd = main_df[main_df["scenario"] == scenario]
        if not sd.empty:
            tier_labels.append(scenario_display_map[scenario])
            comm_data.append(float(sd["comm_up_bytes"].mean() / 1024.0))
            crypto_data.append(float(sd["crypto_overhead_ms"].mean()))

    if tier_labels:
        x = np.arange(len(tier_labels))
        w = 0.35

        # Distinct colors for communication vs crypto
        comm_color = '#1f77b4'  # Blue
        crypto_color = '#ff7f0e'  # Orange

        # Communication bars (left axis)
        comm_bars = ax3.bar(x - w/2, comm_data, w,
                          color=comm_color, alpha=0.85,
                          edgecolor='black', linewidth=1.5,
                          label='Communication')
        ax3.set_xlabel("Scenario", fontweight='bold')
        ax3.set_ylabel("Upload Communication (KB/round)", fontweight='bold', color=comm_color)
        ax3.set_title("C) Enforcement Cost Across Core Scenarios", fontweight='bold', fontsize=12)
        ax3.set_xticks(x)
        ax3.set_xticklabels(tier_labels, fontweight='bold')
        ax3.tick_params(axis='y', labelcolor=comm_color)
        ax3.grid(True, alpha=0.25, linestyle='--', axis='y')

        # Add headroom before placing labels
        comm_ymax = max(comm_data) if comm_data else 1
        ax3.set_ylim(0, comm_ymax * 1.30)

        # Add value labels on communication bars
        for bar, val, label in zip(comm_bars, comm_data, tier_labels):
            height = bar.get_height()
            is_t4 = label == "T4"
            offset = comm_ymax * (0.06 if is_t4 else 0.02)

            ax3.text(
                bar.get_x() + bar.get_width()/2.,
                height + offset,
                f'{val:.1f}',
                ha='center',
                va='bottom',
                fontsize=8 if is_t4 else 9,
                fontweight='bold',
                color=comm_color
            )

        # Crypto bars (right axis)
        ax3b = ax3.twinx()
        crypto_bars = ax3b.bar(x + w/2, crypto_data, w,
                              color=crypto_color, alpha=0.85,
                              edgecolor='black', linewidth=1.5,
                              label='Crypto Overhead')
        ax3b.set_ylabel("Crypto Overhead (ms/round)", fontweight='bold', color=crypto_color)
        ax3b.tick_params(axis='y', labelcolor=crypto_color)

        # Add headroom before placing crypto labels
        crypto_ymax = max(crypto_data) if crypto_data else 1
        ax3b.set_ylim(0, crypto_ymax * 1.40)

        # Add value labels on crypto bars
        for bar, val, label in zip(crypto_bars, crypto_data, tier_labels):
            height = bar.get_height()
            is_t4 = label == "T4"
            offset = crypto_ymax * (0.08 if is_t4 else 0.02)

            ax3b.text(
                bar.get_x() + bar.get_width()/2.,
                height + offset,
                f'{val:.1f}',
                ha='center',
                va='bottom',
                fontsize=8 if is_t4 else 9,
                fontweight='bold',
                color=crypto_color,
                rotation=90 if is_t4 else 0
            )

        # Add legend
        lines1, labels1 = ax3.get_legend_handles_labels()
        lines2, labels2 = ax3b.get_legend_handles_labels()
        ax3.legend(lines1 + lines2, labels1 + labels2,
                  loc='upper left', frameon=True, framealpha=0.9)

        # Highlight attack scenarios with red border
        for i, label in enumerate(tier_labels):
            if "-ATK" in label:
                # Add red rectangle around attack scenario bars
                rect = plt.Rectangle((i - w/2 - 0.1, 0), w*2 + 0.2, max(max(comm_data), max(crypto_data)) * 1.05,
                                  fill=False, edgecolor='red', linewidth=2, linestyle='--', alpha=0.7)
                ax3.add_patch(rect)
                ax3.text(i, max(max(comm_data), max(crypto_data)) * 1.08,
                        'Attack Test', ha='center', va='bottom',
                        fontsize=8, fontweight='bold', color='red')

    # ============================================
    # ============================================
    # PLOT D: Attack Enforcement (Before vs After) ✅ from test_df
    # Legend (3 items): Benign / Malicious active (Before vs After) / Attackers removed
    # X-labels (2 lines): "T# with <Tier>" then "<Attack Type>"
    # ============================================
    ax4 = plt.subplot(2, 2, 4)

    # --- normalize scenario strings (critical) ---
    test_df = test_df.copy()
    test_df["scenario"] = test_df["scenario"].astype(str).str.strip()

    attack_scenarios = [
        "T1_TEST_SIGNATURE_ATTACK",
        "T2_TEST_REPLAY_ATTACK",
        "T3_TEST_POFC_ATTACK",
    ]

    # keep only scenarios that actually exist in test_df (robust)
    test_set = set(test_df["scenario"].unique())
    attack_scenarios = [s for s in attack_scenarios if s in test_set]

    if not attack_scenarios:
        ax4.text(0.5, 0.5, "No attack scenarios found in test_df", ha="center", va="center")
    else:
        # attacker counts (My design)
        # attacker counts used in the experiment design
        attackers_map = {
            "T1_TEST_SIGNATURE_ATTACK": 2.0,
            "T2_TEST_REPLAY_ATTACK": 1.0,
            "T3_TEST_POFC_ATTACK": 3.0,
        }

        # Data-integrity safeguard: must match Table 3 in the manuscript
        EXPECTED_EFFECTIVE_CLIENTS_TABLE3 = {
            "T1_TEST_SIGNATURE_ATTACK": 10.0,
            "T2_TEST_REPLAY_ATTACK": 11.0,
            "T3_TEST_POFC_ATTACK": 9.0,
        }

        # In full manuscript runs, Panel D should contain all three attack scenarios.
        # In smoke tests or tier-only diagnostic runs, some attack scenarios may be absent.
        # Therefore, we warn instead of stopping the whole experiment.
        missing_table3 = sorted(set(EXPECTED_EFFECTIVE_CLIENTS_TABLE3) - set(attack_scenarios))

        if missing_table3:
            print(
                "  NOTE: Panel D is missing attack scenario(s) because they were not run in this experiment subset: "
                f"{missing_table3}"
            )

        # Line1: "T# with <Tier>" , Line2: "<Attack type>"
        pretty = {
            "T1_TEST_SIGNATURE_ATTACK": ("T1 with Plain",    "Bad Signature"),
            "T2_TEST_REPLAY_ATTACK":    ("T2 with DF+KS",    "Replay (Freshness)"),
            "T3_TEST_POFC_ATTACK":      ("T3 with xMK-CKKS", "PoFC Overflow"),
        }

        # admitted baseline
        admitted = 12.0
        if hasattr(experiment, "admitted_client_ids_fixed"):
            admitted = float(len(getattr(experiment, "admitted_client_ids_fixed")))

        scenario_data = []

        for sc in attack_scenarios:
            sdf = test_df[test_df["scenario"] == sc].copy()
            if sdf.empty:
                continue

            # final effective clients from logs
            rf = sdf["round"].max()
            rf_df = sdf[sdf["round"] == rf]
            effective_after = float(rf_df["effective_clients"].mean()) if not rf_df.empty else admitted
            expected_effective = EXPECTED_EFFECTIVE_CLIENTS_TABLE3[sc]
            assert abs(effective_after - expected_effective) < 1e-6, (
                f"Panel D/Table 3 mismatch for {sc}: "
                f"figure effective_after={effective_after}, "
                f"Table 3 expected={expected_effective}"
            )
            # attackers (force correct mapping)
            attackers = float(attackers_map.get(sc.strip(), 0.0))

            # removed attackers inferred from admitted - effective_after (capped by attackers)
            removed = max(0.0, admitted - effective_after)
            removed = min(removed, attackers)

            # malicious remaining after
            mal_after = max(0.0, attackers - removed)

            # BEFORE composition (sum to admitted)
            mal_before = min(attackers, admitted)
            ben_before = max(0.0, admitted - mal_before)

            # AFTER composition shown on admitted baseline (sum to admitted)
            # (benign pool remains; removal shown as its own segment; mal_after shows remaining malicious)
            ben_after = ben_before

            line1, line2 = pretty.get(sc, (sc, ""))

            scenario_data.append({
                "line1": line1,
                "line2": line2,
                "admitted": admitted,
                "effective_after": effective_after,
                "ben_before": ben_before,
                "mal_before": mal_before,
                "ben_after": ben_after,
                "mal_after": mal_after,
                "removed": removed,
            })

        if not scenario_data:
            ax4.text(0.5, 0.5, "Attack scenarios exist but no rows found", ha="center", va="center")
        else:
            x = np.arange(len(scenario_data))
            width = 0.36
            bar_h = 0.55

            # Colors
            COLOR_BEN = "#2ca02c"
            COLOR_MAL = "#d62728"
            COLOR_REM = "#ff7f0e"

            # Same hatch for malicious before and after, so the legend remains truthful
            HATCH_MAL_BEFORE = "//"
            HATCH_MAL_AFTER = HATCH_MAL_BEFORE

            # BEFORE bars: total = admitted
            for i, d in enumerate(scenario_data):
                ax4.bar(
                    x[i] - width/2, d["ben_before"], bar_h,
                    color=COLOR_BEN, edgecolor="black", linewidth=1
                )
                ax4.bar(
                    x[i] - width/2, d["mal_before"], bar_h,
                    bottom=d["ben_before"],
                    color=COLOR_MAL, edgecolor="black", linewidth=1,
                    hatch=HATCH_MAL_BEFORE
                )
                ax4.text(
                    x[i] - width/2, d["admitted"] + 0.25, f"{d['admitted']:.0f}",
                    ha="center", va="bottom", fontsize=8, fontweight="bold"
                )

            # AFTER bars: total STILL = admitted (composition shows removals)
            for i, d in enumerate(scenario_data):
                ax4.bar(
                    x[i] + width/2, d["ben_after"], bar_h,
                    color=COLOR_BEN, edgecolor="black", linewidth=1
                )
                ax4.bar(
                    x[i] + width/2, d["mal_after"], bar_h,
                    bottom=d["ben_after"],
                    color=COLOR_MAL, edgecolor="black", linewidth=1,
                    hatch=HATCH_MAL_AFTER
                )
                ax4.bar(
                    x[i] + width/2, d["removed"], bar_h,
                    bottom=d["ben_after"] + d["mal_after"],
                    color=COLOR_REM, edgecolor="black", linewidth=1,
                    hatch="xx"
                )

                # show effective_after as a horizontal marker line + value
                ax4.hlines(
                    d["effective_after"],
                    x[i] + width/2 - 0.16,
                    x[i] + width/2 + 0.16,
                    colors="black", linewidth=2
                )
                ax4.text(
                    x[i] + width/2, d["effective_after"] + 0.2, f"{d['effective_after']:.0f}",
                    ha="center", va="bottom", fontsize=8, fontweight="bold"
                )

            # ----- Formatting -----
            ax4.set_title("D) Attack Enforcement: Before vs After (Attack Scenarios)", fontweight="bold", fontsize=12)
            ax4.set_ylabel("Client Count", fontweight="bold")
            ax4.set_xlabel("Attack Scenario", fontweight="bold")

            # Custom nested x-labels:
            # first row = Before / After under each bar
            # second/third rows = scenario description
            ax4.set_xticks(x)
            ax4.set_xticklabels([""] * len(x))

            xaxis_trans = ax4.get_xaxis_transform()

            for i, d in enumerate(scenario_data):
                ax4.text(
                    x[i] - width/2, -0.055, "Before",
                    transform=xaxis_trans,
                    ha="center", va="top",
                    fontsize=7, fontweight="bold",
                    clip_on=False
                )
                ax4.text(
                    x[i] + width/2, -0.055, "After",
                    transform=xaxis_trans,
                    ha="center", va="top",
                    fontsize=7, fontweight="bold",
                    clip_on=False
                )
                ax4.text(
                    x[i], -0.145, f"{d['line1']}\n{d['line2']}",
                    transform=xaxis_trans,
                    ha="center", va="top",
                    fontsize=8, fontweight="bold",
                    clip_on=False
                )

            ax4.margins(x=0.10)
            ax4.tick_params(axis="x", length=0)

            ax4.grid(True, alpha=0.25, linestyle="--", axis="y")

            ymax = max(d["admitted"] for d in scenario_data)
            ax4.set_ylim(0, ymax + 3)

            # admitted reference line
            ax4.axhline(y=admitted, linestyle="--", linewidth=1.2, alpha=0.6)

            # ----- Legend (ONLY 3 entries, as requested) -----
            from matplotlib.patches import Patch
            legend_handles = [
                Patch(facecolor=COLOR_BEN, edgecolor="black", label="Benign"),
                Patch(facecolor=COLOR_MAL, edgecolor="black", hatch=HATCH_MAL_BEFORE,
                      label="Malicious active"),
                Patch(facecolor=COLOR_REM, edgecolor="black", hatch="xx",
                      label="Attackers removed"),
            ]

            ax4.legend(
                handles=legend_handles,
                loc="lower right",
                framealpha=0.9,
                ncol=1
            )




    # Save summary data for Plot C
    if tier_labels:
        plot_c_data = pd.DataFrame({
            'tier': tier_labels,
            'comm_kb_per_round': comm_data,
            'crypto_ms_per_round': crypto_data
        })
        plot_c_data.to_csv(os.path.join(out_dir, "plot_c_data.csv"), index=False)

    # ============================================
    # CRITICAL: Save figure at 600 DP
    # ============================================
    plot_path = os.path.join(out_dir, "learning_curves_600dpi.png")
    fig.savefig(plot_path, dpi=600, bbox_inches='tight', pad_inches=0.1)
    print(f"✓ Saved learning curves at 600 DPI: {plot_path}")

    # Also save as PDF for vector format
    pdf_path = os.path.join(out_dir, "learning_curves_vector.pdf")
    fig.savefig(pdf_path, format='pdf', bbox_inches='tight', pad_inches=0.1)
    print(f"✓ Saved learning curves as vector PDF: {pdf_path}")

    return fig
# --------------------------
# MAIN EXECUTION FUNCTION
# --------------------------
def main():
    """Clean reviewer-facing execution with concise final reporting."""

    # Configure notebook display
    try:
        from IPython import get_ipython
        get_ipython().run_line_magic('matplotlib', 'inline')
        print("✓ Notebook display configured")
    except Exception:
        print("Running in script mode")

    print("\n" + "=" * 80)
    print("TADP-SEC EXPERIMENT")
    print("Trustworthy Federated Learning with Governance and Runtime Enforcement")
    print("=" * 80)

    total_start_time = time.time()

    # Initialize and setup
    experiment = TADPSecExperiment()
    experiment.setup_experiment()

    # Phase 1: TADP admission
    print("\n" + "=" * 80)
    print("PHASE 1 — TADP ADMISSION")
    print("=" * 80)

    admission_results = experiment.run_tadp_admission()

    admission_path = os.path.join(STATS_DIR, "admission_results.json")
    with open(admission_path, "w") as f:
        json.dump(admission_results, f, indent=2)

    print(f"✓ Admission results saved: {admission_path}")

    # Phase 2: run scenarios
    print("\n" + "=" * 80)
    print("PHASE 2 — SCENARIO EXECUTION")
    print("=" * 80)

    all_results = experiment.run_all_scenarios()

    if all_results.empty:
        print("\nERROR: No results generated.")
        return

    # Save combined results
    results_path = os.path.join(OUT_DIR, "all_scenarios_results_comprehensive.csv")
    all_results.to_csv(results_path, index=False)

    # Generate protocol enforcement table
    protocol_table = experiment.generate_protocol_enforcement_table()

    # Statistical summary
    final_summary = summarize_final_by_seed(all_results)
    final_summary_path = os.path.join(STATS_DIR, "final_seed_summary.csv")

    if not final_summary.empty:
        final_summary.to_csv(final_summary_path, index=False)

    paired_tests = paired_test_vs_baseline(all_results, baseline="B0_Plain_FL")
    if not paired_tests.empty:
        paired_tests_path = os.path.join(STATS_DIR, "paired_tests_vs_B0.csv")
        paired_tests.to_csv(paired_tests_path, index=False)

    # Phase 3: output generation
    print("\n" + "=" * 80)
    print("PHASE 3 — TABLES AND FIGURES")
    print("=" * 80)

    main_table = generate_main_comparison_table(all_results, FIG_DIR)
    threat_matrix = generate_threat_coverage_matrix(FIG_DIR)
    generate_learning_curves_plot(all_results, experiment, FIG_DIR)

    # --------------------------
    # CLEAN FINAL SUMMARY
    # --------------------------
    print("\n" + "=" * 80)
    print("FINAL REVIEWER-FACING SUMMARY")
    print("=" * 80)

    last_rows = (
        all_results.sort_values(["scenario", "seed", "round"])
        .groupby(["scenario", "seed"])
        .tail(1)
    )

    baseline_rows = last_rows[last_rows["scenario"] == "B0_Plain_FL"]
    baseline_pr_auc = (
        baseline_rows["pr_auc"].mean()
        if not baseline_rows.empty else float("nan")
    )

    non_attack_rows = last_rows[
        ~last_rows["scenario"].str.contains("_TEST_", na=False)
    ]

    if not non_attack_rows.empty:
        non_attack_scores = (
            non_attack_rows.groupby("scenario")["pr_auc"]
            .mean()
            .sort_values(ascending=False)
        )
        best_non_attack_scenario = non_attack_scores.index[0]
        best_non_attack_score = non_attack_scores.iloc[0]
    else:
        best_non_attack_scenario = "N/A"
        best_non_attack_score = float("nan")

    comm_by_scenario = all_results.groupby("scenario")["comm_up_bytes"].mean() / 1024.0

    most_comm_efficient = comm_by_scenario.idxmin()
    most_comm_value = comm_by_scenario.min()

    highest_comm_cost = comm_by_scenario.idxmax()
    highest_comm_value = comm_by_scenario.max()

    admitted_count = experiment.admission_stats.get("admitted", 0)
    rejected_count = experiment.admission_stats.get("rejected", 0)

    print("\nExperimental setup:")
    print(f"• Total clients: {N_CLIENTS}")
    print(f"• TADP-admitted clients: {admitted_count}")
    print(f"• TADP-rejected clients: {rejected_count}")
    print(f"• Rounds: {ROUNDS}")
    print(f"• Seeds: {len(SEEDS)}")

    print("\nPerformance summary:")
    print(f"• Plain FL baseline PR-AUC: {baseline_pr_auc:.4f}")
    print(
        f"• Best non-attack scenario: "
        f"{best_non_attack_scenario} ({best_non_attack_score:.4f} PR-AUC)"
    )

    print("\nProtocol enforcement summary:")
    print("• TADP reduced the eligible client pool before training.")
    print("• T1 blocked invalid-signature clients.")
    print("• T2 blocked replay/stale-update clients.")
    print("• T3 blocked PoFC/rate-limit overflow clients.")
    print("• T4 completed the simulated asymmetric CKKS double key-switching workflow.")
    print("• Attack scenarios intentionally reduce effective clients by rejecting malicious participants.")

    print("\nResource summary:")
    print(
        f"• Most communication-efficient scenario: "
        f"{most_comm_efficient} ({most_comm_value:.2f} KB/round)"
    )
    print(
        f"• Highest communication-cost scenario: "
        f"{highest_comm_cost} ({highest_comm_value:.2f} KB/round)"
    )

    print("\nImportant methodological note:")
    if len(SEEDS) < 5 or ROUNDS < 10:
        print("• This run is a smoke test. Final manuscript results should use at least 5 seeds and longer FL training.")
    else:
        print("• Main scenarios B0–T3 use five seeds and twenty rounds for manuscript-level reporting. T4 is reported separately as a formal CKKS double-key-switching feasibility tier using one seed and five rounds because custom Python RLWE/RNS key switching is computationally expensive.")

    print("\nSaved outputs:")
    print(f"• Full results: {results_path}")
    print(f"• Final seed summary: {final_summary_path}")
    print(f"• Scenario comparison table: {os.path.join(STATS_DIR, 'scenario_comparison_table.csv')}")
    print(f"• Protocol enforcement table: {os.path.join(STATS_DIR, 'protocol_enforcement_table.csv')}")
    print(f"• Main figure: {os.path.join(FIG_DIR, 'learning_curves_600dpi.png')}")
    print(f"• Vector figure: {os.path.join(FIG_DIR, 'learning_curves_vector.pdf')}")

    total_time = time.time() - total_start_time

    print("\nExecution summary:")
    print(f"• Total execution time: {total_time:.2f} seconds")
    print("=" * 80)


# --------------------------
# RUN EXPERIMENT
# --------------------------
if __name__ == "__main__":
    main()

✓ Configured for Q1 journal: ALL plots at 600 DPI
✓ Notebook display configured

TADP-SEC EXPERIMENT
Trustworthy Federated Learning with Governance and Runtime Enforcement
TADP-SEC EXPERIMENT v8.2: PROFESSIONAL PUBLICATION-READY
  Loading dataset from: /content/creditcard.csv
  ✅ Dataset loaded: 284,807 transactions

  GLOBAL DATASET STATISTICS:
  • Total samples: 284,807
  • Features: 29
  • Global fraud rate: 0.1727%

  Creating 20 clients with CLEAN data splitting...
  Step 1: Creating global test set (20% of total data)...
    • Global test set size: 56,962 samples
    • Global test fraud rate: 0.1720%
    • Training data for clients: 227,845 samples (80%)
  Step 2: Distributing 227,845 training samples among 20 clients...
    • Training fraud samples: 394
    • Training normal samples: 227,451
  Step 3: Assigning data to clients...
  Step 4: Scaling global test set...

  CLEAN DATA SPLITTING SUMMARY:
  Client       Samples    Fraud      Fraud %   
  ------------ ---------- -------